# Protein-Cold RPI Benchmark and Artifact Generation

This notebook trains and evaluates the four RPI systems used in the study:
**PairMLP-Cross, GraphSAGE-2L, ProtoContrast, and ZHMolGraph-Best120**.

It standardizes NPInter2, RPI7317, and NPInter5 under the same protein-cold
evaluation protocol, performs split checks, saves compact prediction tables,
and generates the artifacts used by the downstream reliability notebooks.

> **Compatibility note:** a few internal module/output names still contain
> `journal_*` because the downstream notebooks expect those filenames. They
> are implementation names only and do not affect the analysis.


## 0. Configuration


In [ ]:
from pathlib import Path
import os, sys, json, shutil, zipfile, importlib, gc, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns",200)
pd.set_option("display.width",220)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python",sys.version.split()[0],"| PyTorch",torch.__version__,"| Device",DEVICE)
if torch.cuda.is_available(): print("GPU:",torch.cuda.get_device_name(0))

DATA_ROOT_CANDIDATES=[
    Path("/kaggle/input/datasets/abdullahnayemwasi/rnafm-prottrans-dataset"),
    Path("/kaggle/input/datasets/abdullahnayem/dataset"),
    Path("/kaggle/input/rnafm-prottrans-dataset"),
]
DATA_ROOT=next((p for p in DATA_ROOT_CANDIDATES if p.exists()),DATA_ROOT_CANDIDATES[0])
WORK_ROOT=Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUTPUT_ROOT=WORK_ROOT/"dependable_rpi_journal_v1"
ANALYSIS_ROOT=OUTPUT_ROOT/"analysis"
FIGURE_ROOT=OUTPUT_ROOT/"paper_figures"
for p in [OUTPUT_ROOT,ANALYSIS_ROOT,FIGURE_ROOT]: p.mkdir(parents=True,exist_ok=True)

RNA_TAG="rnafm"; PROTEIN_TAG="proteinprottrans"; PROFILE="clean_120"
DATASETS=("NPInter2","RPI7317","NPInter5")
PROTOCOLS=("prot_cold",)
CENTRAL_MODELS=("PairMLP-Cross","GraphSAGE-2L","ProtoContrast","ZHMolGraph-Best120")
SEEDS=(64,1,7); N_SPLITS=5


RUN_DATASETS=DATASETS
RUN_MODELS=CENTRAL_MODELS
RUN_SEEDS=SEEDS
RUN_FOLDS=None

RUN={
    "audit":True,
    "preflight":True,
    "training":True,
    "collect_predictions":True,
    "calibration":True,
    "selective_prediction":True,
    "structural_diagnosis":True,
    "failure_detector":True,
    "leave_one_dataset_out_failure":True,
    "statistics":True,
    "paper_figures":True,
    # Optional or slower analyses.
    "temperature_scaling":True,
    "conformal_supplement":False,
    "ensemble_supplement":False,
    "mmd_supplement":True,
    "fault_supplement":False,
    "xai_supplement":False,
    "thenovel_external":False,
    "compact_export":True,
}

# Optional restore of a previous compatible output archive.
PREVIOUS_RESULTS_ZIP=None
if PREVIOUS_RESULTS_ZIP:
    with zipfile.ZipFile(PREVIOUS_RESULTS_ZIP) as zf: zf.extractall(OUTPUT_ROOT)

print("DATA_ROOT:",DATA_ROOT,"exists=",DATA_ROOT.exists())
print("OUTPUT_ROOT:",OUTPUT_ROOT)
print("Models:",CENTRAL_MODELS)


In [ ]:
def find_unique_file(root,filename):
    direct=root/filename
    if direct.exists(): return direct
    hits=list(root.rglob(filename)) if root.exists() else []
    if len(hits)==1: return hits[0]
    if len(hits)>1: raise RuntimeError(f"Multiple copies of {filename}: {hits}")
    return direct

required=[]
for ds in DATASETS:
    required += [
        f"RPI_{ds}_{RNA_TAG}_embed_normal.pkl",
        f"RPI_{ds}_{PROTEIN_TAG}_embed_normal.pkl",
        f"dataset_RPI_{ds}_RP.csv",
    ]
required.append("NPInter5.xlsx")
rows=[]; missing=[]
for name in required:
    p=find_unique_file(DATA_ROOT,name); rows.append({"file":name,"found":p.exists(),"path":str(p)})
    if not p.exists(): missing.append(str(p))
display(pd.DataFrame(rows))
if missing: raise FileNotFoundError("Missing required files:\n"+"\n".join(missing))
print("Core publication datasets are available.")


## 1. Model implementations

The following cells define the model architectures used in the benchmark.


In [ ]:
%%writefile exact_pair_models.py


import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

def build_train_graph_from_split(r_idx, p_idx, y, train_row_idx, n_rna, n_prot):
    """Sparse bipartite adjacency from TRAIN POSITIVES only (no leakage)."""
    tr = train_row_idx
    pos_mask = (y[tr] == 1)
    r_pos = r_idx[tr][pos_mask]
    p_pos = p_idx[tr][pos_mask]
    if len(r_pos) == 0:
        raise ValueError("No positive edges in train split.")
    ij = np.vstack([r_pos, p_pos])
    vals = np.ones(ij.shape[1], dtype=np.float32)
    A_rp = torch.sparse_coo_tensor(
        torch.tensor(ij, dtype=torch.long),
        torch.tensor(vals),
        size=(n_rna, n_prot)
    ).coalesce()
    rna2prot = [[] for _ in range(n_rna)]
    prot2rna = [[] for _ in range(n_prot)]
    for rr, pp in zip(r_pos.tolist(), p_pos.tolist()):
        rna2prot[rr].append(pp); prot2rna[pp].append(rr)
    return A_rp, rna2prot, prot2rna

def sparse_edge_dropout(A, p, training):
    if (not training) or p <= 0.0: return A
    A = A.coalesce()
    val = A.values()
    keep = (torch.rand_like(val) > p).to(val.dtype)
    val2 = val * keep / (1.0 - p + 1e-12)
    return torch.sparse_coo_tensor(A.indices(), val2, A.size(), device=A.device).coalesce()

def symm_norm_msgs(A, h_rna, h_prot):
    """LightGCN-style symmetric normalization on bipartite graph."""
    A = A.coalesce()
    deg_r = torch.sparse.sum(A, dim=1).to_dense().clamp_min(1.0)
    deg_p = torch.sparse.sum(A, dim=0).to_dense().clamp_min(1.0)
    inv_sqrt_r = deg_r.rsqrt(); inv_sqrt_p = deg_p.rsqrt()
    h_rna_n  = h_rna  * inv_sqrt_r.unsqueeze(-1)
    h_prot_n = h_prot * inv_sqrt_p.unsqueeze(-1)
    rna_msg  = torch.sparse.mm(A,               h_prot_n) * inv_sqrt_r.unsqueeze(-1)
    prot_msg = torch.sparse.mm(A.transpose(0,1), h_rna_n) * inv_sqrt_p.unsqueeze(-1)
    return rna_msg, prot_msg, deg_r, deg_p

class PairMLP(nn.Module):
    """
     PairMLP baseline:
      1. enc_rna : d_rna → d   (Linear → ReLU → Dropout)
      2. enc_prot: d_prot → d  (Linear → ReLU → Dropout)
      3. interaction vector z = [hr; hp; |hr-hp|; hr⊙hp]  ∈ R^{4d}
      4. MLP head: 4d → 512 → 256 → 1

    """
    def __init__(self, d_rna=640, d_prot=1024, d=256, dropout=0.2):
        super().__init__()
        self.enc_rna  = nn.Sequential(nn.Linear(d_rna,  d), nn.ReLU(), nn.Dropout(dropout))
        self.enc_prot = nn.Sequential(nn.Linear(d_prot, d), nn.ReLU(), nn.Dropout(dropout))
        self.mlp = nn.Sequential(
            nn.Linear(4*d, 512), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(512,  256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256,    1)
        )

    def forward(self, x_rna_batch, x_prot_batch):
        """x_rna_batch: (B, d_rna),  x_prot_batch: (B, d_prot) → logits (B,)"""
        hr = self.enc_rna(x_rna_batch)
        hp = self.enc_prot(x_prot_batch)
        z  = torch.cat([hr, hp, torch.abs(hr-hp), hr*hp], dim=1)
        return self.mlp(z).squeeze(-1)

class PairMLPGraphWrapper(nn.Module):
    """
    Wraps PairMLP to expose the encode_nodes / score_edges interface
    used by all CV runners and the retrieval pipeline.
    PairMLP doesn't use the graph — the wrapper just stores x_rna/x_prot
    so score_edges can look them up.
    """
    def __init__(self, d_rna=640, d_prot=1024, d=256, dropout=0.2):
        super().__init__()
        self.model = PairMLP(d_rna, d_prot, d, dropout)
        self._x_rna  = None
        self._x_prot = None

    def encode_nodes(self, x_rna, x_prot, A_rp, rna2prot, prot2rna):
        """Store tensors; return dummies so the interface is satisfied."""
        self._x_rna  = x_rna
        self._x_prot = x_prot
        dummy_core = torch.zeros(1, device=x_rna.device)
        return x_rna, x_prot, dummy_core

    def score_edges(self, h_rna, h_prot, core, r_ids, p_ids):
        return self.model(self._x_rna[r_ids], self._x_prot[p_ids])

class PairMLP_Wide(nn.Module):
    """
    No projection. Uses raw concatenated embeddings.
      - z = [h_rna ; h_prot ; |h_rna - h_prot_proj| ; h_rna_proj * h_prot_proj]
      - But |diff| and product require same dim, so we project ONLY for those
        two features and keep the raw embeddings as-is for the concat.
    Final input dim: d_rna + d_prot + d + d = 640 + 1024 + 256 + 256 = 2176
    MLP head: 2176 -> 1024 -> 512 -> 256 -> 1
    """
    def __init__(self, d_rna=640, d_prot=1024, d=256, dropout=0.2):
        super().__init__()
        # Small projections JUST for the diff and product features
        self.proj_rna  = nn.Sequential(nn.Linear(d_rna,  d), nn.ReLU())
        self.proj_prot = nn.Sequential(nn.Linear(d_prot, d), nn.ReLU())

        in_dim = d_rna + d_prot + d + d
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, 1024), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(1024,    512), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(512,     256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256,       1)
        )

    def forward(self, x_rna_batch, x_prot_batch):
        pr = self.proj_rna(x_rna_batch)    # (B, d)
        pp = self.proj_prot(x_prot_batch)  # (B, d)
        z = torch.cat([
            x_rna_batch,         # raw RNA  (B, 640)
            x_prot_batch,        # raw prot (B, 1024)
            torch.abs(pr - pp),  # diff in projected space (B, 256)
            pr * pp,             # product in projected space (B, 256)
        ], dim=1)
        return self.mlp(z).squeeze(-1)

class PairMLP_Wide_Wrapper(nn.Module):
    def __init__(self, d_rna=640, d_prot=1024, d=256, dropout=0.2):
        super().__init__()
        self.model = PairMLP_Wide(d_rna, d_prot, d, dropout)
        self._x_rna = None
        self._x_prot = None

    def encode_nodes(self, x_rna, x_prot, A_rp, rna2prot, prot2rna):
        self._x_rna  = x_rna
        self._x_prot = x_prot
        return x_rna, x_prot, torch.zeros(1, device=x_rna.device)

    def score_edges(self, h_rna, h_prot, core, r_ids, p_ids):
        return self.model(self._x_rna[r_ids], self._x_prot[p_ids])

class PairMLP_Deep(nn.Module):
    """
    Same input as base PairMLP (4d=1024-dim interaction vector after projection).
    Deeper head with residual blocks:
      4d -> 768 -> 512 -> 384 -> 256 -> 1
    Plus a residual skip from layer 1 to layer 4.
    """
    def __init__(self, d_rna=640, d_prot=1024, d=256, dropout=0.2):
        super().__init__()
        self.enc_rna  = nn.Sequential(nn.Linear(d_rna,  d), nn.ReLU(), nn.Dropout(dropout))
        self.enc_prot = nn.Sequential(nn.Linear(d_prot, d), nn.ReLU(), nn.Dropout(dropout))

        self.layer1 = nn.Sequential(nn.Linear(4*d, 768), nn.ReLU(), nn.Dropout(dropout))
        self.layer2 = nn.Sequential(nn.Linear(768,  512), nn.ReLU(), nn.Dropout(dropout))
        self.layer3 = nn.Sequential(nn.Linear(512,  384), nn.ReLU(), nn.Dropout(dropout))
        self.layer4 = nn.Sequential(nn.Linear(384,  256), nn.ReLU(), nn.Dropout(dropout))

        # Residual projection from layer1 output (768) down to layer4 output (256)
        self.res_proj = nn.Linear(768, 256, bias=False)

        self.head = nn.Linear(256, 1)

    def forward(self, x_rna_batch, x_prot_batch):
        hr = self.enc_rna(x_rna_batch)
        hp = self.enc_prot(x_prot_batch)
        z = torch.cat([hr, hp, torch.abs(hr - hp), hr * hp], dim=1)  # (B, 4d)

        a1 = self.layer1(z)             # (B, 768)
        a2 = self.layer2(a1)            # (B, 512)
        a3 = self.layer3(a2)            # (B, 384)
        a4 = self.layer4(a3)            # (B, 256)
        a4 = a4 + self.res_proj(a1)     # residual from layer1 -> layer4

        return self.head(a4).squeeze(-1)

class PairMLP_Deep_Wrapper(nn.Module):
    def __init__(self, d_rna=640, d_prot=1024, d=256, dropout=0.2):
        super().__init__()
        self.model = PairMLP_Deep(d_rna, d_prot, d, dropout)
        self._x_rna = None
        self._x_prot = None

    def encode_nodes(self, x_rna, x_prot, A_rp, rna2prot, prot2rna):
        self._x_rna  = x_rna
        self._x_prot = x_prot
        return x_rna, x_prot, torch.zeros(1, device=x_rna.device)

    def score_edges(self, h_rna, h_prot, core, r_ids, p_ids):
        return self.model(self._x_rna[r_ids], self._x_prot[p_ids])

class PairMLP_Cross(nn.Module):
    """
    Cross-attention between RNA and protein, then standard 4-feature MLP head.
      hr, hp = enc_rna(x_rna), enc_prot(x_prot)         # (B, d) each
      hr_new = hr + Attn(query=hr, key=hp, value=hp)
      hp_new = hp + Attn(query=hp, key=hr, value=hr)
      z = [hr_new ; hp_new ; |hr_new - hp_new| ; hr_new * hp_new]   # (B, 4d)
      logit = MLP(z)
    """
    def __init__(self, d_rna=640, d_prot=1024, d=256, dropout=0.2, n_heads=4):
        super().__init__()
        assert d % n_heads == 0, f"d={d} must be divisible by n_heads={n_heads}"

        self.enc_rna  = nn.Sequential(nn.Linear(d_rna,  d), nn.ReLU(), nn.Dropout(dropout))
        self.enc_prot = nn.Sequential(nn.Linear(d_prot, d), nn.ReLU(), nn.Dropout(dropout))

        # Two cross-attention modules: RNA queries protein, protein queries RNA
        self.attn_r2p = nn.MultiheadAttention(d, num_heads=n_heads,
                                              dropout=dropout, batch_first=True)
        self.attn_p2r = nn.MultiheadAttention(d, num_heads=n_heads,
                                              dropout=dropout, batch_first=True)

        self.norm_r = nn.LayerNorm(d)
        self.norm_p = nn.LayerNorm(d)

        self.mlp = nn.Sequential(
            nn.Linear(4*d, 512), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(512,  256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256,    1)
        )

    def forward(self, x_rna_batch, x_prot_batch):
        hr = self.enc_rna(x_rna_batch)    # (B, d)
        hp = self.enc_prot(x_prot_batch)  # (B, d)

        # Reshape to (B, seq_len=1, d) for MultiheadAttention with batch_first=True
        hr_seq = hr.unsqueeze(1)
        hp_seq = hp.unsqueeze(1)

        # Cross-attention
        hr_attn, _ = self.attn_r2p(query=hr_seq, key=hp_seq, value=hp_seq)
        hp_attn, _ = self.attn_p2r(query=hp_seq, key=hr_seq, value=hr_seq)

        # Residual + norm, then squeeze back to (B, d)
        hr_new = self.norm_r(hr + hr_attn.squeeze(1))
        hp_new = self.norm_p(hp + hp_attn.squeeze(1))

        # Standard 4-feature interaction
        z = torch.cat([hr_new, hp_new,
                       torch.abs(hr_new - hp_new),
                       hr_new * hp_new], dim=1)
        return self.mlp(z).squeeze(-1)

class PairMLP_Cross_Wrapper(nn.Module):
    def __init__(self, d_rna=640, d_prot=1024, d=256, dropout=0.2, n_heads=4):
        super().__init__()
        self.model = PairMLP_Cross(d_rna, d_prot, d, dropout, n_heads)
        self._x_rna = None
        self._x_prot = None

    def encode_nodes(self, x_rna, x_prot, A_rp, rna2prot, prot2rna):
        self._x_rna  = x_rna
        self._x_prot = x_prot
        return x_rna, x_prot, torch.zeros(1, device=x_rna.device)

    def score_edges(self, h_rna, h_prot, core, r_ids, p_ids):
        return self.model(self._x_rna[r_ids], self._x_prot[p_ids])

class TypeAwareCoreFusion(nn.Module):
    """
    Separate attention pooling for RNA and Protein, then fuse.
    Returns one (d,) core vector so your train/eval code remains unchanged.
    """
    def __init__(self, d):
        super().__init__()
        self.att_r = nn.Linear(d, 1)
        self.att_p = nn.Linear(d, 1)
        self.fuse  = nn.Sequential(
            nn.Linear(2*d, d),
            nn.GELU(),
            nn.Linear(d, d)
        )

    def forward(self, h_rna, h_prot):
        wr = torch.softmax(self.att_r(h_rna).squeeze(-1), dim=0)  # (n_rna,)
        wp = torch.softmax(self.att_p(h_prot).squeeze(-1), dim=0) # (n_prot,)
        cr = (wr.unsqueeze(-1) * h_rna).sum(dim=0)
        cp = (wp.unsqueeze(-1) * h_prot).sum(dim=0)
        return self.fuse(torch.cat([cr, cp], dim=0))

class ProtoGNN_DegGate(nn.Module):
    def __init__(self, d_rna=640, d_prot=1024, d=256, layers=2, dropout=0.1,
                 hyper_topk=20, edge_drop=0.0):
        super().__init__()
        self.layers = layers
        self.edge_drop = edge_drop

        # Encoders: project raw LLM embeddings to shared d-dim space
        self.enc_rna  = nn.Sequential(nn.Linear(d_rna, d), nn.GELU(), nn.Dropout(dropout))
        self.enc_prot = nn.Sequential(nn.Linear(d_prot, d), nn.GELU(), nn.Dropout(dropout))

        # LightGCN-style message passing with post-aggregation mixing
        self.mix_r = nn.ModuleList([nn.Sequential(nn.Linear(d, d), nn.GELU(),
                                    nn.Dropout(dropout)) for _ in range(layers)])
        self.mix_p = nn.ModuleList([nn.Sequential(nn.Linear(d, d), nn.GELU(),
                                    nn.Dropout(dropout)) for _ in range(layers)])
        self.norm_r = nn.ModuleList([nn.LayerNorm(d) for _ in range(layers)])
        self.norm_p = nn.ModuleList([nn.LayerNorm(d) for _ in range(layers)])

        # Degree gate: g = sigmoid(MLP(log(1+deg))), blends raw ↔ graph
        self.deg_gate_r = nn.Sequential(nn.Linear(1, 32), nn.GELU(), nn.Linear(32, 1))
        self.deg_gate_p = nn.Sequential(nn.Linear(1, 32), nn.GELU(), nn.Linear(32, 1))

        # Global core vector (attention-pooled, NOT per-node prototypes)
        self.core = TypeAwareCoreFusion(d)

        # Per-edge pair gate: decides how much to trust graph vs raw per edge
        self.pair_gate = nn.Sequential(
            nn.Linear(d*4, 256), nn.GELU(), nn.Dropout(dropout), nn.Linear(256, 1)
        )
        self.bilin = nn.Bilinear(d, d, 1, bias=False)

        # Edge MLP: [hr, hp, |hr-hp|, hr*hp, hr0, hp0, core] → logit
        in_edge = (d*4) + (d*2) + d
        self.edge_mlp = nn.Sequential(
            nn.Linear(in_edge, 512), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(512, 256),     nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 1)
        )

    def encode_nodes(self, x_rna, x_prot, A_rp, rna2prot, prot2rna):
        h0_r = self.enc_rna(x_rna)
        h0_p = self.enc_prot(x_prot)
        hg_r, hg_p = h0_r, h0_p

        A = sparse_edge_dropout(A_rp, self.edge_drop, self.training)
        for i in range(self.layers):
            r_msg, p_msg, _, _ = symm_norm_msgs(A, hg_r, hg_p)
            hg_r = self.norm_r[i](hg_r + self.mix_r[i](r_msg))
            hg_p = self.norm_p[i](hg_p + self.mix_p[i](p_msg))

        # Degree from ORIGINAL adjacency (not edge-dropped)
        A0 = A_rp.coalesce()
        deg_r = torch.sparse.sum(A0, dim=1).to_dense().clamp_min(0.0)
        deg_p = torch.sparse.sum(A0, dim=0).to_dense().clamp_min(0.0)

        # Cold-node fix: hard-zero gate for degree-0 nodes → pure raw embedding
        is_cold_r = (deg_r == 0).float().unsqueeze(-1)
        is_cold_p = (deg_p == 0).float().unsqueeze(-1)
        g_r = torch.sigmoid(self.deg_gate_r(torch.log1p(deg_r).unsqueeze(-1)))
        g_p = torch.sigmoid(self.deg_gate_p(torch.log1p(deg_p).unsqueeze(-1)))
        h_r = h0_r + (1.0 - is_cold_r) * g_r * (hg_r - h0_r)
        h_p = h0_p + (1.0 - is_cold_p) * g_p * (hg_p - h0_p)

        # Global core from ALL nodes (single vector, no per-node prototypes)
        # Note: this includes cold nodes, but it's just attention-pooled —
        # not a per-node soft assignment like TypeAwareCorePrototypes.
        core = self.core(h_r, h_p)

        self._cache_h0 = (h0_r, h0_p)
        return h_r, h_p, core

    def score_edges(self, h_rna, h_prot, core, r_ids, p_ids):
        hr_g = h_rna[r_ids]
        hp_g = h_prot[p_ids]
        h0_r, h0_p = self._cache_h0
        hr0 = h0_r[r_ids]
        hp0 = h0_p[p_ids]

        # Per-edge gate: blend raw → graph
        a = torch.sigmoid(self.pair_gate(
            torch.cat([hr0, hp0, torch.abs(hr0 - hp0), hr0 * hp0], dim=1)))
        hr = hr0 + a * (hr_g - hr0)
        hp = hp0 + a * (hp_g - hp0)

        feat = torch.cat([
            hr, hp, torch.abs(hr - hp), hr * hp,
            hr0, hp0,
            core.unsqueeze(0).expand(hr.size(0), -1)
        ], dim=1)
        logits = self.edge_mlp(feat).squeeze(-1) + self.bilin(hr, hp).squeeze(-1)
        return logits

class BipartiteWeightConv(nn.Module):
    """
    Attention-weighted bipartite message passing.
    w = sigmoid(<Wq u, Wk v> / sqrt(d)), normalized by sum_w per node.
    Cold nodes (degree 0): no edges → out = zeros → residual keeps raw embedding.
    """
    def __init__(self, d, dropout=0.1):
        super().__init__()
        self.Wq_r = nn.Linear(d, d, bias=False)
        self.Wk_p = nn.Linear(d, d, bias=False)
        self.Wv_p = nn.Linear(d, d, bias=False)
        self.Wq_p = nn.Linear(d, d, bias=False)
        self.Wk_r = nn.Linear(d, d, bias=False)
        self.Wv_r = nn.Linear(d, d, bias=False)
        self.drop  = nn.Dropout(dropout)
        self.norm_r = nn.LayerNorm(d)
        self.norm_p = nn.LayerNorm(d)
        self.act = nn.GELU()

    def forward(self, h_rna, h_prot, A_rp):
        A = A_rp.coalesce()
        idx = A.indices()
        r = idx[0]   # (E,)
        p = idx[1]   # (E,)
        n_rna  = h_rna.size(0)
        n_prot = h_prot.size(0)
        d = h_rna.size(1)
        scale = 1.0 / math.sqrt(d)

        # RNA ← Protein messages (attention-weighted)
        q = self.Wq_r(h_rna[r])
        k = self.Wk_p(h_prot[p])
        w = torch.sigmoid((q * k).sum(-1) * scale)
        v = self.drop(self.Wv_p(h_prot[p])) * w.unsqueeze(-1)
        out_r = torch.zeros(n_rna, d, device=h_rna.device)
        out_r.index_add_(0, r, v)
        den_r = torch.zeros(n_rna, device=h_rna.device)
        den_r.index_add_(0, r, w)
        out_r = out_r / den_r.clamp_min(1e-6).unsqueeze(-1)

        # Protein ← RNA messages (attention-weighted)
        q2 = self.Wq_p(h_prot[p])
        k2 = self.Wk_r(h_rna[r])
        w2 = torch.sigmoid((q2 * k2).sum(-1) * scale)
        v2 = self.drop(self.Wv_r(h_rna[r])) * w2.unsqueeze(-1)
        out_p = torch.zeros(n_prot, d, device=h_rna.device)
        out_p.index_add_(0, p, v2)
        den_p = torch.zeros(n_prot, device=h_rna.device)
        den_p.index_add_(0, p, w2)
        out_p = out_p / den_p.clamp_min(1e-6).unsqueeze(-1)

        # Residual + norm (cold nodes: out=0, so h_r2 ≈ norm(h_rna + act(0)))
        h_r2 = self.norm_r(h_rna + self.act(out_r))
        h_p2 = self.norm_p(h_prot + self.act(out_p))
        return h_r2, h_p2

class ProtoGNN_WeightedBipartite(nn.Module):
    def __init__(self, d_rna=640, d_prot=1024, d=256, layers=2, dropout=0.1,
                 hyper_topk=20, edge_drop=0.0):
        super().__init__()
        self.layers = layers
        self.edge_drop = edge_drop

        self.enc_rna  = nn.Sequential(nn.Linear(d_rna, d), nn.GELU(), nn.Dropout(dropout))
        self.enc_prot = nn.Sequential(nn.Linear(d_prot, d), nn.GELU(), nn.Dropout(dropout))
        self.convs = nn.ModuleList([BipartiteWeightConv(d, dropout=dropout)
                                    for _ in range(layers)])

        # Global core (single vector, no prototypes)
        self.core  = TypeAwareCoreFusion(d)
        self.bilin = nn.Bilinear(d, d, 1, bias=False)

        # Edge MLP: [hr, hp, |hr-hp|, hr*hp, core] → logit
        in_edge = d*4 + d
        self.edge_mlp = nn.Sequential(
            nn.Linear(in_edge, 512), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(512, 256),     nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 1)
        )

    def encode_nodes(self, x_rna, x_prot, A_rp, rna2prot, prot2rna):
        h_r = self.enc_rna(x_rna)
        h_p = self.enc_prot(x_prot)
        A = sparse_edge_dropout(A_rp, self.edge_drop, self.training)
        for conv in self.convs:
            h_r, h_p = conv(h_r, h_p, A)
        core = self.core(h_r, h_p)
        return h_r, h_p, core

    def score_edges(self, h_rna, h_prot, core, r_ids, p_ids):
        hr = h_rna[r_ids]
        hp = h_prot[p_ids]
        feat = torch.cat([
            hr, hp, torch.abs(hr - hp), hr * hp,
            core.unsqueeze(0).expand(hr.size(0), -1)
        ], dim=1)
        logits = self.edge_mlp(feat).squeeze(-1) + self.bilin(hr, hp).squeeze(-1)
        return logits

class CoNeighborLayer(nn.Module):
    """
    1-hop + 2-hop (co-neighbor) propagation on bipartite graph.
      1-hop: RNA ← Prot, Prot ← RNA  (standard LightGCN)
      2-hop: RNA ← Prot ← RNA,  Prot ← RNA ← Prot  (co-neighbor = same-type similarity)
    Cold nodes: both hops produce zero messages → residual preserves input.
    """
    def __init__(self, d, dropout=0.1):
        super().__init__()
        self.mlp_r = nn.Sequential(nn.Linear(2*d, d), nn.GELU(), nn.Dropout(dropout))
        self.mlp_p = nn.Sequential(nn.Linear(2*d, d), nn.GELU(), nn.Dropout(dropout))
        self.norm_r = nn.LayerNorm(d)
        self.norm_p = nn.LayerNorm(d)

    def forward(self, h_r, h_p, A):
        # 1-hop: standard bipartite message passing
        r1, p1, _, _ = symm_norm_msgs(A, h_r, h_p)
        # 2-hop: co-neighbor (same-type indirect similarity through opposite type)
        # RNA gets messages from proteins who already aggregated from RNAs
        r2, _, _, _ = symm_norm_msgs(A, h_r, p1)
        # Protein gets messages from RNAs who already aggregated from proteins
        _, p2, _, _ = symm_norm_msgs(A, r1, h_p)

        h_r2 = self.norm_r(h_r + self.mlp_r(torch.cat([r1, r2], dim=1)))
        h_p2 = self.norm_p(h_p + self.mlp_p(torch.cat([p1, p2], dim=1)))
        return h_r2, h_p2

class ProtoGNN_CoNeighbor(nn.Module):
    def __init__(self, d_rna=640, d_prot=1024, d=256, layers=2, dropout=0.1,
                 hyper_topk=20, edge_drop=0.0):
        super().__init__()
        self.layers = layers
        self.edge_drop = edge_drop

        self.enc_rna  = nn.Sequential(nn.Linear(d_rna, d), nn.GELU(), nn.Dropout(dropout))
        self.enc_prot = nn.Sequential(nn.Linear(d_prot, d), nn.GELU(), nn.Dropout(dropout))
        self.blocks = nn.ModuleList([CoNeighborLayer(d, dropout=dropout)
                                     for _ in range(layers)])

        # Global core (single vector, no prototypes)
        self.core  = TypeAwareCoreFusion(d)
        self.bilin = nn.Bilinear(d, d, 1, bias=False)

        # Edge MLP: [hr, hp, |hr-hp|, hr*hp, core] → logit
        in_edge = d*4 + d
        self.edge_mlp = nn.Sequential(
            nn.Linear(in_edge, 512), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(512, 256),     nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 1)
        )

    def encode_nodes(self, x_rna, x_prot, A_rp, rna2prot, prot2rna):
        h_r = self.enc_rna(x_rna)
        h_p = self.enc_prot(x_prot)
        A = sparse_edge_dropout(A_rp, self.edge_drop, self.training)
        for blk in self.blocks:
            h_r, h_p = blk(h_r, h_p, A)
        core = self.core(h_r, h_p)
        return h_r, h_p, core

    def score_edges(self, h_rna, h_prot, core, r_ids, p_ids):
        hr = h_rna[r_ids]
        hp = h_prot[p_ids]
        feat = torch.cat([
            hr, hp, torch.abs(hr - hp), hr * hp,
            core.unsqueeze(0).expand(hr.size(0), -1)
        ], dim=1)
        logits = self.edge_mlp(feat).squeeze(-1) + self.bilin(hr, hp).squeeze(-1)
        return logits

def _l2norm(x, eps=1e-12):
    return x / (x.norm(dim=-1, keepdim=True).clamp_min(eps))

class TypeAwareCorePrototypes(nn.Module):
    def __init__(self, d, K=4, dropout=0.1):
        super().__init__()
        self.K = K
        self.key_r = nn.Linear(d, d, bias=False)
        self.key_p = nn.Linear(d, d, bias=False)
        self.q_r   = nn.Parameter(torch.randn(K, d) * 0.02)
        self.q_p   = nn.Parameter(torch.randn(K, d) * 0.02)
        self.drop  = nn.Dropout(dropout)
        self.core_out = nn.Sequential(
            nn.Linear(2*d, d), nn.GELU(), nn.Dropout(dropout)
        )

    def forward(self, h_r, h_p):
        # h_r: (N_warm_rna, d)  h_p: (N_warm_prot, d)
        Kr = self.key_r(self.drop(h_r))
        Kp = self.key_p(self.drop(h_p))
        ar = torch.softmax(Kr @ self.q_r.t(), dim=0)  # (N_warm_rna, K)
        ap = torch.softmax(Kp @ self.q_p.t(), dim=0)  # (N_warm_prot, K)
        proto_r = ar.t() @ h_r   # (K, d)
        proto_p = ap.t() @ h_p   # (K, d)
        core_global = self.core_out(
            torch.cat([proto_r.mean(0), proto_p.mean(0)], dim=0)
        )
        return proto_r, proto_p, core_global

class ProtoGNN_ProtoContrast(nn.Module):
    """
    ProtoGNN with three inductive fixes applied:
      Fix #1  inner val matches outer split  (in runner, not here)
      Fix #2  cold nodes forced to raw embeddings (gate hard-zeroed)
      Fix #3  prototypes built from warm (training) nodes only
    """
    def __init__(self, d_rna=640, d_prot=1024, d=256, layers=2, dropout=0.1,
                 edge_drop=0.0, K=4, tau=0.2, contrast_w=0.2):
        super().__init__()
        self.layers = layers; self.edge_drop = edge_drop
        self.K = K; self.tau = tau; self.contrast_w = contrast_w

        self.enc_rna  = nn.Sequential(nn.Linear(d_rna,  d), nn.GELU(), nn.Dropout(dropout))
        self.enc_prot = nn.Sequential(nn.Linear(d_prot, d), nn.GELU(), nn.Dropout(dropout))

        self.mix_r  = nn.ModuleList([nn.Sequential(nn.Linear(d,d), nn.GELU(),
                                     nn.Dropout(dropout)) for _ in range(layers)])
        self.mix_p  = nn.ModuleList([nn.Sequential(nn.Linear(d,d), nn.GELU(),
                                     nn.Dropout(dropout)) for _ in range(layers)])
        self.norm_r = nn.ModuleList([nn.LayerNorm(d) for _ in range(layers)])
        self.norm_p = nn.ModuleList([nn.LayerNorm(d) for _ in range(layers)])

        self.deg_gate_r = nn.Sequential(nn.Linear(1,32), nn.GELU(), nn.Linear(32,1))
        self.deg_gate_p = nn.Sequential(nn.Linear(1,32), nn.GELU(), nn.Linear(32,1))

        self.pair_gate = nn.Sequential(
            nn.Linear(d*4, 256), nn.GELU(), nn.Dropout(dropout), nn.Linear(256,1)
        )
        self.proto_core     = TypeAwareCorePrototypes(d, K=K, dropout=dropout)
        self.edge_core_q    = nn.Linear(d, d, bias=False)
        self.edge_core_fuse = nn.Sequential(nn.Linear(2*d,d), nn.GELU(), nn.Dropout(dropout))
        self.bilin          = nn.Bilinear(d, d, 1, bias=False)

        in_edge = (d*4) + (d*2) + d
        self.edge_mlp = nn.Sequential(
            nn.Linear(in_edge, 512), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(512, 256),     nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 1)
        )

    # ── encode ──────────────────────────────────────────────────────────────
    def encode_nodes(self, x_rna, x_prot, A_rp, rna2prot, prot2rna):
        h0_r = self.enc_rna(x_rna)
        h0_p = self.enc_prot(x_prot)
        hg_r, hg_p = h0_r, h0_p
        A = sparse_edge_dropout(A_rp, self.edge_drop, self.training)

        for i in range(self.layers):
            r_msg, p_msg, _, _ = symm_norm_msgs(A, hg_r, hg_p)
            hg_r = self.norm_r[i](hg_r + self.mix_r[i](r_msg))
            hg_p = self.norm_p[i](hg_p + self.mix_p[i](p_msg))

        # degree from ORIGINAL adjacency (not dropped)
        A0    = A_rp.coalesce()
        deg_r = torch.sparse.sum(A0, dim=1).to_dense().clamp_min(0.0)
        deg_p = torch.sparse.sum(A0, dim=0).to_dense().clamp_min(0.0)

        # Fix #2: hard-zero gate for cold nodes → they keep pure raw embedding
        is_cold_r = (deg_r == 0).float().unsqueeze(-1)
        is_cold_p = (deg_p == 0).float().unsqueeze(-1)
        g_r = torch.sigmoid(self.deg_gate_r(torch.log1p(deg_r).unsqueeze(-1)))
        g_p = torch.sigmoid(self.deg_gate_p(torch.log1p(deg_p).unsqueeze(-1)))
        h_r = h0_r + (1.0 - is_cold_r) * g_r * (hg_r - h0_r)
        h_p = h0_p + (1.0 - is_cold_p) * g_p * (hg_p - h0_p)

        # Fix #3: prototypes from warm (training) nodes only
        warm_r = (deg_r > 0)
        warm_p = (deg_p > 0)
        proto_r, proto_p, core_global = self.proto_core(h_r[warm_r], h_p[warm_p])

        self._cache_h0 = (h0_r, h0_p)
        self._proto_r  = proto_r   # (K, d) — warm nodes only
        self._proto_p  = proto_p
        return h_r, h_p, core_global

    # ── edge core from prototypes ────────────────────────────────────────────
    def _edge_core_from_prototypes(self, hr, hp):
        scale = 1.0 / math.sqrt(hr.size(1))
        qr = self.edge_core_q(hr)
        qp = self.edge_core_q(hp)
        ar = torch.softmax((qr @ self._proto_r.t()) * scale, dim=1)
        ap = torch.softmax((qp @ self._proto_p.t()) * scale, dim=1)
        return self.edge_core_fuse(
            torch.cat([ar @ self._proto_r, ap @ self._proto_p], dim=1)
        )

    # ── contrastive score ────────────────────────────────────────────────────
    def _contrastive_score(self, hr, hp):
        tau  = max(self.tau, 1e-6)
        hr_n = _l2norm(hr); hp_n = _l2norm(hp)
        pr   = _l2norm(self._proto_r); pp = _l2norm(self._proto_p)
        sim_pos = (hr_n * hp_n).sum(-1) / tau
        neg_p   = torch.logsumexp((hr_n @ pp.t()) / tau, dim=1)
        neg_r   = torch.logsumexp((hp_n @ pr.t()) / tau, dim=1)
        return sim_pos - 0.5 * (neg_p + neg_r)

    # ── score edges ──────────────────────────────────────────────────────────
    def score_edges(self, h_rna, h_prot, core, r_ids, p_ids):
        hr_g = h_rna[r_ids]; hp_g = h_prot[p_ids]
        h0_r, h0_p = self._cache_h0
        hr0 = h0_r[r_ids];   hp0 = h0_p[p_ids]

        a  = torch.sigmoid(self.pair_gate(
            torch.cat([hr0, hp0, torch.abs(hr0-hp0), hr0*hp0], dim=1)))
        hr = hr0 + a * (hr_g - hr0)
        hp = hp0 + a * (hp_g - hp0)

        feat = torch.cat([
            hr, hp, torch.abs(hr-hp), hr*hp,
            hr0, hp0,
            self._edge_core_from_prototypes(hr, hp)
        ], dim=1)

        logits = (self.edge_mlp(feat).squeeze(-1)
                  + self.bilin(hr, hp).squeeze(-1)
                  + self.contrast_w * self._contrastive_score(hr, hp))
        return logits

def build_neighbour_lists(A_rp, n_rna, n_prot, device):
    """A_rp: sparse (n_rna, n_prot). Returns (rna2prot_list, prot2rna_list)."""
    A = A_rp.coalesce()
    idx = A.indices()                    # (2, E)
    r_ids = idx[0]                       # (E,)
    p_ids = idx[1]                       # (E,)

    # Group protein indices by RNA
    rna2prot = [torch.empty(0, dtype=torch.long, device=device) for _ in range(n_rna)]
    prot2rna = [torch.empty(0, dtype=torch.long, device=device) for _ in range(n_prot)]

    # Vectorised grouping using sorting
    if r_ids.numel() > 0:
        # RNA -> protein
        order_r = torch.argsort(r_ids)
        r_sorted = r_ids[order_r]
        p_sorted = p_ids[order_r]
        # find boundaries where r changes
        unique_r, counts_r = torch.unique_consecutive(r_sorted, return_counts=True)
        # Split p_sorted into chunks corresponding to each unique r
        offsets_r = torch.cat([torch.zeros(1, dtype=torch.long, device=device),
                               counts_r.cumsum(0)])
        for k, r_val in enumerate(unique_r.tolist()):
            rna2prot[r_val] = p_sorted[offsets_r[k]:offsets_r[k+1]]

        # Protein -> RNA
        order_p = torch.argsort(p_ids)
        p_sorted2 = p_ids[order_p]
        r_sorted2 = r_ids[order_p]
        unique_p, counts_p = torch.unique_consecutive(p_sorted2, return_counts=True)
        offsets_p = torch.cat([torch.zeros(1, dtype=torch.long, device=device),
                               counts_p.cumsum(0)])
        for k, p_val in enumerate(unique_p.tolist()):
            prot2rna[p_val] = r_sorted2[offsets_p[k]:offsets_p[k+1]]

    return rna2prot, prot2rna

class BipartiteSAGELayer(nn.Module):
    def __init__(self, d, dropout=0.1, sample_k=25, use_l2norm=True):
        """
        sample_k: int = sample this many neighbours per node during training,
                  None = use ALL neighbours (no sampling).
        use_l2norm: bool = apply F.normalize(..., p=2) per layer.
        """
        super().__init__()
        self.d = d
        self.sample_k = sample_k
        self.use_l2norm = use_l2norm
        self.W_rna  = nn.Linear(2 * d, d)
        self.W_prot = nn.Linear(2 * d, d)
        self.dropout = nn.Dropout(dropout)

    def _sample_and_aggregate(self, h_neighbour, neighbour_lists, n_self):
        d = h_neighbour.size(1)
        device = h_neighbour.device
        agg = torch.zeros(n_self, d, device=device)

        # Use all neighbours when (a) sample_k is None or (b) we're in eval mode
        use_all = (self.sample_k is None) or (not self.training)

        if use_all:
            for i in range(n_self):
                nb = neighbour_lists[i]
                if nb.numel() > 0:
                    agg[i] = h_neighbour[nb].mean(dim=0)
        else:
            for i in range(n_self):
                nb = neighbour_lists[i]
                if nb.numel() == 0:
                    continue
                if nb.numel() <= self.sample_k:
                    chosen = nb
                else:
                    perm = torch.randperm(nb.numel(), device=device)[:self.sample_k]
                    chosen = nb[perm]
                agg[i] = h_neighbour[chosen].mean(dim=0)
        return agg

    def forward(self, h_rna, h_prot, rna2prot, prot2rna):
        agg_for_rna  = self._sample_and_aggregate(h_prot, rna2prot, h_rna.size(0))
        agg_for_prot = self._sample_and_aggregate(h_rna, prot2rna, h_prot.size(0))

        h_rna_new  = F.relu(self.W_rna( self.dropout(torch.cat([h_rna,  agg_for_rna],  dim=1))))
        h_prot_new = F.relu(self.W_prot(self.dropout(torch.cat([h_prot, agg_for_prot], dim=1))))

        if self.use_l2norm:
            h_rna_new  = F.normalize(h_rna_new,  p=2, dim=1)
            h_prot_new = F.normalize(h_prot_new, p=2, dim=1)

        return h_rna_new, h_prot_new

class ProtoGNN_GraphSAGE(nn.Module):
    def __init__(self, d_rna=640, d_prot=1024, d=256, layers=2, dropout=0.1,
                 sample_k=25, edge_drop=0.0, use_l2norm=True):
        super().__init__()
        self.layers = layers
        self.edge_drop = edge_drop
        self.use_l2norm = use_l2norm
        self.sample_k = sample_k

        self.enc_rna  = nn.Sequential(nn.Linear(d_rna,  d), nn.GELU(), nn.Dropout(dropout))
        self.enc_prot = nn.Sequential(nn.Linear(d_prot, d), nn.GELU(), nn.Dropout(dropout))

        self.sage_layers = nn.ModuleList([
            BipartiteSAGELayer(d, dropout=dropout, sample_k=sample_k, use_l2norm=use_l2norm)
            for _ in range(layers)
        ])

        self.pair_gate = nn.Sequential(
            nn.Linear(d * 4, 256), nn.GELU(), nn.Dropout(dropout), nn.Linear(256, 1)
        )
        self.bilin = nn.Bilinear(d, d, 1, bias=False)

        in_edge = (d * 4) + (d * 2)
        self.edge_mlp = nn.Sequential(
            nn.Linear(in_edge, 512), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(512, 256),     nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 1)
        )

    def encode_nodes(self, x_rna, x_prot, A_rp, rna2prot=None, prot2rna=None):
        h0_r = self.enc_rna(x_rna)
        h0_p = self.enc_prot(x_prot)
        n_rna  = x_rna.size(0)
        n_prot = x_prot.size(0)
        device = x_rna.device

        if self.training and self.edge_drop > 0:
            A_dropped = sparse_edge_dropout(A_rp, self.edge_drop, training=True)
        else:
            A_dropped = A_rp

        rna2prot_use, prot2rna_use = build_neighbour_lists(A_dropped, n_rna, n_prot, device)

        h_r, h_p = h0_r, h0_p
        for layer in self.sage_layers:
            h_r, h_p = layer(h_r, h_p, rna2prot_use, prot2rna_use)

        self._cache_h0 = (h0_r, h0_p)
        dummy_core = torch.zeros(1, device=device)
        return h_r, h_p, dummy_core

    def score_edges(self, h_rna, h_prot, core, r_ids, p_ids):
        hr_g = h_rna[r_ids]; hp_g = h_prot[p_ids]
        h0_r, h0_p = self._cache_h0
        hr0 = h0_r[r_ids]; hp0 = h0_p[p_ids]
        a = torch.sigmoid(self.pair_gate(
            torch.cat([hr0, hp0, torch.abs(hr0 - hp0), hr0 * hp0], dim=1)))
        hr = hr0 + a * (hr_g - hr0)
        hp = hp0 + a * (hp_g - hp0)
        feat = torch.cat([hr, hp, torch.abs(hr - hp), hr * hp, hr0, hp0], dim=1)
        return self.edge_mlp(feat).squeeze(-1) + self.bilin(hr, hp).squeeze(-1)


In [ ]:
%%writefile exact_zhmolgraph.py


import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset

def build_unified_graph_features(x_rna, x_prot):
    """Pad both molecule types to one feature width, as in ZHMolGraph's graph files."""
    d = max(x_rna.shape[1], x_prot.shape[1])
    r_pad = np.zeros((x_rna.shape[0], d), dtype=np.float32)
    p_pad = np.zeros((x_prot.shape[0], d), dtype=np.float32)
    r_pad[:, :x_rna.shape[1]] = x_rna
    p_pad[:, :x_prot.shape[1]] = x_prot
    return np.concatenate([r_pad, p_pad], axis=0)

def build_positive_train_adjacency(r_idx, p_idx, y, train_idx, n_rna, n_prot):
    n_nodes = n_rna + n_prot
    adj = [set() for _ in range(n_nodes)]
    pos_rows = np.asarray(train_idx)[y[np.asarray(train_idx)] == 1]
    for r, p in zip(r_idx[pos_rows].tolist(), p_idx[pos_rows].tolist()):
        u, v = int(r), int(n_rna + p)
        adj[u].add(v)
        adj[v].add(u)
    warm = np.asarray([i for i, nei in enumerate(adj) if len(nei) > 0], dtype=np.int64)
    if len(warm) == 0:
        raise ValueError("Training graph has no positive edges.")
    return adj, warm

class SageLayer(nn.Module):
    """Repository-faithful GraphSAGE layer: concat(self, mean-neighbour) → W → ReLU."""
    def __init__(self, input_size, out_size, gcn=False):
        super().__init__()
        self.gcn = gcn
        width = input_size if gcn else 2 * input_size
        self.weight = nn.Parameter(torch.empty(out_size, width))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, self_feats, aggregate_feats):
        combined = aggregate_feats if self.gcn else torch.cat([self_feats, aggregate_feats], dim=1)
        return F.relu(self.weight.mm(combined.t())).t()

class OriginalGraphSage(nn.Module):
    """Modernized copy of the repository's two-layer sampled GraphSAGE.

    Python-set sampling is converted to list sampling for Python 3.11 compatibility;
    the mathematical behavior is unchanged.
    """
    def __init__(self, num_layers, input_size, out_size, raw_features, adj_lists,
                 gcn=False, agg_func="MEAN", num_sample=10):
        super().__init__()
        self.num_layers = num_layers
        self.input_size = input_size
        self.out_size = out_size
        self.gcn = gcn
        self.agg_func = agg_func
        self.num_sample = num_sample
        self.adj_lists = adj_lists
        # Features are fixed input, not trainable and not part of checkpoints.
        self.register_buffer("raw_features", raw_features, persistent=False)
        for index in range(1, num_layers + 1):
            layer_size = out_size if index != 1 else input_size
            setattr(self, f"sage_layer{index}", SageLayer(layer_size, out_size, gcn=gcn))

    def _get_unique_neighs_list(self, nodes):
        sampled = []
        for node in nodes:
            neigh = self.adj_lists[int(node)]
            if self.num_sample is not None and len(neigh) >= self.num_sample:
                neigh = set(random.sample(list(neigh), self.num_sample))
            else:
                neigh = set(neigh)
            neigh.add(int(node))  # repository includes self before aggregation
            sampled.append(neigh)
        unique_list = sorted(set().union(*sampled))
        unique_map = {n: i for i, n in enumerate(unique_list)}
        return sampled, unique_map, unique_list

    @staticmethod
    def _nodes_map(nodes, pre_neighs):
        _, _, layer_nodes_dict = pre_neighs
        return [layer_nodes_dict[int(x)] for x in nodes]

    def aggregate(self, nodes, pre_hidden_embs, pre_neighs):
        unique_nodes_list, samp_neighs, unique_nodes = pre_neighs
        if len(pre_hidden_embs) == len(unique_nodes):
            embed_matrix = pre_hidden_embs
        else:
            idx = torch.as_tensor(unique_nodes_list, dtype=torch.long, device=pre_hidden_embs.device)
            embed_matrix = pre_hidden_embs[idx]

        mask = torch.zeros(
            (len(samp_neighs), len(unique_nodes)),
            dtype=embed_matrix.dtype,
            device=embed_matrix.device,
        )
        row_indices, col_indices = [], []
        for i, neigh in enumerate(samp_neighs):
            row_indices.extend([i] * len(neigh))
            col_indices.extend([unique_nodes[n] for n in neigh])
        mask[row_indices, col_indices] = 1.0

        if self.agg_func == "MEAN":
            mask = mask / mask.sum(1, keepdim=True).clamp_min(1.0)
            return mask.mm(embed_matrix)
        if self.agg_func == "MAX":
            outputs = []
            for i in range(len(samp_neighs)):
                cols = mask[i].nonzero(as_tuple=False).squeeze(1)
                outputs.append(embed_matrix[cols].max(dim=0).values)
            return torch.stack(outputs)
        raise ValueError(self.agg_func)

    def forward(self, nodes_batch):
        lower_layer_nodes = [int(x) for x in list(nodes_batch)]
        nodes_batch_layers = [(lower_layer_nodes,)]
        for _ in range(self.num_layers):
            lower_samp_neighs, lower_nodes_dict, lower_layer_nodes = self._get_unique_neighs_list(lower_layer_nodes)
            nodes_batch_layers.insert(0, (lower_layer_nodes, lower_samp_neighs, lower_nodes_dict))

        pre_hidden_embs = self.raw_features
        for index in range(1, self.num_layers + 1):
            nb = nodes_batch_layers[index][0]
            pre_neighs = nodes_batch_layers[index - 1]
            aggregate_feats = self.aggregate(nb, pre_hidden_embs, pre_neighs)
            sage_layer = getattr(self, f"sage_layer{index}")
            if index > 1:
                nb = self._nodes_map(nb, pre_neighs)
            cur_hidden = sage_layer(pre_hidden_embs[nb], aggregate_feats)
            pre_hidden_embs = cur_hidden
        return pre_hidden_embs

class UnsupervisedGraphLoss:
    """Repository objective with modern-safe sampling and empty-batch handling."""
    def __init__(self, adj_lists, train_nodes, device, q=10, n_walks=6,
                 walk_len=1, negative_hops=3, margin=3.0):
        self.Q = q
        self.N_WALKS = n_walks
        self.WALK_LEN = walk_len
        self.N_WALK_LEN = negative_hops
        self.MARGIN = margin
        self.adj_lists = adj_lists
        self.train_nodes = [int(x) for x in train_nodes]
        self.train_set = set(self.train_nodes)
        self.device = device
        self.node_positive_pairs = {}
        self.node_negative_pairs = {}
        self.unique_nodes_batch = []

    def _positive_nodes(self, nodes):
        pairs = []
        self.node_positive_pairs = {}
        for node in nodes:
            node = int(node)
            cur = []
            if len(self.adj_lists[node]) > 0:
                for _ in range(self.N_WALKS):
                    curr = node
                    for _ in range(self.WALK_LEN):
                        next_node = random.choice(list(self.adj_lists[curr]))
                        if next_node != node and next_node in self.train_set:
                            pairs.append((node, next_node))
                            cur.append((node, next_node))
                        curr = next_node
            self.node_positive_pairs[node] = cur
        return pairs

    def _negative_nodes(self, nodes, num_neg):
        pairs = []
        self.node_negative_pairs = {}
        for node in nodes:
            node = int(node)
            neighbours = {node}
            frontier = {node}
            for _ in range(self.N_WALK_LEN):
                current = set()
                for outer in frontier:
                    current |= self.adj_lists[int(outer)]
                frontier = current - neighbours
                neighbours |= current
            far = list(self.train_set - neighbours)
            chosen = random.sample(far, num_neg) if num_neg < len(far) else far
            cur = [(node, int(n)) for n in chosen]
            pairs.extend(cur)
            self.node_negative_pairs[node] = cur
        return pairs

    def extend_nodes(self, nodes, num_neg):
        nodes = [int(x) for x in nodes]
        pos = self._positive_nodes(nodes)
        neg = self._negative_nodes(nodes, num_neg)
        unique = sorted(set(i for pair in (pos + neg) for i in pair))
        # Every target warm node should have at least one sampled positive and negative.
        if not unique or not set(nodes).issubset(set(unique)):
            return None
        self.unique_nodes_batch = unique
        return np.asarray(unique, dtype=np.int64)

    def get_loss_sage(self, embeddings, nodes):
        node2index = {n: i for i, n in enumerate(self.unique_nodes_batch)}
        node_losses = []
        for node, pos_pairs in self.node_positive_pairs.items():
            neg_pairs = self.node_negative_pairs.get(node, [])
            if not pos_pairs or not neg_pairs:
                continue
            pi0 = torch.as_tensor([node2index[a] for a, _ in pos_pairs], device=self.device)
            pi1 = torch.as_tensor([node2index[b] for _, b in pos_pairs], device=self.device)
            ni0 = torch.as_tensor([node2index[a] for a, _ in neg_pairs], device=self.device)
            ni1 = torch.as_tensor([node2index[b] for _, b in neg_pairs], device=self.device)
            pos_score = torch.log(torch.sigmoid(F.cosine_similarity(embeddings[pi0], embeddings[pi1])) + 1e-12)
            neg_score = self.Q * torch.mean(
                torch.log(torch.sigmoid(-F.cosine_similarity(embeddings[ni0], embeddings[ni1])) + 1e-12)
            )
            node_losses.append(torch.mean(-pos_score - neg_score).view(1))
        return torch.cat(node_losses).mean() if node_losses else None

    def get_loss_margin(self, embeddings, nodes):
        node2index = {n: i for i, n in enumerate(self.unique_nodes_batch)}
        node_losses = []
        for node, pos_pairs in self.node_positive_pairs.items():
            neg_pairs = self.node_negative_pairs.get(node, [])
            if not pos_pairs or not neg_pairs:
                continue
            pi0 = torch.as_tensor([node2index[a] for a, _ in pos_pairs], device=self.device)
            pi1 = torch.as_tensor([node2index[b] for _, b in pos_pairs], device=self.device)
            ni0 = torch.as_tensor([node2index[a] for a, _ in neg_pairs], device=self.device)
            ni1 = torch.as_tensor([node2index[b] for _, b in neg_pairs], device=self.device)
            pos = torch.log(torch.sigmoid(F.cosine_similarity(embeddings[pi0], embeddings[pi1])) + 1e-12).min()
            neg = torch.log(torch.sigmoid(F.cosine_similarity(embeddings[ni0], embeddings[ni1])) + 1e-12).max()
            node_losses.append(torch.clamp(neg - pos + self.MARGIN, min=0.0).view(1))
        return torch.cat(node_losses).mean() if node_losses else None

def conv_pool_flat_dim(input_len: int) -> int:
    # Conv1d kernel=3, no padding: L -> L-2; AvgPool1d(2,2): floor((L-2)/2); 8 channels.
    after_conv = input_len - 2
    after_pool = math.floor((after_conv - 2) / 2 + 1)
    if after_pool <= 0:
        raise ValueError(f"Input length {input_len} is too small for VecNN Conv+Pool.")
    return 8 * after_pool

class VecNN(nn.Module):
    """ZHMolGraph VecNN, returning logits for stable BCEWithLogitsLoss."""
    def __init__(self, protein_input_len: int, rna_input_len: int):
        super().__init__()
        self.conv1d_target = nn.Conv1d(1, 8, kernel_size=3)
        self.conv1d_rna = nn.Conv1d(1, 8, kernel_size=3)
        self.pool = nn.AvgPool1d(kernel_size=2, stride=2)
        self.target_layer = nn.Sequential(
            nn.ReLU(),
            nn.Linear(conv_pool_flat_dim(protein_input_len), 2048),
            nn.ReLU(),
        )
        self.rna_layer = nn.Sequential(
            nn.ReLU(),
            nn.Linear(conv_pool_flat_dim(rna_input_len), 2048),
            nn.ReLU(),
        )
        self.concat_layer = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(4096, 512),
            nn.ReLU(),
        )
        self.output_layer = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 1),
        )

    def forward(self, protein_input, rna_input):
        hp = self.pool(self.conv1d_target(protein_input.unsqueeze(1))).flatten(1)
        hr = self.pool(self.conv1d_rna(rna_input.unsqueeze(1))).flatten(1)
        hp = self.target_layer(hp)
        hr = self.rna_layer(hr)
        return self.output_layer(self.concat_layer(torch.cat([hp, hr], dim=1))).squeeze(1)

class EdgeIndexDataset(Dataset):
    def __init__(self, r_idx, p_idx, y, row_idx):
        self.r = torch.as_tensor(r_idx[row_idx], dtype=torch.long)
        self.p = torch.as_tensor(p_idx[row_idx], dtype=torch.long)
        self.y = torch.as_tensor(y[row_idx], dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.r[i], self.p[i], self.y[i]


## 2. Benchmark engine

Shared data loading, splitting, training, evaluation, and persistence utilities.


In [ ]:
%%writefile journal_benchmark.py

from __future__ import annotations

import gc
import json
import math
import os
import pickle as pkl
import random
import re
import shutil
import time
import zipfile
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Callable, Iterable, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import (
    average_precision_score,
    matthews_corrcoef,
    roc_auc_score,
)
from sklearn.model_selection import KFold, StratifiedKFold, StratifiedGroupKFold, train_test_split

import exact_pair_models as pm
import exact_zhmolgraph as zh


# =============================================================================
# Reproducibility and configuration
# =============================================================================

def seed_all(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


@dataclass(frozen=True)
class EvaluationProfile:
    name: str
    pair_conflict_policy: str
    drop_duplicate_pair_label_rows: bool
    protocol_matched_validation: bool
    include_validation_edges_in_graph: bool
    zhmolgraph_early_stopping: bool
    zhmolgraph_vec_epochs: int
    description: str


PROFILES: dict[str, EvaluationProfile] = {
    "legacy_repro": EvaluationProfile(
        name="legacy_repro",
        pair_conflict_policy="keep_raw",
        drop_duplicate_pair_label_rows=False,
        protocol_matched_validation=False,
        include_validation_edges_in_graph=True,
        zhmolgraph_early_stopping=False,
        zhmolgraph_vec_epochs=120,
        description=(
            "Legacy-oriented reproduction: raw contradictory/duplicate rows are retained, "
            "inner validation is an edge split, validation positives are permitted in the "
            "GraphSAGE graph, and VecNN trains for the full 120 epochs."
        ),
    ),
    "clean_120": EvaluationProfile(
        name="clean_120",
        pair_conflict_policy="drop_both",
        drop_duplicate_pair_label_rows=True,
        protocol_matched_validation=True,
        include_validation_edges_in_graph=False,
        zhmolgraph_early_stopping=False,
        zhmolgraph_vec_epochs=120,
        description=(
            "Preferred thesis benchmark: contradictory exact sequence-pairs are removed, "
            "duplicate pair-label rows are removed, validation matches the outer protocol, "
            "the graph uses positive inner-training edges only, and VecNN trains for 120 epochs."
        ),
    ),
    "clean_early": EvaluationProfile(
        name="clean_early",
        pair_conflict_policy="drop_both",
        drop_duplicate_pair_label_rows=True,
        protocol_matched_validation=True,
        include_validation_edges_in_graph=False,
        zhmolgraph_early_stopping=True,
        zhmolgraph_vec_epochs=120,
        description=(
            "The earlier clean benchmark with validation-AUROC early stopping. Retained only "
            "for comparison with already completed runs; clean_120 is the preferred rerun."
        ),
    ),
}


def is_zhmolgraph_model(model_name: str) -> bool:
    return model_name in {"ZHMolGraph", "ZHMolGraph-Best120"}


@dataclass
class PairModelTrainConfig:
    max_epochs: int = 50
    early_stop_patience: int = 5
    learning_rate: float = 1e-3
    batch_size: int = 4096
    hidden_dim: int = 256
    pair_dropout: float = 0.2
    graph_dropout: float = 0.1
    n_heads: int = 4
    graphsage_layers: int = 2
    graphsage_sample_k: int = 25


@dataclass
class ZHMolGraphTrainConfig:
    sage_layers: int = 2
    sage_hidden: int = 100
    sage_neighbours: int = 10
    sage_batch_size: int = 20
    sage_lr: float = 0.65
    sage_grad_clip: float = 5.0
    sage_epochs: int = 10
    sage_num_negative: int = 10
    sage_loss: str = "normal"
    vec_batch_size: int = 128
    vec_lr: float = 1e-3
    vec_weight_decay: float = 0.0
    vec_patience: int = 15
    vec_min_epochs: int = 20
    use_amp: bool = True


@dataclass
class ExperimentPlan:
    name: str
    profile: str
    datasets: tuple[str, ...]
    protocols: tuple[str, ...]
    models: tuple[str, ...]
    seeds: tuple[int, ...]
    n_splits: int = 5
    val_fraction: float = 0.10
    folds: Optional[tuple[int, ...]] = None
    run_thenovel: bool = False
    resume: bool = True
    overwrite: bool = False


PRIORITY_PRESETS: dict[str, ExperimentPlan] = {
    "P1_LEGACY_REPRO": ExperimentPlan(
        name="P1_LEGACY_REPRO",
        profile="legacy_repro",
        datasets=("NPInter2", "RPI7317"),
        protocols=("edge",),
        models=("ZHMolGraph",),
        seeds=(64,),
        run_thenovel=True,
    ),
    "P2_CLEAN_PAIRED": ExperimentPlan(
        name="P2_CLEAN_PAIRED",
        profile="clean_120",
        datasets=("NPInter2", "RPI7317"),
        protocols=("edge", "rna_cold", "prot_cold"),
        models=("PairMLP", "PairMLP-Cross", "GraphSAGE-2L", "ZHMolGraph"),
        seeds=(64,),
        run_thenovel=True,
    ),
    "P3_MULTI_SEED": ExperimentPlan(
        name="P3_MULTI_SEED",
        profile="clean_120",
        datasets=("NPInter2", "RPI7317"),
        protocols=("rna_cold", "prot_cold"),
        models=("PairMLP", "PairMLP-Cross", "GraphSAGE-2L", "ZHMolGraph"),
        seeds=(42, 64, 128),
        run_thenovel=False,
    ),
    "P4_ZH_BEST120_SEED64": ExperimentPlan(
        name="P4_ZH_BEST120_SEED64",
        profile="clean_120",
        datasets=("NPInter2", "RPI7317"),
        protocols=("edge", "rna_cold", "prot_cold"),
        models=("ZHMolGraph-Best120",),
        seeds=(64,),
        run_thenovel=True,
    ),
    "P5_CENTRAL_MULTI_SEED": ExperimentPlan(
        name="P5_CENTRAL_MULTI_SEED",
        profile="clean_120",
        datasets=("NPInter2", "RPI7317"),
        protocols=("rna_cold", "prot_cold"),
        models=("PairMLP-Cross", "ZHMolGraph-Best120"),
        seeds=(64, 2026, 3407),
        run_thenovel=False,
    ),
    "P6_EXTENDED_MULTI_SEED": ExperimentPlan(
        name="P6_EXTENDED_MULTI_SEED",
        profile="clean_120",
        datasets=("NPInter2", "RPI7317"),
        protocols=("rna_cold", "prot_cold"),
        models=("PairMLP", "PairMLP-Cross", "GraphSAGE-2L", "ZHMolGraph-Best120"),
        seeds=(64, 2026, 3407),
        run_thenovel=False,
    ),
}


# =============================================================================
# Data loading
# =============================================================================

@dataclass
class DatasetBundle:
    name: str
    profile: str
    x_rna: np.ndarray
    x_prot: np.ndarray
    r_idx: np.ndarray
    p_idx: np.ndarray
    y: np.ndarray
    rows: pd.DataFrame
    rna_keys: list[str]
    prot_keys: list[str]
    meta: dict[str, Any]


def canonical_sequence(value: Any) -> str:
    return re.sub(r"\s+", "", str(value)).upper()


def pick_col(df: pd.DataFrame, candidates: Iterable[str], label: str) -> str:
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(f"Could not find {label}. Tried {list(candidates)}; got {list(df.columns)}")


def _find_file(data_root: Path, filename: str) -> Path:
    direct = data_root / filename
    if direct.exists():
        return direct
    matches = list(data_root.rglob(filename))
    if len(matches) == 1:
        return matches[0]
    if not matches:
        raise FileNotFoundError(f"Could not find {filename!r} under {data_root}")
    raise RuntimeError(f"Found multiple files named {filename!r}: {matches}")


def _deduplicate_embedding_table(
    df: pd.DataFrame, seq_col: str, emb_col: str, side: str
) -> tuple[list[str], np.ndarray]:
    work = df[[seq_col, emb_col]].copy()
    work["_key"] = work[seq_col].map(canonical_sequence)
    keep: list[tuple[str, np.ndarray]] = []
    inconsistent: list[str] = []
    for key, group in work.groupby("_key", sort=False):
        vectors = [np.asarray(v, dtype=np.float32) for v in group[emb_col].tolist()]
        ref = vectors[0]
        if any(v.shape != ref.shape or not np.allclose(v, ref, rtol=1e-5, atol=1e-6) for v in vectors[1:]):
            inconsistent.append(key)
        keep.append((key, ref))
    if inconsistent:
        raise ValueError(
            f"{side}: duplicate exact sequences have inconsistent embeddings; first={inconsistent[0][:30]}"
        )
    keys = [k for k, _ in keep]
    matrix = np.stack([v for _, v in keep]).astype(np.float32)
    if not np.isfinite(matrix).all():
        raise ValueError(f"{side}: embedding matrix contains NaN/Inf")
    return keys, matrix



def _accession_prefix(value: Any) -> str:
    """Return the accession prefix used to join NPInter5.xlsx to the sequence CSV."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return ""
    text = str(value).strip()
    return text.split("-", 1)[0].strip()


def _load_npinter5_balanced_interactions(
    data_root: Path,
    positive_csv_path: Path,
    audit_dir: Path,
) -> tuple[pd.DataFrame, Path]:
    """Load the balanced NPInter5/TheNovel labels.

    Important:
      - dataset_RPI_NPInter5_RP.csv contains the positive interaction/sequence lookup.
      - NPInter5.xlsx contains the balanced positive + negative evaluation pairs.
    Reading the CSV as the label table produces a one-class dataset, which makes
    binary protein-cold training mathematically impossible.
    """
    xlsx_path = _find_file(data_root, "NPInter5.xlsx")
    positive_df = pd.read_csv(positive_csv_path)
    label_df = pd.read_excel(xlsx_path)

    rna_name_csv = pick_col(
        positive_df,
        ["RNA names", "rna names", "RNA_name", "RNA"],
        "NPInter5 CSV RNA accession",
    )
    prot_name_csv = pick_col(
        positive_df,
        ["Protein names", "protein names", "Protein_name", "Protein"],
        "NPInter5 CSV protein accession",
    )
    rna_seq_csv = pick_col(
        positive_df,
        ["RNA_aa_code", "rna_seq", "RNA_seq"],
        "NPInter5 CSV RNA sequence",
    )
    prot_seq_csv = pick_col(
        positive_df,
        ["target_aa_code", "protein_seq", "prot_seq"],
        "NPInter5 CSV protein sequence",
    )

    lookup = positive_df.copy()
    lookup["_rna_accession"] = lookup[rna_name_csv].map(_accession_prefix)
    lookup["_prot_accession"] = lookup[prot_name_csv].map(_accession_prefix)

    rna_accession_to_sequence = (
        lookup[["_rna_accession", rna_seq_csv]]
        .dropna()
        .drop_duplicates("_rna_accession")
        .set_index("_rna_accession")[rna_seq_csv]
        .map(canonical_sequence)
        .to_dict()
    )
    prot_accession_to_sequence = (
        lookup[["_prot_accession", prot_seq_csv]]
        .dropna()
        .drop_duplicates("_prot_accession")
        .set_index("_prot_accession")[prot_seq_csv]
        .map(canonical_sequence)
        .to_dict()
    )

    rna_name_xlsx = pick_col(
        label_df,
        ["RNA names", "rna names", "RNA_name", "RNA"],
        "NPInter5 XLSX RNA accession",
    )
    prot_name_xlsx = pick_col(
        label_df,
        ["Protein names", "protein names", "Protein_name", "Protein"],
        "NPInter5 XLSX protein accession",
    )
    label_col = pick_col(
        label_df,
        ["Labels", "labels", "Label", "Y", "y"],
        "NPInter5 XLSX label",
    )

    records: list[dict[str, Any]] = []
    missing_records: list[dict[str, Any]] = []

    for original_row, row in label_df.iterrows():
        rna_accession = _accession_prefix(row[rna_name_xlsx])
        prot_accession = _accession_prefix(row[prot_name_xlsx])
        rna_sequence = rna_accession_to_sequence.get(rna_accession)
        prot_sequence = prot_accession_to_sequence.get(prot_accession)

        if rna_sequence is None or prot_sequence is None:
            missing_records.append({
                "original_row": int(original_row),
                "rna_accession": rna_accession,
                "protein_accession": prot_accession,
                "missing_rna_sequence": rna_sequence is None,
                "missing_protein_sequence": prot_sequence is None,
                "y": int(row[label_col]),
            })
            continue

        records.append({
            "original_row": int(original_row),
            "rna_sequence": rna_sequence,
            "protein_sequence": prot_sequence,
            "y": int(row[label_col]),
        })

    if missing_records:
        pd.DataFrame(missing_records).to_csv(
            audit_dir / "npinter5_xlsx_rows_missing_accession_lookup.csv",
            index=False,
        )

    work = pd.DataFrame(records)
    if work.empty:
        raise RuntimeError(
            "NPInter5 balanced loader matched zero XLSX rows to the sequence lookup."
        )

    class_counts = work["y"].value_counts().to_dict()
    if set(work["y"].unique()) != {0, 1}:
        raise RuntimeError(
            "NPInter5.xlsx did not produce a binary dataset after joining. "
            f"Class counts={class_counts}."
        )

    print(
        "[NPInter5 balanced loader] "
        f"XLSX rows={len(label_df)}, matched={len(work)}, "
        f"positive={(work['y'] == 1).sum()}, negative={(work['y'] == 0).sum()}, "
        f"unmatched={len(missing_records)}"
    )

    return work, xlsx_path


def load_dataset(
    data_root: str | Path,
    dataset_name: str,
    profile_name: str,
    output_root: str | Path,
    rna_tag: str = "rnafm",
    protein_tag: str = "proteinprottrans",
) -> DatasetBundle:
    profile = PROFILES[profile_name]
    data_root = Path(data_root)
    output_root = Path(output_root)
    audit_dir = output_root / profile_name / dataset_name / "data_audit"
    audit_dir.mkdir(parents=True, exist_ok=True)

    rna_path = _find_file(data_root, f"RPI_{dataset_name}_{rna_tag}_embed_normal.pkl")
    prot_path = _find_file(data_root, f"RPI_{dataset_name}_{protein_tag}_embed_normal.pkl")
    positive_csv_path = _find_file(data_root, f"dataset_RPI_{dataset_name}_RP.csv")

    with rna_path.open("rb") as handle:
        rna_df = pkl.load(handle)
    with prot_path.open("rb") as handle:
        prot_df = pkl.load(handle)

    r_seq = pick_col(rna_df, ["RNA_aa_code", "RNA_seq", "rna_seq"], "RNA sequence")
    p_seq = pick_col(prot_df, ["target_aa_code", "protein_seq", "prot_seq"], "protein sequence")
    r_emb = pick_col(rna_df, ["normalized_embeddings", "embeddings", "embed"], "RNA embedding")
    p_emb = pick_col(prot_df, ["normalized_embeddings", "embeddings", "embed"], "protein embedding")

    rna_keys, x_rna = _deduplicate_embedding_table(rna_df, r_seq, r_emb, "RNA")
    prot_keys, x_prot = _deduplicate_embedding_table(prot_df, p_seq, p_emb, "Protein")
    rna_map = {key: i for i, key in enumerate(rna_keys)}
    prot_map = {key: i for i, key in enumerate(prot_keys)}

    if dataset_name.lower() == "npinter5":
        work, label_source_path = _load_npinter5_balanced_interactions(
            data_root,
            positive_csv_path,
            audit_dir,
        )
    else:
        interaction_df = pd.read_csv(positive_csv_path)
        ir = pick_col(
            interaction_df,
            ["RNA_aa_code", "RNA", "rna", "rna_seq"],
            "interaction RNA",
        )
        ip = pick_col(
            interaction_df,
            ["target_aa_code", "Protein", "protein", "prot_seq"],
            "interaction protein",
        )
        iy = pick_col(
            interaction_df,
            ["Y", "label", "Label", "y", "Labels"],
            "label",
        )
        work = pd.DataFrame({
            "original_row": np.arange(len(interaction_df), dtype=np.int64),
            "rna_sequence": interaction_df[ir].map(canonical_sequence),
            "protein_sequence": interaction_df[ip].map(canonical_sequence),
            "y": interaction_df[iy].astype(np.int64),
        })
        label_source_path = positive_csv_path

    work["r_idx"] = work["rna_sequence"].map(rna_map)
    work["p_idx"] = work["protein_sequence"].map(prot_map)

    missing = work["r_idx"].isna() | work["p_idx"].isna()
    if missing.any():
        work.loc[missing].to_csv(audit_dir / "rows_missing_embeddings.csv", index=False)
        work = work.loc[~missing].copy()

    work["r_idx"] = work["r_idx"].astype(np.int64)
    work["p_idx"] = work["p_idx"].astype(np.int64)

    if not set(work["y"].unique()).issubset({0, 1}):
        raise ValueError(f"{dataset_name}: labels are not binary")

    # Fail immediately with a useful diagnosis instead of reaching the split code.
    final_class_counts = work["y"].value_counts().to_dict()
    if len(final_class_counts) < 2:
        raise RuntimeError(
            f"{dataset_name}: the loaded interaction table contains one class only: "
            f"{final_class_counts}. For NPInter5, labels must come from NPInter5.xlsx, "
            "not from the positive-only dataset_RPI_NPInter5_RP.csv."
        )

    conflicts = work.groupby(["r_idx", "p_idx"])["y"].nunique()
    conflicts = conflicts[conflicts > 1]
    if len(conflicts):
        pair_index = pd.MultiIndex.from_frame(work[["r_idx", "p_idx"]])
        conflict_mask = pair_index.isin(conflicts.index)
        work.loc[conflict_mask].to_csv(
            audit_dir / "conflicting_sequence_pairs.csv",
            index=False,
        )
        if profile.pair_conflict_policy == "drop_both":
            work = work.loc[~conflict_mask].copy()
        elif profile.pair_conflict_policy == "keep_positive":
            work = work.loc[~conflict_mask | (work["y"] == 1)].copy()
        elif profile.pair_conflict_policy == "error":
            raise ValueError(f"{dataset_name}: conflicting exact sequence-pair labels")
        elif profile.pair_conflict_policy != "keep_raw":
            raise ValueError(profile.pair_conflict_policy)

    duplicate_count = int(work.duplicated(["r_idx", "p_idx", "y"]).sum())
    if duplicate_count:
        work.loc[
            work.duplicated(["r_idx", "p_idx", "y"], keep=False)
        ].to_csv(
            audit_dir / "duplicate_pair_label_rows.csv",
            index=False,
        )
    if profile.drop_duplicate_pair_label_rows:
        work = work.drop_duplicates(["r_idx", "p_idx", "y"], keep="first").copy()

    work = work.reset_index(drop=True)
    work["row_id"] = np.arange(len(work), dtype=np.int64)
    work.to_csv(audit_dir / "analysis_rows.csv", index=False)

    final_class_counts = work["y"].value_counts().to_dict()
    if len(final_class_counts) < 2:
        raise RuntimeError(
            f"{dataset_name}: cleaning removed one class. Final counts={final_class_counts}."
        )

    meta = {
        "dataset": dataset_name,
        "profile": profile_name,
        "rna_embedding_path": str(rna_path),
        "protein_embedding_path": str(prot_path),
        "sequence_lookup_path": str(positive_csv_path),
        "interaction_label_path": str(label_source_path),
        "n_rna": int(len(rna_keys)),
        "n_prot": int(len(prot_keys)),
        "d_rna": int(x_rna.shape[1]),
        "d_prot": int(x_prot.shape[1]),
        "n_rows": int(len(work)),
        "n_positive": int((work["y"] == 1).sum()),
        "n_negative": int((work["y"] == 0).sum()),
        "conflicting_pairs_detected": int(len(conflicts)),
        "duplicate_pair_label_rows_detected": duplicate_count,
        "profile_config": asdict(profile),
    }
    (audit_dir / "dataset_meta.json").write_text(json.dumps(meta, indent=2))

    return DatasetBundle(
        name=dataset_name,
        profile=profile_name,
        x_rna=x_rna,
        x_prot=x_prot,
        r_idx=work["r_idx"].to_numpy(np.int64),
        p_idx=work["p_idx"].to_numpy(np.int64),
        y=work["y"].to_numpy(np.int64),
        rows=work,
        rna_keys=rna_keys,
        prot_keys=prot_keys,
        meta=meta,
    )


# =============================================================================
# Metrics and split persistence
# =============================================================================

def sigmoid_np(values: np.ndarray) -> np.ndarray:
    x = np.asarray(values, dtype=np.float64)
    out = np.empty_like(x)
    positive = x >= 0
    out[positive] = 1.0 / (1.0 + np.exp(-x[positive]))
    exp_x = np.exp(x[~positive])
    out[~positive] = exp_x / (1.0 + exp_x)
    return out


def confusion_counts(y_true: np.ndarray, y_pred: np.ndarray) -> tuple[int, int, int, int]:
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    return tp, tn, fp, fn


def mcc_from_counts(tp: int, tn: int, fp: int, fn: int) -> float:
    denominator = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return float((tp * tn - fp * fn) / denominator) if denominator else 0.0


def best_threshold_by_mcc(y_true: np.ndarray, logits: np.ndarray, grid_size: int = 1000) -> tuple[float, float]:
    probabilities = sigmoid_np(logits)
    best_threshold, best_mcc = 0.5, -2.0
    for threshold in np.linspace(0.0, 1.0, grid_size + 1):
        tp, tn, fp, fn = confusion_counts(y_true, probabilities >= threshold)
        score = mcc_from_counts(tp, tn, fp, fn)
        if score > best_mcc:
            best_threshold, best_mcc = float(threshold), float(score)
    return best_threshold, best_mcc


def metrics_from_logits(y_true: np.ndarray, logits: np.ndarray, threshold: float) -> dict[str, float | int]:
    y_true = np.asarray(y_true, dtype=int)
    probabilities = sigmoid_np(logits)
    predictions = (probabilities >= threshold).astype(int)
    tp, tn, fp, fn = confusion_counts(y_true, predictions)
    two_classes = len(np.unique(y_true)) == 2
    return {
        "ACC": float((tp + tn) / max(tp + tn + fp + fn, 1)),
        "SEN": float(tp / max(tp + fn, 1)),
        "SPE": float(tn / max(tn + fp, 1)),
        "PRE": float(tp / max(tp + fp, 1)),
        "F1": float(2 * tp / max(2 * tp + fp + fn, 1)),
        "MCC": mcc_from_counts(tp, tn, fp, fn),
        "AUROC": float(roc_auc_score(y_true, probabilities)) if two_classes else float("nan"),
        "AUPRC": float(average_precision_score(y_true, probabilities)) if two_classes else float("nan"),
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn,
    }


def _outer_splits(
    bundle: DatasetBundle,
    protocol: str,
    n_splits: int,
    seed: int,
):
    """splitter: one standardized strategy for every dataset.

    * edge: StratifiedKFold over rows.
    * rna_cold/prot_cold: StratifiedGroupKFold over rows with the held-out
      entity as the grouping variable. Candidate random states are searched
      and the valid split with the best fold-size/prevalence balance is kept.
    """
    rows = np.arange(len(bundle.y), dtype=np.int64)
    y = np.asarray(bundle.y, dtype=np.int64)
    if len(np.unique(y)) < 2:
        raise RuntimeError(f"{bundle.name}: one-class dataset: {pd.Series(y).value_counts().to_dict()}")

    if protocol == "edge":
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        return [(tr.astype(np.int64), te.astype(np.int64)) for tr, te in skf.split(rows, y)]

    if protocol not in {"rna_cold", "prot_cold"}:
        raise ValueError(protocol)
    groups = bundle.r_idx if protocol == "rna_cold" else bundle.p_idx
    unique_groups = np.unique(groups)
    if len(unique_groups) < n_splits:
        raise RuntimeError(f"{bundle.name}/{protocol}: only {len(unique_groups)} groups for {n_splits} folds")

    stats = pd.DataFrame({"g": groups, "y": y}).groupby("g")["y"].agg(["count","sum"])
    stats["neg"] = stats["count"] - stats["sum"]
    pos_groups = int((stats["sum"] > 0).sum())
    neg_groups = int((stats["neg"] > 0).sum())
    if pos_groups < n_splits or neg_groups < n_splits:
        raise RuntimeError(
            f"{bundle.name}/{protocol}: {n_splits}-fold binary group CV infeasible; "
            f"groups={len(stats)}, positive-bearing={pos_groups}, negative-bearing={neg_groups}"
        )

    best = None
    target_n = len(rows) / n_splits
    global_prev = float(y.mean())
    for offset in range(512):
        sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed + offset)
        candidate=[]; score=0.0; valid=True
        try:
            for tr, te in sgkf.split(np.zeros(len(rows)), y, groups):
                tr=np.asarray(tr,dtype=np.int64); te=np.asarray(te,dtype=np.int64)
                if len(np.unique(y[tr]))<2 or len(np.unique(y[te]))<2:
                    valid=False; break
                if np.intersect1d(np.unique(groups[tr]), np.unique(groups[te])).size:
                    valid=False; break
                score += abs(len(te)-target_n)/max(target_n,1.0)
                score += 0.5*abs(float(y[te].mean())-global_prev)
                candidate.append((tr,te))
        except ValueError:
            valid=False
        if valid and len(candidate)==n_splits and (best is None or score<best[0]):
            best=(score,candidate)
    if best is None:
        raise RuntimeError(f"{bundle.name}/{protocol}: no valid SGKF split after 512 random states")
    return best[1]


def _group_validation_split(
    outer_train: np.ndarray,
    groups: np.ndarray,
    labels: np.ndarray,
    val_fraction: float,
    seed: int,
    max_attempts: int = 4000,
) -> tuple[np.ndarray, np.ndarray]:
    """Create a group-disjoint validation split containing both classes.

    The earlier implementation sampled a fixed number of validation groups.
    That can be impossible when individual groups are single-class (for example,
    when one protein contributes only positive or only negative rows). This
    implementation first searches StratifiedGroupKFold candidates and then uses
    a variable-size randomized group search. It preserves the cold-start rule:
    no group appears in both inner-train and validation.
    """
    outer_train = np.asarray(outer_train, dtype=np.int64)
    group_values = np.asarray(groups)[outer_train]
    y_values = np.asarray(labels)[outer_train]
    unique_groups = np.unique(group_values)

    if len(outer_train) < 4 or len(unique_groups) < 2:
        raise RuntimeError(
            f"Protocol-matched validation is impossible: rows={len(outer_train)}, "
            f"groups={len(unique_groups)}"
        )
    if len(np.unique(y_values)) < 2:
        raise RuntimeError("Outer-training fold contains only one class")

    target_fraction = float(np.clip(val_fraction, 1.0 / len(unique_groups), 0.5))
    candidates: list[tuple[float, np.ndarray, np.ndarray]] = []

    def consider(train_rel: np.ndarray, val_rel: np.ndarray) -> None:
        if len(train_rel) == 0 or len(val_rel) == 0:
            return
        train_idx = outer_train[np.asarray(train_rel, dtype=np.int64)]
        val_idx = outer_train[np.asarray(val_rel, dtype=np.int64)]
        if len(np.unique(labels[train_idx])) < 2 or len(np.unique(labels[val_idx])) < 2:
            return
        # Group disjointness is mandatory for protocol-matched validation.
        if set(groups[train_idx].tolist()) & set(groups[val_idx].tolist()):
            return
        fraction_gap = abs(len(val_idx) / len(outer_train) - target_fraction)
        prevalence_gap = abs(float(labels[val_idx].mean()) - float(labels[outer_train].mean()))
        # Fraction match is primary; class-prevalence similarity breaks ties.
        score = fraction_gap + 0.20 * prevalence_gap
        candidates.append((score, train_idx.astype(np.int64), val_idx.astype(np.int64)))

    # 1) Prefer sklearn's stratified group splitter. Try several fold counts and
    # random states because difficult datasets can have strongly single-class groups.
    max_splits = min(len(unique_groups), max(2, int(round(1.0 / target_fraction))))
    split_counts = sorted({
        2,
        min(len(unique_groups), 3),
        min(len(unique_groups), 5),
        max_splits,
        min(len(unique_groups), max_splits + 1),
        min(len(unique_groups), max_splits + 2),
    })
    split_counts = [n for n in split_counts if 2 <= n <= len(unique_groups)]

    for offset in range(24):
        for n_splits in split_counts:
            try:
                sgkf = StratifiedGroupKFold(
                    n_splits=n_splits,
                    shuffle=True,
                    random_state=seed + offset,
                )
                for train_rel, val_rel in sgkf.split(
                    np.zeros(len(outer_train), dtype=np.int8),
                    y_values,
                    group_values,
                ):
                    consider(train_rel, val_rel)
            except ValueError:
                # Continue to the randomized fallback below.
                pass

    if candidates:
        _, train_idx, val_idx = min(candidates, key=lambda item: item[0])
        return train_idx, val_idx

    # 2) Fallback: vary the number of selected groups. The previous code fixed
    # this number, so n_validation=1 could never work when every group was
    # single-class. Trying 2+ groups solves that common case.
    group_to_rel = {
        g: np.flatnonzero(group_values == g)
        for g in unique_groups
    }
    target_groups = max(1, int(round(len(unique_groups) * target_fraction)))
    max_k = min(len(unique_groups) - 1, max(target_groups + 12, 2 * target_groups, 4))
    k_values = list(range(1, max_k + 1))
    rng = np.random.default_rng(seed)

    for attempt in range(max_attempts):
        k = k_values[attempt % len(k_values)]
        selected = rng.choice(unique_groups, size=k, replace=False)
        val_rel = np.concatenate([group_to_rel[g] for g in selected])
        val_mask = np.zeros(len(outer_train), dtype=bool)
        val_mask[val_rel] = True
        train_rel = np.flatnonzero(~val_mask)
        consider(train_rel, val_rel)

    if candidates:
        _, train_idx, val_idx = min(candidates, key=lambda item: item[0])
        return train_idx, val_idx

    # Detailed diagnostics make any genuinely impossible dataset fail immediately
    # and explain why, instead of failing after model training has started.
    stats = pd.DataFrame({"group": unique_groups})
    stats["n"] = [len(group_to_rel[g]) for g in unique_groups]
    stats["positive"] = [int(y_values[group_to_rel[g]].sum()) for g in unique_groups]
    stats["negative"] = stats["n"] - stats["positive"]
    mixed = int(((stats["positive"] > 0) & (stats["negative"] > 0)).sum())
    pos_groups = int((stats["positive"] > 0).sum())
    neg_groups = int((stats["negative"] > 0).sum())
    raise RuntimeError(
        "Could not construct a two-class group-held-out validation set even "
        "after stratified and variable-size search. "
        f"rows={len(outer_train)}, groups={len(unique_groups)}, mixed_groups={mixed}, "
        f"positive_groups={pos_groups}, negative_groups={neg_groups}."
    )


def _inner_split(
    bundle: DatasetBundle,
    outer_train: np.ndarray,
    protocol: str,
    profile: EvaluationProfile,
    val_fraction: float,
    seed: int,
) -> tuple[np.ndarray, np.ndarray]:
    if not profile.protocol_matched_validation or protocol == "edge":
        tr, va = train_test_split(
            outer_train,
            test_size=val_fraction,
            random_state=seed,
            stratify=bundle.y[outer_train],
        )
        return np.asarray(tr, np.int64), np.asarray(va, np.int64)
    if protocol == "rna_cold":
        return _group_validation_split(outer_train, bundle.r_idx, bundle.y, val_fraction, seed)
    if protocol == "prot_cold":
        return _group_validation_split(outer_train, bundle.p_idx, bundle.y, val_fraction, seed)
    raise ValueError(protocol)


def _assert_split_integrity(bundle: DatasetBundle, protocol: str, train_idx, val_idx, test_idx) -> None:
    row_sets = [set(np.asarray(x, dtype=int).tolist()) for x in (train_idx, val_idx, test_idx)]
    if row_sets[0] & row_sets[1] or row_sets[0] & row_sets[2] or row_sets[1] & row_sets[2]:
        raise AssertionError("Row overlap across train/validation/test")
    if protocol == "rna_cold":
        sets = [set(bundle.r_idx[x].tolist()) for x in (train_idx, val_idx, test_idx)]
        if sets[0] & sets[1] or sets[0] & sets[2] or sets[1] & sets[2]:
            raise AssertionError("RNA overlap across protocol-matched splits")
    if protocol == "prot_cold":
        sets = [set(bundle.p_idx[x].tolist()) for x in (train_idx, val_idx, test_idx)]
        if sets[0] & sets[1] or sets[0] & sets[2] or sets[1] & sets[2]:
            raise AssertionError("Protein overlap across protocol-matched splits")


def split_directory(output_root: Path, profile: str, dataset: str, protocol: str, seed: int, fold: int) -> Path:
    return (
        output_root
        / profile
        / dataset
        / protocol
        / f"seed_{seed}"
        / "splits"
        / f"fold_{fold}"
    )


def prepare_or_load_splits(
    bundle: DatasetBundle,
    output_root: str | Path,
    protocol: str,
    seed: int,
    n_splits: int = 5,
    val_fraction: float = 0.10,
    overwrite: bool = False,
) -> list[dict[str, np.ndarray]]:
    import hashlib
    output_root = Path(output_root)
    profile = PROFILES[bundle.profile]
    outer = _outer_splits(bundle, protocol, n_splits, seed)
    result=[]

    def _sha(arr):
        return hashlib.sha256(np.asarray(arr,dtype=np.int64).tobytes()).hexdigest()

    for fold,(outer_train,test_idx) in enumerate(outer):
        directory=split_directory(output_root,bundle.profile,bundle.name,protocol,seed,fold)
        paths={"train":directory/"inner_train_indices.npy","val":directory/"validation_indices.npy","test":directory/"test_indices.npy"}
        if all(p.exists() for p in paths.values()) and not overwrite:
            train_idx=np.load(paths["train"]); val_idx=np.load(paths["val"]); stored=np.load(paths["test"])
            if not np.array_equal(np.sort(test_idx),np.sort(stored)):
                raise RuntimeError(f"Stored split mismatch: {directory}. Use a fresh journal OUTPUT_ROOT.")
            test_idx=stored
        else:
            train_idx,val_idx=_inner_split(bundle,outer_train,protocol,profile,val_fraction,seed+fold*1009)
            _assert_split_integrity(bundle,protocol,train_idx,val_idx,test_idx)
            directory.mkdir(parents=True,exist_ok=True)
            np.save(paths["train"],train_idx); np.save(paths["val"],val_idx); np.save(paths["test"],test_idx)
            compact=[]
            for split_name,idx in [("train",train_idx),("validation",val_idx),("test",test_idx)]:
                part=bundle.rows.iloc[idx][["row_id","original_row","r_idx","p_idx","y"]].copy()
                part.insert(0,"split",split_name); compact.append(part)
            pd.concat(compact,ignore_index=True).to_csv(directory/"split_rows.csv.gz",index=False,compression="gzip")
            meta={
                "profile":bundle.profile,"dataset":bundle.name,"protocol":protocol,"seed":int(seed),"fold":int(fold),
                "n_train":int(len(train_idx)),"n_validation":int(len(val_idx)),"n_test":int(len(test_idx)),
                "train_positive":int(bundle.y[train_idx].sum()),"validation_positive":int(bundle.y[val_idx].sum()),"test_positive":int(bundle.y[test_idx].sum()),
                "train_sha256":_sha(train_idx),"validation_sha256":_sha(val_idx),"test_sha256":_sha(test_idx),
                "outer_splitter":"StratifiedGroupKFold" if protocol!="edge" else "StratifiedKFold",
                "protocol_matched_validation":True,
            }
            (directory/"split_meta.json").write_text(json.dumps(meta,indent=2))
        result.append({"train":train_idx,"val":val_idx,"test":test_idx})
    return result


def create_pair_model(name: str, d_rna: int, d_prot: int, config: PairModelTrainConfig) -> nn.Module:
    d = config.hidden_dim
    registry: dict[str, Callable[[], nn.Module]] = {
        "PairMLP": lambda: pm.PairMLPGraphWrapper(d_rna, d_prot, d, config.pair_dropout),
        "PairMLP-Wide": lambda: pm.PairMLP_Wide_Wrapper(d_rna, d_prot, d, config.pair_dropout),
        "PairMLP-Deep": lambda: pm.PairMLP_Deep_Wrapper(d_rna, d_prot, d, config.pair_dropout),
        "PairMLP-Cross": lambda: pm.PairMLP_Cross_Wrapper(
            d_rna, d_prot, d, config.pair_dropout, config.n_heads
        ),
        "DegGate": lambda: pm.ProtoGNN_DegGate(
            d_rna, d_prot, d, layers=2, dropout=config.graph_dropout
        ),
        "WeightedBipartite": lambda: pm.ProtoGNN_WeightedBipartite(
            d_rna, d_prot, d, layers=2, dropout=config.graph_dropout
        ),
        "CoNeighbor": lambda: pm.ProtoGNN_CoNeighbor(
            d_rna, d_prot, d, layers=2, dropout=config.graph_dropout
        ),
        "ProtoContrast": lambda: pm.ProtoGNN_ProtoContrast(
            d_rna, d_prot, d, layers=2, dropout=config.graph_dropout, K=4, tau=0.2, contrast_w=0.2
        ),
        "GraphSAGE-2L": lambda: pm.ProtoGNN_GraphSAGE(
            d_rna,
            d_prot,
            d,
            layers=config.graphsage_layers,
            dropout=config.graph_dropout,
            sample_k=config.graphsage_sample_k,
            edge_drop=0.0,
            use_l2norm=True,
        ),
    }
    if name not in registry:
        raise KeyError(f"Unknown exact model {name!r}; available={list(registry)}")
    return registry[name]()


@torch.no_grad()
def score_pair_model(
    model: nn.Module,
    x_rna: torch.Tensor,
    x_prot: torch.Tensor,
    adjacency: torch.Tensor,
    rna2prot,
    prot2rna,
    r_idx: np.ndarray,
    p_idx: np.ndarray,
    device: torch.device,
) -> np.ndarray:
    model.eval()
    h_rna, h_prot, core = model.encode_nodes(x_rna, x_prot, adjacency, rna2prot, prot2rna)
    logits = model.score_edges(
        h_rna,
        h_prot,
        core,
        torch.as_tensor(r_idx, dtype=torch.long, device=device),
        torch.as_tensor(p_idx, dtype=torch.long, device=device),
    )
    return logits.detach().cpu().numpy()


def train_pair_model_fold(
    model_name: str,
    bundle: DatasetBundle,
    split: dict[str, np.ndarray],
    profile: EvaluationProfile,
    seed: int,
    config: PairModelTrainConfig,
    device: torch.device,
    output_dir: Path,
) -> dict[str, Any]:
    seed_all(seed)
    output_dir.mkdir(parents=True, exist_ok=True)
    train_idx, val_idx, test_idx = split["train"], split["val"], split["test"]
    graph_idx = (
        np.concatenate([train_idx, val_idx])
        if profile.include_validation_edges_in_graph
        else train_idx
    )

    x_rna = torch.as_tensor(bundle.x_rna, dtype=torch.float32, device=device)
    x_prot = torch.as_tensor(bundle.x_prot, dtype=torch.float32, device=device)
    adjacency, rna2prot, prot2rna = pm.build_train_graph_from_split(
        bundle.r_idx,
        bundle.p_idx,
        bundle.y,
        graph_idx,
        bundle.x_rna.shape[0],
        bundle.x_prot.shape[0],
    )
    adjacency = adjacency.to(device)
    model = create_pair_model(
        model_name, bundle.x_rna.shape[1], bundle.x_prot.shape[1], config
    ).to(device)

    positive_weight = torch.tensor(
        [(bundle.y[train_idx] == 0).sum() / max((bundle.y[train_idx] == 1).sum(), 1)],
        dtype=torch.float32,
        device=device,
    )
    criterion = nn.BCEWithLogitsLoss(pos_weight=positive_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)

    tr_r = torch.as_tensor(bundle.r_idx[train_idx], dtype=torch.long, device=device)
    tr_p = torch.as_tensor(bundle.p_idx[train_idx], dtype=torch.long, device=device)
    tr_y = torch.as_tensor(bundle.y[train_idx], dtype=torch.float32, device=device)

    best_state = None
    best_val_auroc = -np.inf
    bad_epochs = 0
    history = []

    for epoch in range(1, config.max_epochs + 1):
        model.train()
        h_rna, h_prot, core = model.encode_nodes(x_rna, x_prot, adjacency, rna2prot, prot2rna)
        optimizer.zero_grad(set_to_none=True)
        permutation = torch.randperm(len(tr_y), device=device)
        total_loss = torch.zeros((), dtype=torch.float32, device=device)
        for start in range(0, len(tr_y), config.batch_size):
            batch = permutation[start : start + config.batch_size]
            logits = model.score_edges(h_rna, h_prot, core, tr_r[batch], tr_p[batch])
            total_loss = total_loss + criterion(logits, tr_y[batch]) * (len(batch) / len(tr_y))
        total_loss.backward()
        optimizer.step()

        val_logits = score_pair_model(
            model,
            x_rna,
            x_prot,
            adjacency,
            rna2prot,
            prot2rna,
            bundle.r_idx[val_idx],
            bundle.p_idx[val_idx],
            device,
        )
        val_prob = sigmoid_np(val_logits)
        val_auroc = float(roc_auc_score(bundle.y[val_idx], val_prob))
        _, val_mcc = best_threshold_by_mcc(bundle.y[val_idx], val_logits, 200)
        history.append({
            "epoch": epoch,
            "train_loss": float(total_loss.detach().cpu()),
            "val_AUROC": val_auroc,
            "val_MCC_best": val_mcc,
        })
        if val_auroc > best_val_auroc:
            best_val_auroc = val_auroc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= config.early_stop_patience:
                break

    if best_state is None:
        raise RuntimeError("No valid pair-model checkpoint")
    model.load_state_dict(best_state)

    val_logits = score_pair_model(
        model, x_rna, x_prot, adjacency, rna2prot, prot2rna,
        bundle.r_idx[val_idx], bundle.p_idx[val_idx], device
    )
    test_logits = score_pair_model(
        model, x_rna, x_prot, adjacency, rna2prot, prot2rna,
        bundle.r_idx[test_idx], bundle.p_idx[test_idx], device
    )
    threshold, val_mcc = best_threshold_by_mcc(bundle.y[val_idx], val_logits)
    metrics = metrics_from_logits(bundle.y[test_idx], test_logits, threshold)

    pd.DataFrame(history).to_csv(output_dir / "training_history.csv.gz", index=False, compression="gzip")
    torch.save(model.state_dict(), output_dir / "model.pt")
    prediction_frame = bundle.rows.iloc[test_idx][
        ["row_id", "original_row", "r_idx", "p_idx", "y"]
    ].copy()
    prediction_frame["logit"] = test_logits
    prediction_frame["probability"] = sigmoid_np(test_logits)
    prediction_frame["prediction"] = (
        prediction_frame["probability"].to_numpy() >= threshold
    ).astype(int)
    prediction_frame.to_csv(output_dir / "test_predictions.csv.gz", index=False, compression="gzip")
    pd.DataFrame({
        "row_idx": val_idx,
        "y": bundle.y[val_idx],
        "logit": val_logits,
    }).to_csv(output_dir / "validation_predictions.csv.gz", index=False, compression="gzip")

    result = {
        "model": model_name,
        "dataset": bundle.name,
        "profile": bundle.profile,
        "seed": seed,
        "threshold": threshold,
        "validation_MCC": val_mcc,
        "best_validation_AUROC": best_val_auroc,
        "epochs_completed": len(history),
        "graph_rows": int(len(graph_idx)),
        **metrics,
    }
    (output_dir / "metrics.json").write_text(json.dumps(result, indent=2, allow_nan=True))
    del model, x_rna, x_prot, adjacency
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result


# =============================================================================
# ZHMolGraph runner using exact architecture classes
# =============================================================================

def train_zh_graphsage(
    bundle: DatasetBundle,
    graph_idx: np.ndarray,
    seed: int,
    config: ZHMolGraphTrainConfig,
    device: torch.device,
):
    seed_all(seed)
    n_rna, n_prot = len(bundle.x_rna), len(bundle.x_prot)
    raw_np = zh.build_unified_graph_features(bundle.x_rna, bundle.x_prot)
    raw = torch.as_tensor(raw_np, dtype=torch.float32, device=device)
    adjacency, warm_nodes = zh.build_positive_train_adjacency(
        bundle.r_idx, bundle.p_idx, bundle.y, graph_idx, n_rna, n_prot
    )
    model = zh.OriginalGraphSage(
        config.sage_layers,
        raw.shape[1],
        config.sage_hidden,
        raw,
        adjacency,
        gcn=False,
        agg_func="MEAN",
        num_sample=config.sage_neighbours,
    ).to(device)
    unsupervised = zh.UnsupervisedGraphLoss(
        adjacency, warm_nodes, device, q=config.sage_num_negative
    )
    history = []
    for epoch in range(1, config.sage_epochs + 1):
        model.train()
        optimizer = torch.optim.SGD(model.parameters(), lr=config.sage_lr)
        order = np.random.permutation(warm_nodes)
        losses = []
        used = 0
        for start in range(0, len(order), config.sage_batch_size):
            target = order[start : start + config.sage_batch_size]
            expanded = unsupervised.extend_nodes(target, num_neg=config.sage_num_negative)
            if expanded is None:
                continue
            optimizer.zero_grad(set_to_none=True)
            embeddings = model(expanded)
            loss = (
                unsupervised.get_loss_sage(embeddings, expanded)
                if config.sage_loss == "normal"
                else unsupervised.get_loss_margin(embeddings, expanded)
            )
            if loss is None or not torch.isfinite(loss):
                continue
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), config.sage_grad_clip)
            optimizer.step()
            losses.append(float(loss.detach().cpu()))
            used += len(target)
        history.append({
            "epoch": epoch,
            "loss": float(np.mean(losses)) if losses else float("nan"),
            "warm_targets": used,
        })

    seed_all(seed + 100_000)
    model.eval()
    all_nodes = np.arange(n_rna + n_prot, dtype=np.int64)
    chunks = []
    with torch.no_grad():
        for start in range(0, len(all_nodes), 500):
            chunks.append(model(all_nodes[start : start + 500]).cpu())
    z = torch.cat(chunks).numpy().astype(np.float32)
    return model, z[:n_rna], z[n_rna:], pd.DataFrame(history), adjacency


class _EdgeDataset(Dataset):
    def __init__(self, bundle: DatasetBundle, indices: np.ndarray):
        self.r = torch.as_tensor(bundle.r_idx[indices], dtype=torch.long)
        self.p = torch.as_tensor(bundle.p_idx[indices], dtype=torch.long)
        self.y = torch.as_tensor(bundle.y[indices], dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):
        return self.r[index], self.p[index], self.y[index]


@torch.inference_mode()
def score_vecnn(model, xr, xp, bundle: DatasetBundle, indices: np.ndarray, device, batch_size=512):
    model.eval()
    chunks = []
    for start in range(0, len(indices), batch_size):
        batch = indices[start : start + batch_size]
        rr = torch.as_tensor(bundle.r_idx[batch], dtype=torch.long, device=device)
        pp = torch.as_tensor(bundle.p_idx[batch], dtype=torch.long, device=device)
        logits = model(xp[pp], xr[rr])
        chunks.append(logits.float().cpu().numpy())
    return np.concatenate(chunks) if chunks else np.empty(0, dtype=np.float32)


def train_zh_vecnn(
    bundle: DatasetBundle,
    z_rna: np.ndarray,
    z_prot: np.ndarray,
    train_idx: np.ndarray,
    val_idx: np.ndarray,
    seed: int,
    profile: EvaluationProfile,
    config: ZHMolGraphTrainConfig,
    device: torch.device,
    select_best_checkpoint: bool = False,
):
    seed_all(seed)
    x_rna_aug = torch.as_tensor(
        np.concatenate([z_rna, bundle.x_rna], axis=1).astype(np.float32), device=device
    )
    x_prot_aug = torch.as_tensor(
        np.concatenate([z_prot, bundle.x_prot], axis=1).astype(np.float32), device=device
    )
    model = zh.VecNN(x_prot_aug.shape[1], x_rna_aug.shape[1]).to(device)
    loader = DataLoader(
        _EdgeDataset(bundle, train_idx),
        batch_size=config.vec_batch_size,
        shuffle=True,
        num_workers=0,
        drop_last=False,
    )
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config.vec_lr,
        betas=(0.9, 0.999),
        weight_decay=config.vec_weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=profile.zhmolgraph_vec_epochs
    )
    use_amp = bool(config.use_amp and torch.cuda.is_available())
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    best_state = None
    best_val = -np.inf
    best_epoch = -1
    bad = 0
    history = []

    for epoch in range(1, profile.zhmolgraph_vec_epochs + 1):
        model.train()
        loss_sum, seen = 0.0, 0
        for rr, pp, yy in loader:
            rr, pp, yy = rr.to(device), pp.to(device), yy.to(device)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                logits = model(x_prot_aug[pp], x_rna_aug[rr])
                loss = criterion(logits, yy)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            loss_sum += float(loss.detach().cpu()) * len(yy)
            seen += len(yy)
        scheduler.step()

        val_logits = score_vecnn(model, x_rna_aug, x_prot_aug, bundle, val_idx, device)
        val_prob = sigmoid_np(val_logits)
        val_auroc = float(roc_auc_score(bundle.y[val_idx], val_prob))
        _, val_mcc = best_threshold_by_mcc(bundle.y[val_idx], val_logits, 400)
        history.append({
            "epoch": epoch,
            "train_loss": loss_sum / max(seen, 1),
            "val_AUROC": val_auroc,
            "val_MCC_best": val_mcc,
            "lr": optimizer.param_groups[0]["lr"],
        })

        # Track the best validation checkpoint even when all 120 epochs are completed.
        if val_auroc > best_val + 1e-6:
            best_val = val_auroc
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1

        # clean_early may stop; Best120 never stops early and only selects after epoch 120.
        if profile.zhmolgraph_early_stopping:
            if epoch >= config.vec_min_epochs and bad >= config.vec_patience:
                break

    final_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    final_val_auroc = float(history[-1]["val_AUROC"])
    if profile.zhmolgraph_early_stopping or select_best_checkpoint:
        if best_state is None:
            raise RuntimeError("No ZHMolGraph VecNN checkpoint")
        model.load_state_dict(best_state)
        selected_epoch = best_epoch
        selected_val_auroc = best_val
    else:
        selected_epoch = int(history[-1]["epoch"])
        selected_val_auroc = final_val_auroc

    checkpoint_info = {
        "best_state": best_state,
        "final_state": final_state,
        "best_epoch": int(best_epoch),
        "selected_epoch": int(selected_epoch),
        "best_validation_AUROC": float(best_val),
        "final_validation_AUROC": float(final_val_auroc),
        "selected_validation_AUROC": float(selected_val_auroc),
    }
    return model, x_rna_aug, x_prot_aug, pd.DataFrame(history), checkpoint_info


def train_zhmolgraph_fold(
    bundle: DatasetBundle,
    split: dict[str, np.ndarray],
    profile: EvaluationProfile,
    seed: int,
    config: ZHMolGraphTrainConfig,
    device: torch.device,
    output_dir: Path,
    model_name: str = "ZHMolGraph",
) -> dict[str, Any]:
    output_dir.mkdir(parents=True, exist_ok=True)
    train_idx, val_idx, test_idx = split["train"], split["val"], split["test"]
    graph_idx = (
        np.concatenate([train_idx, val_idx])
        if profile.include_validation_edges_in_graph
        else train_idx
    )
    select_best = model_name == "ZHMolGraph-Best120"

    sage_model, z_rna, z_prot, sage_history, adjacency = train_zh_graphsage(
        bundle, graph_idx, seed, config, device
    )
    vec_model, xr, xp, vec_history, checkpoint_info = train_zh_vecnn(
        bundle, z_rna, z_prot, train_idx, val_idx, seed + 17, profile, config, device,
        select_best_checkpoint=select_best,
    )

    val_logits = score_vecnn(vec_model, xr, xp, bundle, val_idx, device)
    test_logits = score_vecnn(vec_model, xr, xp, bundle, test_idx, device)
    threshold, val_mcc = best_threshold_by_mcc(bundle.y[val_idx], val_logits)
    metrics = metrics_from_logits(bundle.y[test_idx], test_logits, threshold)

    sage_history.to_csv(output_dir / "graphsage_history.csv.gz", index=False, compression="gzip")
    vec_history.to_csv(output_dir / "vecnn_history.csv.gz", index=False, compression="gzip")
    torch.save(sage_model.state_dict(), output_dir / "graphsage.pt")
    torch.save(vec_model.state_dict(), output_dir / "vecnn.pt")
    pd.DataFrame({
        "row_idx": val_idx,
        "y": bundle.y[val_idx],
        "logit": val_logits,
    }).to_csv(output_dir / "validation_predictions.csv.gz", index=False, compression="gzip")

    prediction_frame = bundle.rows.iloc[test_idx][
        ["row_id", "original_row", "r_idx", "p_idx", "y"]
    ].copy()
    prediction_frame["logit"] = test_logits
    prediction_frame["probability"] = sigmoid_np(test_logits)
    prediction_frame["prediction"] = (
        prediction_frame["probability"].to_numpy() >= threshold
    ).astype(int)
    prediction_frame.to_csv(output_dir / "test_predictions.csv.gz", index=False, compression="gzip")

    result = {
        "model": model_name,
        "dataset": bundle.name,
        "profile": bundle.profile,
        "seed": seed,
        "threshold": threshold,
        "validation_MCC": val_mcc,
        "best_validation_AUROC": checkpoint_info["best_validation_AUROC"],
        "final_validation_AUROC": checkpoint_info["final_validation_AUROC"],
        "selected_validation_AUROC": checkpoint_info["selected_validation_AUROC"],
        "best_epoch": checkpoint_info["best_epoch"],
        "selected_epoch": checkpoint_info["selected_epoch"],
        "checkpoint_policy": "best_validation_AUROC_after_120" if select_best else (
            "early_stopping_best_validation_AUROC" if profile.zhmolgraph_early_stopping else "final_epoch_120"
        ),
        "vec_epochs_completed": int(len(vec_history)),
        "sage_epochs_completed": int(len(sage_history)),
        "graph_rows": int(len(graph_idx)),
        **metrics,
    }
    (output_dir / "metrics.json").write_text(json.dumps(result, indent=2, allow_nan=True))
    del sage_model, vec_model, xr, xp
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result


# =============================================================================
# Matrix runner and summaries
# =============================================================================

def model_directory(
    output_root: Path,
    profile: str,
    dataset: str,
    protocol: str,
    seed: int,
    fold: int,
    model: str,
) -> Path:
    slug = model.lower().replace(" ", "_").replace("-", "_")
    return output_root / profile / dataset / protocol / f"seed_{seed}" / f"fold_{fold}" / slug


def disk_guard(path: str | Path, min_free_gb: float = 2.0) -> float:
    """Fail before serialization/training if Kaggle working disk is nearly full."""
    import shutil
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    free_gb = shutil.disk_usage(path).free / (1024**3)
    if free_gb < min_free_gb:
        raise RuntimeError(
            f"Only {free_gb:.2f} GB free under {path}. Export completed compact artifacts or prune old checkpoints before continuing."
        )
    return free_gb


def run_plan(
    plan: ExperimentPlan,
    data_root: str | Path,
    output_root: str | Path,
    pair_config: Optional[PairModelTrainConfig] = None,
    zh_config: Optional[ZHMolGraphTrainConfig] = None,
    device: Optional[torch.device] = None,
    rna_tag: str = "rnafm",
    protein_tag: str = "proteinprottrans",
) -> pd.DataFrame:
    if plan.profile not in PROFILES:
        raise KeyError(plan.profile)
    profile = PROFILES[plan.profile]
    pair_config = pair_config or PairModelTrainConfig()
    zh_config = zh_config or ZHMolGraphTrainConfig()
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    output_root = Path(output_root)
    output_root.mkdir(parents=True, exist_ok=True)
    (output_root / f"plan_{plan.name}.json").write_text(json.dumps(asdict(plan), indent=2))

    rows = []
    bundles: dict[str, DatasetBundle] = {}
    for dataset_name in plan.datasets:
        bundle = load_dataset(
            data_root, dataset_name, plan.profile, output_root, rna_tag, protein_tag
        )
        bundles[dataset_name] = bundle
        for protocol in plan.protocols:
            for seed in plan.seeds:
                splits = prepare_or_load_splits(
                    bundle,
                    output_root,
                    protocol,
                    seed,
                    n_splits=plan.n_splits,
                    val_fraction=plan.val_fraction,
                    overwrite=plan.overwrite,
                )
                folds = range(plan.n_splits) if plan.folds is None else plan.folds
                for fold in folds:
                    split = splits[fold]
                    for model_name in plan.models:
                        disk_guard(output_root, min_free_gb=2.0)
                        directory = model_directory(
                            output_root, plan.profile, dataset_name, protocol, seed, fold, model_name
                        )
                        metrics_path = directory / "metrics.json"
                        if metrics_path.exists() and plan.resume and not plan.overwrite:
                            result = json.loads(metrics_path.read_text())
                        else:
                            fold_seed = seed + fold * 1009
                            if is_zhmolgraph_model(model_name):
                                result = train_zhmolgraph_fold(
                                    bundle,
                                    split,
                                    profile,
                                    fold_seed,
                                    zh_config,
                                    device,
                                    directory,
                                    model_name=model_name,
                                )
                            else:
                                result = train_pair_model_fold(
                                    model_name,
                                    bundle,
                                    split,
                                    profile,
                                    fold_seed,
                                    pair_config,
                                    device,
                                    directory,
                                )
                        # Keep the base experimental seed separate from the fold-specific training seed.
                        fold_seed = seed + fold * 1009
                        result["run_seed"] = int(seed)
                        result["fold_seed"] = int(result.get("fold_seed", result.get("seed", fold_seed)))
                        result["seed"] = int(seed)
                        result["protocol"] = protocol
                        result["fold"] = int(fold)
                        metrics_path.parent.mkdir(parents=True, exist_ok=True)
                        metrics_path.write_text(json.dumps(result, indent=2, allow_nan=True))
                        rows.append(result)
                        pd.DataFrame(rows).to_csv(output_root / "current_run_fold_results.csv", index=False)

    frame = pd.DataFrame(rows)
    if len(frame):
        frame.to_csv(output_root / f"{plan.name}_fold_results.csv", index=False)
        summarize_results(output_root)
    return frame


def collect_metrics(output_root: str | Path) -> pd.DataFrame:
    rows = []
    for path in Path(output_root).rglob("metrics.json"):
        try:
            row = json.loads(path.read_text())
            parts = path.parts
            if "model" in row and "dataset" in row:
                if "protocol" not in row:
                    for i, part in enumerate(parts):
                        if part.startswith("seed_") and i > 0:
                            row["protocol"] = parts[i - 1]
                            break
                if "fold" not in row:
                    for part in parts:
                        if part.startswith("fold_"):
                            row["fold"] = int(part.split("_", 1)[1])
                fold = int(row.get("fold", 0))
                stored_seed = int(row.get("seed", 0))
                if "run_seed" not in row:
                    row["run_seed"] = stored_seed - fold * 1009
                if "fold_seed" not in row:
                    row["fold_seed"] = stored_seed
                row["seed"] = int(row["run_seed"])
                rows.append(row)
        except Exception:
            continue
    return pd.DataFrame(rows)


def summarize_results(output_root: str | Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    output_root = Path(output_root)
    frame = collect_metrics(output_root)
    if frame.empty:
        return frame, frame
    frame = frame.drop_duplicates(
        ["profile", "dataset", "protocol", "run_seed", "fold", "model"], keep="last"
    )
    frame.to_csv(output_root / "all_model_fold_results.csv", index=False)
    metrics = ["ACC", "SEN", "SPE", "PRE", "F1", "MCC", "AUROC", "AUPRC"]
    per_seed = (
        frame.groupby(["profile", "dataset", "protocol", "model", "run_seed"])[metrics]
        .agg(["mean", "std", "count"])
        .reset_index()
    )
    per_seed.columns = [
        "_".join(str(x) for x in col if str(x)) if isinstance(col, tuple) else col
        for col in per_seed.columns
    ]
    overall = (
        frame.groupby(["profile", "dataset", "protocol", "model"])[metrics]
        .agg(["mean", "std", "count"])
        .reset_index()
    )
    overall.columns = [
        "_".join(str(x) for x in col if str(x)) if isinstance(col, tuple) else col
        for col in overall.columns
    ]
    per_seed.to_csv(output_root / "summary_per_seed.csv", index=False)
    overall.to_csv(output_root / "summary_overall.csv", index=False)
    return per_seed, overall


# =============================================================================
# TheNovel evaluation
# =============================================================================

def _accession_prefix(value: Any) -> str:
    value = str(value).strip()
    return value.split("-", 1)[0] if "-" in value else value


def load_thenovel(
    data_root: str | Path,
    output_root: str | Path,
    profile_name: str,
    rna_tag: str = "rnafm",
    protein_tag: str = "proteinprottrans",
) -> DatasetBundle:
    data_root = Path(data_root)
    rna_path = _find_file(data_root, f"RPI_NPInter5_{rna_tag}_embed_normal.pkl")
    prot_path = _find_file(data_root, f"RPI_NPInter5_{protein_tag}_embed_normal.pkl")
    csv_path = _find_file(data_root, "dataset_RPI_NPInter5_RP.csv")
    xlsx_path = _find_file(data_root, "NPInter5.xlsx")

    with rna_path.open("rb") as handle:
        rna_df = pkl.load(handle)
    with prot_path.open("rb") as handle:
        prot_df = pkl.load(handle)
    r_seq = pick_col(rna_df, ["RNA_aa_code", "RNA_seq", "rna_seq"], "RNA sequence")
    p_seq = pick_col(prot_df, ["target_aa_code", "protein_seq", "prot_seq"], "protein sequence")
    r_emb = pick_col(rna_df, ["normalized_embeddings", "embeddings", "embed"], "RNA embedding")
    p_emb = pick_col(prot_df, ["normalized_embeddings", "embeddings", "embed"], "protein embedding")
    rna_keys, x_rna = _deduplicate_embedding_table(rna_df, r_seq, r_emb, "TheNovel RNA")
    prot_keys, x_prot = _deduplicate_embedding_table(prot_df, p_seq, p_emb, "TheNovel Protein")
    rna_map = {s: i for i, s in enumerate(rna_keys)}
    prot_map = {s: i for i, s in enumerate(prot_keys)}

    csv_df = pd.read_csv(csv_path)
    rn = pick_col(csv_df, ["RNA names", "rna names", "RNA_name", "RNA"], "RNA name")
    pn = pick_col(csv_df, ["Protein names", "protein names", "Protein_name", "Protein"], "protein name")
    rs = pick_col(csv_df, ["RNA_aa_code", "RNA_seq", "rna_seq"], "RNA sequence")
    ps = pick_col(csv_df, ["target_aa_code", "protein_seq", "prot_seq"], "protein sequence")
    csv_df["_rk"] = csv_df[rn].map(_accession_prefix)
    csv_df["_pk"] = csv_df[pn].map(_accession_prefix)
    r_lookup = (
        csv_df[["_rk", rs]].dropna().drop_duplicates("_rk")
        .set_index("_rk")[rs].map(canonical_sequence).to_dict()
    )
    p_lookup = (
        csv_df[["_pk", ps]].dropna().drop_duplicates("_pk")
        .set_index("_pk")[ps].map(canonical_sequence).to_dict()
    )

    xlsx = pd.read_excel(xlsx_path)
    xrn = pick_col(xlsx, ["RNA names", "rna names", "RNA_name", "RNA"], "RNA name")
    xpn = pick_col(xlsx, ["Protein names", "protein names", "Protein_name", "Protein"], "protein name")
    xy = pick_col(xlsx, ["Labels", "labels", "Label", "Y", "y"], "label")
    rows = []
    for original_row, row in xlsx.iterrows():
        rna_sequence = r_lookup.get(_accession_prefix(row[xrn]))
        protein_sequence = p_lookup.get(_accession_prefix(row[xpn]))
        if rna_sequence in rna_map and protein_sequence in prot_map:
            rows.append({
                "original_row": original_row,
                "rna_sequence": rna_sequence,
                "protein_sequence": protein_sequence,
                "r_idx": rna_map[rna_sequence],
                "p_idx": prot_map[protein_sequence],
                "y": int(row[xy]),
            })
    frame = pd.DataFrame(rows)
    frame["row_id"] = np.arange(len(frame), dtype=np.int64)
    meta = {
        "dataset": "TheNovel",
        "profile": profile_name,
        "n_rna": len(rna_keys),
        "n_prot": len(prot_keys),
        "d_rna": x_rna.shape[1],
        "d_prot": x_prot.shape[1],
        "n_rows": len(frame),
    }
    return DatasetBundle(
        "TheNovel",
        profile_name,
        x_rna,
        x_prot,
        frame["r_idx"].to_numpy(np.int64),
        frame["p_idx"].to_numpy(np.int64),
        frame["y"].to_numpy(np.int64),
        frame,
        rna_keys,
        prot_keys,
        meta,
    )


def score_pair_checkpoint_on_thenovel(
    model_name: str,
    checkpoint: Path,
    novel: DatasetBundle,
    pair_config: PairModelTrainConfig,
    device: torch.device,
) -> np.ndarray:
    model = create_pair_model(
        model_name, novel.x_rna.shape[1], novel.x_prot.shape[1], pair_config
    ).to(device)
    model.load_state_dict(torch.load(checkpoint, map_location=device))
    n_rna, n_prot = len(novel.x_rna), len(novel.x_prot)
    indices = torch.zeros((2, 0), dtype=torch.long, device=device)
    values = torch.zeros((0,), dtype=torch.float32, device=device)
    adjacency = torch.sparse_coo_tensor(indices, values, (n_rna, n_prot)).coalesce()
    rna2prot = [torch.zeros(0, dtype=torch.long, device=device) for _ in range(n_rna)]
    prot2rna = [torch.zeros(0, dtype=torch.long, device=device) for _ in range(n_prot)]
    logits = score_pair_model(
        model,
        torch.as_tensor(novel.x_rna, dtype=torch.float32, device=device),
        torch.as_tensor(novel.x_prot, dtype=torch.float32, device=device),
        adjacency,
        rna2prot,
        prot2rna,
        novel.r_idx,
        novel.p_idx,
        device,
    )
    return logits


def score_zh_checkpoint_on_thenovel(
    fold_dir: Path,
    novel: DatasetBundle,
    zh_config: ZHMolGraphTrainConfig,
    device: torch.device,
) -> np.ndarray:
    raw_np = zh.build_unified_graph_features(novel.x_rna, novel.x_prot)
    raw = torch.as_tensor(raw_np, dtype=torch.float32, device=device)
    empty_adj = [set() for _ in range(len(raw_np))]
    sage = zh.OriginalGraphSage(
        zh_config.sage_layers,
        raw.shape[1],
        zh_config.sage_hidden,
        raw,
        empty_adj,
        gcn=False,
        agg_func="MEAN",
        num_sample=zh_config.sage_neighbours,
    ).to(device)
    sage.load_state_dict(torch.load(fold_dir / "graphsage.pt", map_location=device))
    sage.eval()
    nodes = np.arange(len(raw_np), dtype=np.int64)
    chunks = []
    with torch.no_grad():
        for start in range(0, len(nodes), 500):
            chunks.append(sage(nodes[start : start + 500]).cpu())
    z = torch.cat(chunks).numpy().astype(np.float32)
    z_rna, z_prot = z[: len(novel.x_rna)], z[len(novel.x_rna) :]
    xr = torch.as_tensor(np.concatenate([z_rna, novel.x_rna], axis=1), device=device)
    xp = torch.as_tensor(np.concatenate([z_prot, novel.x_prot], axis=1), device=device)
    vec = zh.VecNN(xp.shape[1], xr.shape[1]).to(device)
    vec.load_state_dict(torch.load(fold_dir / "vecnn.pt", map_location=device))
    all_rows = np.arange(len(novel.y), dtype=np.int64)
    return score_vecnn(vec, xr, xp, novel, all_rows, device)


def run_thenovel(
    profile_name: str,
    seed: int,
    models: Iterable[str],
    data_root: str | Path,
    output_root: str | Path,
    pair_config: Optional[PairModelTrainConfig] = None,
    zh_config: Optional[ZHMolGraphTrainConfig] = None,
    device: Optional[torch.device] = None,
    n_splits: int = 5,
    rna_tag: str = "rnafm",
    protein_tag: str = "proteinprottrans",
) -> pd.DataFrame:
    """Score five NPInter2 edge-level source checkpoints on TheNovel.

    Predictions are stored as gzip CSV rather than NPZ so later figures and
    threshold analyses can be reproduced without custom binary loading.
    """
    pair_config = pair_config or PairModelTrainConfig()
    zh_config = zh_config or ZHMolGraphTrainConfig()
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    output_root = Path(output_root)
    novel = load_thenovel(data_root, output_root, profile_name, rna_tag, protein_tag)
    source = load_dataset(data_root, "NPInter2", profile_name, output_root, rna_tag, protein_tag)
    r_overlap = set(source.rna_keys) & set(novel.rna_keys)
    p_overlap = set(source.prot_keys) & set(novel.prot_keys)
    if r_overlap or p_overlap:
        raise AssertionError(f"TheNovel exact overlap detected: RNA={len(r_overlap)}, Protein={len(p_overlap)}")

    rows = []
    for model_name in models:
        fold_probabilities = []
        for fold in range(n_splits):
            directory = model_directory(output_root, profile_name, "NPInter2", "edge", seed, fold, model_name)
            if not (directory / "metrics.json").exists():
                raise FileNotFoundError(
                    f"Missing NPInter2 edge-level source checkpoint for TheNovel: {directory}. "
                    "Run the optional TheNovel source plan first."
                )
            logits = (score_zh_checkpoint_on_thenovel(directory, novel, zh_config, device)
                      if is_zhmolgraph_model(model_name)
                      else score_pair_checkpoint_on_thenovel(model_name, directory / "model.pt", novel, pair_config, device))
            probability = sigmoid_np(logits)
            fold_probabilities.append(probability)
            rows.append({"profile":profile_name,"seed":seed,"model":model_name,"aggregation":"individual_fold","fold":fold,
                         "AUROC":float(roc_auc_score(novel.y,probability)),"AUPRC":float(average_precision_score(novel.y,probability))})
        matrix=np.stack(fold_probabilities); ensemble=matrix.mean(axis=0)
        individual=[r for r in rows if r["model"]==model_name and r["aggregation"]=="individual_fold"]
        rows.append({"profile":profile_name,"seed":seed,"model":model_name,"aggregation":"mean_of_fold_metrics","fold":-1,
                     "AUROC":float(np.mean([r["AUROC"] for r in individual])),"AUPRC":float(np.mean([r["AUPRC"] for r in individual]))})
        rows.append({"profile":profile_name,"seed":seed,"model":model_name,"aggregation":"probability_ensemble","fold":-1,
                     "AUROC":float(roc_auc_score(novel.y,ensemble)),"AUPRC":float(average_precision_score(novel.y,ensemble))})
        out_dir=output_root/profile_name/"TheNovel"/f"seed_{seed}"/model_name.lower().replace("-","_")
        out_dir.mkdir(parents=True,exist_ok=True)
        pred=pd.DataFrame({"row_id":np.arange(len(novel.y)),"y":novel.y,"ensemble_probability":ensemble})
        for f in range(n_splits): pred[f"fold_{f}_probability"]=matrix[f]
        pred.to_csv(out_dir/"predictions.csv.gz",index=False,compression="gzip")
    result=pd.DataFrame(rows)
    result_path=output_root/profile_name/"thenovel_results.csv"
    if result_path.exists():
        previous=pd.read_csv(result_path); result=pd.concat([previous,result],ignore_index=True)
        result=result.drop_duplicates(["profile","seed","model","aggregation","fold"],keep="last")
    result.to_csv(result_path,index=False)
    return result


def _safe_group_metrics(frame: pd.DataFrame, threshold: float) -> dict[str, float]:
    y = frame["y"].to_numpy(int)
    prob = frame["probability"].to_numpy(float)
    pred = (prob >= threshold).astype(int)
    tp, tn, fp, fn = confusion_counts(y, pred)
    return {
        "MCC": mcc_from_counts(tp, tn, fp, fn),
        "AUROC": float(roc_auc_score(y, prob)) if len(np.unique(y)) == 2 else float("nan"),
        "AUPRC": float(average_precision_score(y, prob)) if len(np.unique(y)) == 2 else float("nan"),
        "n_pairs": int(len(frame)),
        "n_positive": int((y == 1).sum()),
        "n_negative": int((y == 0).sum()),
    }


def _cosine_max(query: np.ndarray, matrix: np.ndarray) -> float:
    if len(matrix) == 0:
        return float("nan")
    q = query / max(np.linalg.norm(query), 1e-12)
    m = matrix / np.clip(np.linalg.norm(matrix, axis=1, keepdims=True), 1e-12, None)
    return float(np.max(m @ q))


def protein_endpoint_analysis(
    data_root: str | Path,
    output_root: str | Path,
    profile_name: str = "clean_120",
    datasets: Iterable[str] = ("NPInter2", "RPI7317"),
    seeds: Iterable[int] = (64,),
    model_a: str = "PairMLP-Cross",
    model_b: str = "ZHMolGraph",
    rna_tag: str = "rnafm",
    protein_tag: str = "proteinprottrans",
) -> pd.DataFrame:
    output_root = Path(output_root)
    rows = []
    for dataset_name in datasets:
        bundle = load_dataset(data_root, dataset_name, profile_name, output_root, rna_tag, protein_tag)
        full_positive_degree = np.bincount(
            bundle.p_idx[bundle.y == 1], minlength=len(bundle.prot_keys)
        )
        for seed in seeds:
            splits = prepare_or_load_splits(bundle, output_root, "prot_cold", seed)
            for fold, split in enumerate(splits):
                train_idx, test_idx = split["train"], split["test"]
                train_positive = train_idx[bundle.y[train_idx] == 1]
                train_rna_degree = np.bincount(
                    bundle.r_idx[train_positive], minlength=len(bundle.rna_keys)
                )
                train_proteins = np.unique(bundle.p_idx[train_idx])
                prediction_frames = {}
                thresholds = {}
                for model_name in (model_a, model_b):
                    directory = model_directory(
                        output_root, profile_name, dataset_name, "prot_cold", seed, fold, model_name
                    )
                    prediction_frames[model_name] = pd.read_csv(directory / "test_predictions.csv")
                    thresholds[model_name] = float(json.loads((directory / "metrics.json").read_text())["threshold"])

                for protein_id in np.unique(bundle.p_idx[test_idx]):
                    test_rows = test_idx[bundle.p_idx[test_idx] == protein_id]
                    positive_partner_rnas = np.unique(
                        bundle.r_idx[test_rows[bundle.y[test_rows] == 1]]
                    )
                    orphan_count = int((train_rna_degree[positive_partner_rnas] == 0).sum())
                    partner_degrees = train_rna_degree[positive_partner_rnas]
                    record = {
                        "dataset": dataset_name,
                        "profile": profile_name,
                        "seed": seed,
                        "fold": fold,
                        "protein_id": int(protein_id),
                        "full_positive_degree": int(full_positive_degree[protein_id]),
                        "positive_partner_rnas": int(len(positive_partner_rnas)),
                        "orphan_partner_rnas": orphan_count,
                        "orphan_fraction": float(orphan_count / max(len(positive_partner_rnas), 1)),
                        "mean_remaining_partner_rna_degree": float(np.mean(partner_degrees)) if len(partner_degrees) else float("nan"),
                        "max_similarity_to_training_protein": _cosine_max(
                            bundle.x_prot[protein_id], bundle.x_prot[train_proteins]
                        ),
                    }
                    for model_name in (model_a, model_b):
                        frame = prediction_frames[model_name]
                        part = frame.loc[frame["p_idx"] == protein_id]
                        metrics = _safe_group_metrics(part, thresholds[model_name])
                        for key, value in metrics.items():
                            record[f"{model_name}_{key}"] = value
                    record["delta_MCC"] = record[f"{model_b}_MCC"] - record[f"{model_a}_MCC"]
                    record["delta_AUROC"] = record[f"{model_b}_AUROC"] - record[f"{model_a}_AUROC"]
                    rows.append(record)

    result = pd.DataFrame(rows)
    analysis_dir = output_root / profile_name / "protein_endpoint_analysis"
    analysis_dir.mkdir(parents=True, exist_ok=True)
    result.to_csv(analysis_dir / f"{model_b}_minus_{model_a}_per_protein.csv", index=False)

    # Per-protein AUROC is highly unstable for proteins with one or two positives.
    # Save transparent support-aware summaries rather than relying only on the raw scatter.
    from scipy.stats import spearmanr
    correlation_rows = []
    for dataset_name, group in result.groupby("dataset"):
        for x_column in [
            "full_positive_degree",
            "orphan_fraction",
            "mean_remaining_partner_rna_degree",
            "max_similarity_to_training_protein",
        ]:
            for y_column in ["delta_AUROC", "delta_MCC"]:
                part = group[[x_column, y_column]].dropna()
                rho, p_value = spearmanr(part[x_column], part[y_column])
                correlation_rows.append({
                    "dataset": dataset_name,
                    "x": x_column,
                    "y": y_column,
                    "n": len(part),
                    "spearman_rho": float(rho),
                    "p_value": float(p_value),
                })
    pd.DataFrame(correlation_rows).to_csv(
        analysis_dir / "protein_graph_utility_spearman.csv", index=False
    )

    degree_bins = [0, 1, 2, 5, 10, 25, 100, 500, np.inf]
    degree_labels = ["1", "2", "3-5", "6-10", "11-25", "26-100", "101-500", ">500"]
    result["degree_bin"] = pd.cut(
        result["full_positive_degree"], degree_bins, labels=degree_labels, include_lowest=True
    )
    binned = (
        result.groupby(["dataset", "degree_bin"], observed=False)
        .agg(
            n_proteins=("protein_id", "size"),
            mean_delta_AUROC=("delta_AUROC", "mean"),
            median_delta_AUROC=("delta_AUROC", "median"),
            mean_delta_MCC=("delta_MCC", "mean"),
            median_delta_MCC=("delta_MCC", "median"),
            mean_positive_support=("positive_partner_rnas", "mean"),
        )
        .reset_index()
    )
    binned.to_csv(analysis_dir / "protein_graph_utility_by_degree_bin.csv", index=False)

    try:
        import matplotlib.pyplot as plt
        for dataset_name, group in result.groupby("dataset"):
            for x_column, filename, xlabel, log_x in [
                ("full_positive_degree", "delta_vs_degree", "Held-out protein positive degree", True),
                ("orphan_fraction", "delta_vs_orphan_fraction", "Orphan partner-RNA fraction", False),
                ("max_similarity_to_training_protein", "delta_vs_similarity", "Maximum protein embedding similarity", False),
            ]:
                fig, ax = plt.subplots(figsize=(7, 5))
                sizes = 18 + 8 * np.log1p(group["positive_partner_rnas"].to_numpy())
                ax.scatter(group[x_column], group["delta_AUROC"], s=sizes, alpha=0.55)
                ax.axhline(0.0, linewidth=1)
                if log_x:
                    ax.set_xscale("log")
                ax.set_xlabel(xlabel)
                ax.set_ylabel(f"AUROC difference: {model_b} - {model_a}")
                ax.set_title(f"{dataset_name}: protein-held-out graph utility")
                fig.tight_layout()
                fig.savefig(analysis_dir / f"{filename}_{dataset_name}.png", dpi=200)
                plt.close(fig)
    except Exception as exc:
        (analysis_dir / "plot_error.txt").write_text(str(exc))
    return result


# =============================================================================
# Paired comparisons and packaging
# =============================================================================

def _holm_adjust(p_values: list[float]) -> list[float]:
    values = np.asarray(p_values, dtype=float)
    adjusted = np.full(len(values), np.nan, dtype=float)
    valid = np.flatnonzero(np.isfinite(values))
    if len(valid) == 0:
        return adjusted.tolist()
    order = valid[np.argsort(values[valid])]
    running = 0.0
    m = len(order)
    for rank, index in enumerate(order):
        candidate = (m - rank) * values[index]
        running = max(running, candidate)
        adjusted[index] = min(running, 1.0)
    return adjusted.tolist()


def paired_fold_comparisons(
    output_root: str | Path,
    reference_model: str = "PairMLP-Cross",
    metric: str = "MCC",
) -> pd.DataFrame:
    frame = collect_metrics(output_root)
    if frame.empty:
        return frame
    try:
        from scipy.stats import ttest_rel, wilcoxon
    except Exception as exc:
        raise RuntimeError("scipy is required for paired tests") from exc

    rows = []
    grouping = ["profile", "dataset", "protocol", "run_seed"]
    for keys, group in frame.groupby(grouping):
        reference = group.loc[group["model"] == reference_model, ["fold", metric]].dropna()
        for model_name in sorted(set(group["model"]) - {reference_model}):
            other = group.loc[group["model"] == model_name, ["fold", metric]].dropna()
            merged = reference.merge(other, on="fold", suffixes=("_ref", "_other"))
            if len(merged) < 2:
                continue
            difference = merged[f"{metric}_other"] - merged[f"{metric}_ref"]
            t_result = ttest_rel(
                merged[f"{metric}_other"], merged[f"{metric}_ref"], nan_policy="omit"
            )
            try:
                w_result = wilcoxon(difference)
                w_p = float(w_result.pvalue)
            except Exception:
                w_p = float("nan")
            rows.append({
                **dict(zip(grouping, keys if isinstance(keys, tuple) else (keys,))),
                "reference_model": reference_model,
                "comparison_model": model_name,
                "metric": metric,
                "n_folds": len(merged),
                "wins_comparison": int((difference > 0).sum()),
                "losses_comparison": int((difference < 0).sum()),
                "ties": int((difference == 0).sum()),
                "mean_difference_comparison_minus_reference": float(difference.mean()),
                "median_difference_comparison_minus_reference": float(difference.median()),
                "paired_t_p": float(t_result.pvalue),
                "wilcoxon_p": w_p,
            })
    result = pd.DataFrame(rows)
    if len(result):
        result["paired_t_p_holm"] = _holm_adjust(result["paired_t_p"].tolist())
        result["wilcoxon_p_holm"] = _holm_adjust(result["wilcoxon_p"].tolist())
    result.to_csv(Path(output_root) / f"paired_fold_tests_{metric}.csv", index=False)
    return result


def pooled_oof_results(output_root: str | Path) -> pd.DataFrame:
    output_root = Path(output_root)
    frame = collect_metrics(output_root)
    rows = []
    grouping = ["profile", "dataset", "protocol", "run_seed", "model"]
    for keys, group in frame.groupby(grouping):
        profile, dataset, protocol, run_seed, model = keys
        predictions = []
        fold_metrics = []
        for _, metric_row in group.sort_values("fold").iterrows():
            fold = int(metric_row["fold"])
            directory = model_directory(
                output_root, profile, dataset, protocol, int(run_seed), fold, model
            )
            path = directory / "test_predictions.csv"
            if not path.exists():
                continue
            part = pd.read_csv(path)
            part["fold"] = fold
            part["fold_threshold"] = float(metric_row["threshold"])
            part["prediction"] = (part["probability"] >= part["fold_threshold"]).astype(int)
            predictions.append(part)
            fold_metrics.append(metric_row)
        if not predictions:
            continue
        pooled = pd.concat(predictions, ignore_index=True)
        y = pooled["y"].to_numpy(int)
        prob = pooled["probability"].to_numpy(float)
        pred = pooled["prediction"].to_numpy(int)
        tp, tn, fp, fn = confusion_counts(y, pred)
        weights = np.asarray([len(p) for p in predictions], dtype=float)
        fold_frame = pd.DataFrame(fold_metrics)
        row = {
            "profile": profile,
            "dataset": dataset,
            "protocol": protocol,
            "run_seed": int(run_seed),
            "model": model,
            "n_folds": len(predictions),
            "n_pairs": len(pooled),
            "pooled_MCC": mcc_from_counts(tp, tn, fp, fn),
            "pooled_AUROC": float(roc_auc_score(y, prob)),
            "pooled_AUPRC": float(average_precision_score(y, prob)),
            "pooled_ACC": float((tp + tn) / max(tp + tn + fp + fn, 1)),
            "pooled_SEN": float(tp / max(tp + fn, 1)),
            "pooled_SPE": float(tn / max(tn + fp, 1)),
            "macro_MCC": float(fold_frame["MCC"].mean()),
            "macro_AUROC": float(fold_frame["AUROC"].mean()),
            "macro_AUPRC": float(fold_frame["AUPRC"].mean()),
            "weighted_MCC": float(np.average(fold_frame["MCC"], weights=weights)),
            "weighted_AUROC": float(np.average(fold_frame["AUROC"], weights=weights)),
            "weighted_AUPRC": float(np.average(fold_frame["AUPRC"], weights=weights)),
        }
        rows.append(row)
    result = pd.DataFrame(rows)
    result.to_csv(output_root / "pooled_oof_results.csv", index=False)
    return result


def _expected_calibration_error(y: np.ndarray, prob: np.ndarray, n_bins: int = 10) -> float:
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    total = len(y)
    value = 0.0
    for left, right in zip(edges[:-1], edges[1:]):
        mask = (prob >= left) & (prob < right if right < 1.0 else prob <= right)
        if mask.any():
            value += mask.mean() * abs(prob[mask].mean() - y[mask].mean())
    return float(value)


def calibration_diagnostics(output_root: str | Path, n_bins: int = 10) -> tuple[pd.DataFrame, pd.DataFrame]:
    output_root = Path(output_root)
    frame = collect_metrics(output_root)
    fold_rows = []
    reliability_rows = []
    grouping = ["profile", "dataset", "protocol", "run_seed", "model"]
    for keys, group in frame.groupby(grouping):
        profile, dataset, protocol, run_seed, model = keys
        pooled_parts = []
        for _, metric_row in group.sort_values("fold").iterrows():
            fold = int(metric_row["fold"])
            directory = model_directory(output_root, profile, dataset, protocol, int(run_seed), fold, model)
            path = directory / "test_predictions.csv"
            if not path.exists():
                continue
            part = pd.read_csv(path)
            y = part["y"].to_numpy(int)
            prob = part["probability"].to_numpy(float)
            logits = part["logit"].to_numpy(float)
            threshold = float(metric_row["threshold"])
            applied_mcc = mcc_from_counts(*confusion_counts(y, prob >= threshold))
            oracle_threshold, oracle_mcc = best_threshold_by_mcc(y, logits)
            fold_rows.append({
                "profile": profile,
                "dataset": dataset,
                "protocol": protocol,
                "run_seed": int(run_seed),
                "model": model,
                "fold": fold,
                "n_pairs": len(part),
                "positive_rate": float(y.mean()),
                "validation_threshold": threshold,
                "test_oracle_threshold_diagnostic_only": oracle_threshold,
                "test_MCC_with_validation_threshold": applied_mcc,
                "test_oracle_MCC_diagnostic_only": oracle_mcc,
                "threshold_transfer_gap": float(oracle_mcc - applied_mcc),
                "brier_score": float(np.mean((prob - y) ** 2)),
                "ECE": _expected_calibration_error(y, prob, n_bins),
            })
            pooled_parts.append(part[["y", "probability"]])
        if pooled_parts:
            pooled = pd.concat(pooled_parts, ignore_index=True)
            y = pooled["y"].to_numpy(int)
            prob = pooled["probability"].to_numpy(float)
            edges = np.linspace(0.0, 1.0, n_bins + 1)
            for bin_id, (left, right) in enumerate(zip(edges[:-1], edges[1:])):
                mask = (prob >= left) & (prob < right if right < 1.0 else prob <= right)
                if mask.any():
                    reliability_rows.append({
                        "profile": profile,
                        "dataset": dataset,
                        "protocol": protocol,
                        "run_seed": int(run_seed),
                        "model": model,
                        "bin": bin_id,
                        "bin_left": left,
                        "bin_right": right,
                        "count": int(mask.sum()),
                        "mean_probability": float(prob[mask].mean()),
                        "observed_positive_rate": float(y[mask].mean()),
                    })
    fold_result = pd.DataFrame(fold_rows)
    reliability = pd.DataFrame(reliability_rows)
    fold_result.to_csv(output_root / "calibration_fold_diagnostics.csv", index=False)
    reliability.to_csv(output_root / "calibration_reliability_bins.csv", index=False)
    return fold_result, reliability


def zip_results(output_root: str | Path, destination: Optional[str | Path] = None) -> Path:
    output_root = Path(output_root)
    if destination is None:
        destination = output_root.parent / f"{output_root.name}.zip"
    destination = Path(destination)
    if destination.exists():
        destination.unlink()
    with zipfile.ZipFile(destination, "w", zipfile.ZIP_DEFLATED) as archive:
        for path in output_root.rglob("*"):
            if path.is_file():
                archive.write(path, arcname=path.relative_to(output_root))
    return destination


## 3. Reliability-analysis utilities


In [ ]:
%%writefile journal_dependability.py
from __future__ import annotations

import gc
import json
import math
import os
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.optimize import minimize_scalar
from scipy.spatial.distance import jensenshannon
from scipy.stats import mannwhitneyu, spearmanr, wilcoxon
from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupKFold, StratifiedKFold
try:
    from sklearn.model_selection import StratifiedGroupKFold
except ImportError:
    StratifiedGroupKFold = None
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

import exact_pair_models as pm
import journal_benchmark as ub

EPS = 1e-12

# -----------------------------------------------------------------------------
# General helpers
# -----------------------------------------------------------------------------

def ensure_dir(path: str | Path) -> Path:
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def safe_metric(func, *args, default=np.nan, **kwargs):
    try:
        return float(func(*args, **kwargs))
    except Exception:
        return default


def binary_entropy(prob: np.ndarray) -> np.ndarray:
    p = np.clip(np.asarray(prob, dtype=float), EPS, 1.0 - EPS)
    return -(p * np.log(p) + (1.0 - p) * np.log(1.0 - p)) / np.log(2.0)


def confidence_from_probability(prob: np.ndarray) -> np.ndarray:
    p = np.asarray(prob, dtype=float)
    return np.maximum(p, 1.0 - p)


def expected_calibration_error_positive(y: np.ndarray, prob: np.ndarray, n_bins: int = 10) -> float:
    """Standard binary ECE: mean positive probability vs observed positive rate."""
    y = np.asarray(y, dtype=int)
    p = np.asarray(prob, dtype=float)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i, (lo, hi) in enumerate(zip(bins[:-1], bins[1:])):
        mask = (p >= lo) & (p <= hi if i == n_bins - 1 else p < hi)
        if mask.any():
            ece += mask.mean() * abs(y[mask].mean() - p[mask].mean())
    return float(ece)


def adaptive_ece_positive(y: np.ndarray, prob: np.ndarray, n_bins: int = 10) -> float:
    y = np.asarray(y, dtype=int)
    p = np.asarray(prob, dtype=float)
    order = np.argsort(p)
    bins = np.array_split(order, min(n_bins, len(order)))
    ece = 0.0
    for idx in bins:
        if len(idx):
            ece += (len(idx) / len(y)) * abs(y[idx].mean() - p[idx].mean())
    return float(ece)


def confidence_ece(y: np.ndarray, prob: np.ndarray, n_bins: int = 10) -> float:
    y = np.asarray(y, dtype=int)
    p = np.asarray(prob, dtype=float)
    pred = (p >= 0.5).astype(int)
    conf = confidence_from_probability(p)
    correct = (pred == y).astype(float)
    bins = np.linspace(0.5, 1.0, n_bins + 1)
    ece = 0.0
    for i, (lo, hi) in enumerate(zip(bins[:-1], bins[1:])):
        mask = (conf >= lo) & (conf <= hi if i == n_bins - 1 else conf < hi)
        if mask.any():
            ece += mask.mean() * abs(correct[mask].mean() - conf[mask].mean())
    return float(ece)


def calibration_bins(y: np.ndarray, prob: np.ndarray, n_bins: int = 10) -> pd.DataFrame:
    y = np.asarray(y, dtype=int)
    p = np.asarray(prob, dtype=float)
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    rows = []
    for i, (lo, hi) in enumerate(zip(edges[:-1], edges[1:])):
        mask = (p >= lo) & (p <= hi if i == n_bins - 1 else p < hi)
        rows.append({
            "bin": i,
            "lower": lo,
            "upper": hi,
            "n": int(mask.sum()),
            "mean_probability": float(p[mask].mean()) if mask.any() else np.nan,
            "observed_positive_rate": float(y[mask].mean()) if mask.any() else np.nan,
        })
    return pd.DataFrame(rows)


def fit_temperature(val_logits: np.ndarray, val_y: np.ndarray) -> float:
    logits = np.asarray(val_logits, dtype=float)
    y = np.asarray(val_y, dtype=int)
    def objective(log_t: float) -> float:
        t = math.exp(log_t)
        p = np.clip(ub.sigmoid_np(logits / t), EPS, 1 - EPS)
        return log_loss(y, p, labels=[0, 1])
    result = minimize_scalar(objective, bounds=(-4.0, 4.0), method="bounded")
    return float(math.exp(result.x)) if result.success else 1.0


def classification_metrics(y: np.ndarray, prob: np.ndarray, pred: Optional[np.ndarray] = None) -> dict[str, float]:
    y = np.asarray(y, dtype=int)
    p = np.asarray(prob, dtype=float)
    pred = np.asarray(pred if pred is not None else p >= 0.5, dtype=int)
    return {
        "AUROC": safe_metric(roc_auc_score, y, p) if len(np.unique(y)) == 2 else np.nan,
        "AUPRC": safe_metric(average_precision_score, y, p) if len(np.unique(y)) == 2 else np.nan,
        "MCC": safe_metric(matthews_corrcoef, y, pred),
        "ACC": safe_metric(accuracy_score, y, pred),
        "PRE": safe_metric(precision_score, y, pred, zero_division=0),
        "REC": safe_metric(recall_score, y, pred, zero_division=0),
        "F1": safe_metric(f1_score, y, pred, zero_division=0),
        "ERROR": float((pred != y).mean()),
    }


def reliability_metrics(y: np.ndarray, prob: np.ndarray, pred: Optional[np.ndarray] = None) -> dict[str, float]:
    y = np.asarray(y, dtype=int)
    p = np.asarray(prob, dtype=float)
    pred = np.asarray(pred if pred is not None else p >= 0.5, dtype=int)
    conf = confidence_from_probability(p)
    wrong = pred != y
    out = {
        "BRIER": safe_metric(brier_score_loss, y, p),
        "NLL": safe_metric(log_loss, y, np.clip(p, EPS, 1 - EPS), labels=[0, 1]),
        "ECE": expected_calibration_error_positive(y, p),
        "ADAPTIVE_ECE": adaptive_ece_positive(y, p),
        "CONF_ECE": confidence_ece(y, p),
        "MEAN_CONF": float(conf.mean()),
    }
    for threshold in (0.80, 0.90, 0.95):
        mask = conf >= threshold
        key = str(int(threshold * 100))
        out[f"HC{key}_COVERAGE"] = float(mask.mean())
        out[f"HC{key}_ERROR"] = float(wrong[mask].mean()) if mask.any() else np.nan
        out[f"HC{key}_N_WRONG"] = int((wrong & mask).sum())
    return out


# -----------------------------------------------------------------------------
# Collect saved benchmark predictions and add structural/calibration features
# -----------------------------------------------------------------------------

def _path_metadata(prediction_path: Path, output_root: Path) -> dict[str, Any]:
    model_dir = prediction_path.parent
    metrics_path = model_dir / "metrics.json"
    if not metrics_path.exists():
        raise FileNotFoundError(metrics_path)
    metrics = json.loads(metrics_path.read_text())
    rel = model_dir.relative_to(output_root)
    parts = rel.parts
    # profile/dataset/protocol/seed_X/fold_Y/model_slug
    if len(parts) < 6:
        raise ValueError(f"Unexpected model directory: {model_dir}")
    return {
        "profile": metrics.get("profile", parts[0]),
        "dataset": metrics.get("dataset", parts[1]),
        "protocol": metrics.get("protocol", parts[2]),
        "seed": int(metrics.get("run_seed", metrics.get("seed", parts[3].split("_")[-1]))),
        "fold": int(metrics.get("fold", parts[4].split("_")[-1])),
        "model": metrics.get("model", parts[5]),
        "threshold": float(metrics.get("threshold", 0.5)),
        "model_dir": model_dir,
        "metrics": metrics,
    }



def _centroid_ood_features(x: np.ndarray, train_entity_ids: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Deployable entity-level OOD descriptors fitted only on training entities.

    Returns
    -------
    cosine_to_train_centroid:
        Cosine similarity to the training-entity centroid (higher = more familiar).
    diagonal_ood:
        RMS standardized distance from the training distribution (higher = more shifted).
    """
    x = np.asarray(x, dtype=np.float32)
    ids = np.unique(np.asarray(train_entity_ids, dtype=int))
    if len(ids) == 0:
        return np.zeros(len(x), dtype=np.float32), np.zeros(len(x), dtype=np.float32)
    ref = x[ids]
    mu = ref.mean(axis=0)
    sigma = ref.std(axis=0)
    sigma = np.where(sigma < 1e-6, 1.0, sigma)
    denom = np.linalg.norm(x, axis=1) * max(float(np.linalg.norm(mu)), 1e-12)
    cosine = (x @ mu) / np.maximum(denom, 1e-12)
    diagonal_ood = np.sqrt(np.mean(((x - mu) / sigma) ** 2, axis=1))
    return cosine.astype(np.float32), diagonal_ood.astype(np.float32)


def collect_prediction_records(
    data_root: str | Path,
    output_root: str | Path,
    analysis_root: str | Path,
    rna_tag: str = "rnafm",
    protein_tag: str = "proteinprottrans",
    protocols: Optional[Iterable[str]] = None,
    datasets: Optional[Iterable[str]] = None,
    models: Optional[Iterable[str]] = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    data_root, output_root = Path(data_root), Path(output_root)
    analysis_root = ensure_dir(analysis_root)
    protocol_set = set(protocols) if protocols else None
    dataset_set = set(datasets) if datasets else None
    model_set = set(models) if models else None
    bundle_cache: dict[tuple[str, str], ub.DatasetBundle] = {}
    split_feature_cache: dict[tuple[str, str, str, int, int], dict[str, np.ndarray]] = {}
    frames, temp_rows = [], []

    pred_paths = sorted(output_root.rglob("test_predictions.csv.gz"))
    if not pred_paths:
        pred_paths = sorted(output_root.rglob("test_predictions.csv"))
    for pred_path in pred_paths:
        try:
            meta = _path_metadata(pred_path, output_root)
        except Exception:
            continue
        if protocol_set and meta["protocol"] not in protocol_set:
            continue
        if dataset_set and meta["dataset"] not in dataset_set:
            continue
        if model_set and meta["model"] not in model_set:
            continue

        key = (meta["profile"], meta["dataset"])
        if key not in bundle_cache:
            bundle_cache[key] = ub.load_dataset(
                data_root, meta["dataset"], meta["profile"], output_root,
                rna_tag=rna_tag, protein_tag=protein_tag,
            )
        bundle = bundle_cache[key]
        split_dir = ub.split_directory(
            output_root, meta["profile"], meta["dataset"], meta["protocol"], meta["seed"], meta["fold"]
        )
        train_idx = np.load(split_dir / "inner_train_indices.npy")
        val_idx = np.load(split_dir / "validation_indices.npy")

        full_rna_deg = np.bincount(bundle.r_idx[bundle.y == 1], minlength=len(bundle.x_rna))
        full_prot_deg = np.bincount(bundle.p_idx[bundle.y == 1], minlength=len(bundle.x_prot))
        tr_pos = train_idx[bundle.y[train_idx] == 1]
        train_rna_deg = np.bincount(bundle.r_idx[tr_pos], minlength=len(bundle.x_rna))
        train_prot_deg = np.bincount(bundle.p_idx[tr_pos], minlength=len(bundle.x_prot))

        split_key = (
            meta["profile"], meta["dataset"], meta["protocol"], meta["seed"], meta["fold"]
        )
        if split_key not in split_feature_cache:
            train_rna_ids = np.unique(bundle.r_idx[train_idx])
            train_prot_ids = np.unique(bundle.p_idx[train_idx])
            rna_cos, rna_ood = _centroid_ood_features(bundle.x_rna, train_rna_ids)
            prot_cos, prot_ood = _centroid_ood_features(bundle.x_prot, train_prot_ids)
            split_feature_cache[split_key] = {
                "rna_centroid_cosine": rna_cos,
                "rna_diagonal_ood": rna_ood,
                "protein_centroid_cosine": prot_cos,
                "protein_diagonal_ood": prot_ood,
            }

        frame = pd.read_csv(pred_path)
        frame["profile"] = meta["profile"]
        frame["dataset"] = meta["dataset"]
        frame["protocol"] = meta["protocol"]
        frame["seed"] = meta["seed"]
        frame["fold"] = meta["fold"]
        frame["model"] = meta["model"]
        frame["threshold"] = meta["threshold"]
        frame["run_id"] = (
            frame["dataset"] + "|" + frame["protocol"] + "|" + frame["model"]
            + "|s" + frame["seed"].astype(str) + "|f" + frame["fold"].astype(str)
        )
        frame["pair_id"] = frame["dataset"] + "|" + frame["r_idx"].astype(str) + "|" + frame["p_idx"].astype(str)
        frame["raw_probability"] = frame.get("probability", ub.sigmoid_np(frame["logit"].to_numpy()))
        frame["raw_prediction"] = frame.get(
            "prediction", (frame["raw_probability"] >= meta["threshold"]).astype(int)
        )
        frame["wrong"] = (frame["raw_prediction"].astype(int) != frame["y"].astype(int)).astype(int)
        frame["raw_confidence"] = confidence_from_probability(frame["raw_probability"])
        frame["raw_entropy"] = binary_entropy(frame["raw_probability"])
        frame["raw_margin"] = np.abs(frame["raw_probability"] - 0.5) * 2.0

        rids, pids = frame["r_idx"].to_numpy(int), frame["p_idx"].to_numpy(int)
        frame["full_rna_degree"] = full_rna_deg[rids]
        frame["full_protein_degree"] = full_prot_deg[pids]
        frame["train_rna_degree"] = train_rna_deg[rids]
        frame["train_protein_degree"] = train_prot_deg[pids]
        frame["rna_orphan"] = (frame["train_rna_degree"] == 0).astype(int)
        frame["protein_orphan"] = (frame["train_protein_degree"] == 0).astype(int)
        for col in ["full_rna_degree", "full_protein_degree", "train_rna_degree", "train_protein_degree"]:
            frame[f"log1p_{col}"] = np.log1p(frame[col])

        split_features = split_feature_cache[split_key]
        frame["rna_centroid_cosine"] = split_features["rna_centroid_cosine"][rids]
        frame["protein_centroid_cosine"] = split_features["protein_centroid_cosine"][pids]
        frame["rna_diagonal_ood"] = split_features["rna_diagonal_ood"][rids]
        frame["protein_diagonal_ood"] = split_features["protein_diagonal_ood"][pids]

        # Full positive degrees use labels from the complete benchmark graph.
        # Keep them only as retrospective diagnostic variables, never as
        # pre-deployment failure-detector inputs.
        frame["diagnostic_full_rna_degree"] = frame["full_rna_degree"]
        frame["diagnostic_full_protein_degree"] = frame["full_protein_degree"]

        val_csv = meta["model_dir"] / "validation_predictions.csv.gz"
        if val_csv.exists():
            val_frame = pd.read_csv(val_csv)
            val_logits = val_frame["logit"].to_numpy(float)
            val_y = val_frame["y"].to_numpy(int)
        else:
            val_npz = np.load(meta["model_dir"] / "validation_predictions.npz")
            val_logits = val_npz["logits"]
            val_y = val_npz["y"]
        temperature = fit_temperature(val_logits, val_y)
        frame["temperature"] = temperature
        frame["temp_probability"] = ub.sigmoid_np(frame["logit"].to_numpy() / temperature)
        frame["temp_confidence"] = confidence_from_probability(frame["temp_probability"])
        frame["temp_entropy"] = binary_entropy(frame["temp_probability"])
        temp_rows.append({
            "profile": meta["profile"], "dataset": meta["dataset"], "protocol": meta["protocol"],
            "seed": meta["seed"], "fold": meta["fold"], "model": meta["model"],
            "temperature": temperature,
        })
        frames.append(frame)

    if not frames:
        raise FileNotFoundError(f"No test_predictions.csv files found under {output_root}")
    all_predictions = pd.concat(frames, ignore_index=True)
    temperatures = pd.DataFrame(temp_rows)
    all_predictions.to_csv(analysis_root / "all_predictions_enriched.csv.gz", index=False, compression="gzip")
    temperatures.to_csv(analysis_root / "temperature_scaling_parameters.csv", index=False)
    return all_predictions, temperatures


def summarize_saved_predictions(predictions: pd.DataFrame, analysis_root: str | Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    analysis_root = ensure_dir(analysis_root)
    rows = []
    group_cols = ["profile", "dataset", "protocol", "model", "seed", "fold"]
    for keys, g in predictions.groupby(group_cols):
        y = g["y"].to_numpy(int)
        raw_prob = g["raw_probability"].to_numpy(float)
        raw_pred = g["raw_prediction"].to_numpy(int)
        temp_prob = g["temp_probability"].to_numpy(float)
        row = dict(zip(group_cols, keys))
        row.update({f"RAW_{k}": v for k, v in classification_metrics(y, raw_prob, raw_pred).items()})
        row.update({f"RAW_{k}": v for k, v in reliability_metrics(y, raw_prob, raw_pred).items()})
        row.update({f"TEMP_{k}": v for k, v in reliability_metrics(y, temp_prob, raw_pred).items()})
        row["N"] = len(g)
        rows.append(row)
    fold_summary = pd.DataFrame(rows)
    fold_summary.to_csv(analysis_root / "dependability_fold_summary.csv", index=False)

    metric_cols = [c for c in fold_summary.columns if c not in group_cols and c != "N"]
    overall = (
        fold_summary.groupby(["profile", "dataset", "protocol", "model"])[metric_cols]
        .agg(["mean", "std", "count"]).reset_index()
    )
    overall.columns = [
        "_".join(str(x) for x in col if str(x)) if isinstance(col, tuple) else col
        for col in overall.columns
    ]
    overall.to_csv(analysis_root / "dependability_overall_summary.csv", index=False)
    return fold_summary, overall


# -----------------------------------------------------------------------------
# Selective prediction and abstention
# -----------------------------------------------------------------------------

def selective_curve_for_group(g: pd.DataFrame, risk_col: str, coverages: Iterable[float]) -> pd.DataFrame:
    ordered = g.sort_values(risk_col, ascending=True).reset_index(drop=True)
    rows = []
    for coverage in coverages:
        k = max(1, int(math.ceil(float(coverage) * len(ordered))))
        accepted = ordered.iloc[:k]
        metrics = classification_metrics(
            accepted["y"].to_numpy(int),
            accepted["raw_probability"].to_numpy(float),
            accepted["raw_prediction"].to_numpy(int),
        )
        rows.append({"coverage": float(coverage), "accepted_n": k, **metrics})
    return pd.DataFrame(rows)


def aurc_eaurc(y: np.ndarray, wrong: np.ndarray, risk: np.ndarray) -> tuple[float, float]:
    y = np.asarray(y)
    errors = np.asarray(wrong, dtype=float)
    order = np.argsort(risk)
    cumulative_risk = np.cumsum(errors[order]) / np.arange(1, len(errors) + 1)
    aurc = float(cumulative_risk.mean())
    # Oracle ordering accepts all correct cases first, then errors.
    oracle_order = np.argsort(errors)
    oracle = np.cumsum(errors[oracle_order]) / np.arange(1, len(errors) + 1)
    optimal_aurc = float(oracle.mean())
    return aurc, aurc - optimal_aurc


def run_selective_analysis(
    predictions: pd.DataFrame,
    analysis_root: str | Path,
    coverages: Iterable[float] = (1.0, 0.95, 0.9, 0.8, 0.7, 0.6, 0.5),
    risk_columns: Optional[dict[str, str]] = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    analysis_root = ensure_dir(analysis_root)
    predictions = predictions.copy()
    risk_columns = risk_columns or {
        "raw_confidence": "risk_raw_confidence",
        "temp_confidence": "risk_temp_confidence",
        "raw_entropy": "risk_raw_entropy",
    }
    predictions["risk_raw_confidence"] = 1.0 - predictions["raw_confidence"]
    predictions["risk_temp_confidence"] = 1.0 - predictions["temp_confidence"]
    predictions["risk_raw_entropy"] = predictions["raw_entropy"]
    group_cols = ["profile", "dataset", "protocol", "model", "seed", "fold"]
    curve_rows, aurc_rows = [], []
    for keys, g in predictions.groupby(group_cols):
        base = dict(zip(group_cols, keys))
        for label, risk_col in risk_columns.items():
            curve = selective_curve_for_group(g, risk_col, coverages)
            for _, row in curve.iterrows():
                curve_rows.append({**base, "risk_method": label, **row.to_dict()})
            a, ea = aurc_eaurc(g["y"].to_numpy(), g["wrong"].to_numpy(), g[risk_col].to_numpy())
            aurc_rows.append({**base, "risk_method": label, "AURC": a, "E_AURC": ea})
    curves = pd.DataFrame(curve_rows)
    aurc = pd.DataFrame(aurc_rows)
    curves.to_csv(analysis_root / "selective_risk_coverage.csv", index=False)
    aurc.to_csv(analysis_root / "selective_aurc.csv", index=False)
    return curves, aurc


# -----------------------------------------------------------------------------
# Failure detector with leakage-resistant grouped meta-CV
# -----------------------------------------------------------------------------

DEFAULT_FAILURE_FEATURES = {
    "confidence_only": ["raw_confidence", "raw_margin", "raw_entropy"],
    "deployable_support_only": [
        "log1p_train_rna_degree", "log1p_train_protein_degree",
        "rna_orphan", "protein_orphan",
        "rna_centroid_cosine", "protein_centroid_cosine",
        "rna_diagonal_ood", "protein_diagonal_ood",
    ],
    "deployable_confidence_plus_support": [
        "raw_confidence", "raw_margin", "raw_entropy",
        "log1p_train_rna_degree", "log1p_train_protein_degree",
        "rna_orphan", "protein_orphan",
        "rna_centroid_cosine", "protein_centroid_cosine",
        "rna_diagonal_ood", "protein_diagonal_ood",
    ],
    "calibrated_deployable_plus_support": [
        "temp_confidence", "temp_entropy",
        "log1p_train_rna_degree", "log1p_train_protein_degree",
        "rna_orphan", "protein_orphan",
        "rna_centroid_cosine", "protein_centroid_cosine",
        "rna_diagonal_ood", "protein_diagonal_ood",
    ],
    "retrospective_oracle_structure": [
        "log1p_full_protein_degree", "log1p_full_rna_degree",
        "log1p_train_rna_degree", "log1p_train_protein_degree",
        "rna_orphan", "protein_orphan",
    ],
    "retrospective_oracle_combined": [
        "raw_confidence", "raw_margin", "raw_entropy",
        "log1p_full_protein_degree", "log1p_full_rna_degree",
        "log1p_train_rna_degree", "log1p_train_protein_degree",
        "rna_orphan", "protein_orphan",
    ],
}


def _make_meta_splits(X, y, groups, n_splits=5, random_state=17):
    unique_groups = np.unique(groups)
    n_splits = max(2, min(n_splits, len(unique_groups)))
    if StratifiedGroupKFold is not None:
        splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
        return list(splitter.split(X, y, groups))
    splitter = GroupKFold(n_splits=n_splits)
    return list(splitter.split(X, y, groups))


def _new_failure_classifier(kind: str, random_state: int = 17):
    if kind == "rf":
        return RandomForestClassifier(
            n_estimators=350, max_depth=6, min_samples_leaf=12,
            class_weight="balanced", random_state=random_state, n_jobs=-1,
        )
    if kind == "logreg":
        return make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=2000, class_weight="balanced", solver="lbfgs"),
        )
    raise ValueError(kind)


def evaluate_failure_detectors(
    predictions: pd.DataFrame,
    analysis_root: str | Path,
    feature_sets: Optional[dict[str, list[str]]] = None,
    classifiers: Iterable[str] = ("logreg", "rf"),
    n_splits: int = 5,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    analysis_root = ensure_dir(analysis_root)
    feature_sets = feature_sets or DEFAULT_FAILURE_FEATURES
    result_rows, scored_frames = [], []
    for (dataset, protocol, model), g0 in predictions.groupby(["dataset", "protocol", "model"]):
        g = g0.copy().reset_index(drop=True)
        y = g["wrong"].to_numpy(int)
        if len(np.unique(y)) < 2:
            continue
        # Keep the held-out entity disjoint across
        # failure-detector training and evaluation.
        if protocol == "prot_cold":
            groups = (g["dataset"].astype(str) + "|p" + g["p_idx"].astype(str)).to_numpy()
        elif protocol == "rna_cold":
            groups = (g["dataset"].astype(str) + "|r" + g["r_idx"].astype(str)).to_numpy()
        else:
            groups = g["pair_id"].astype(str).to_numpy()
        for feature_name, features in feature_sets.items():
            Xdf = g[features].replace([np.inf, -np.inf], np.nan).fillna(0.0)
            X = Xdf.to_numpy(float)
            splits = _make_meta_splits(X, y, groups, n_splits=n_splits)
            for classifier_name in classifiers:
                oof = np.full(len(g), np.nan, dtype=float)
                for fold_id, (tr, te) in enumerate(splits):
                    if len(np.unique(y[tr])) < 2:
                        oof[te] = y[tr].mean()
                        continue
                    clf = _new_failure_classifier(classifier_name, random_state=17 + fold_id)
                    clf.fit(X[tr], y[tr])
                    oof[te] = clf.predict_proba(X[te])[:, 1]
                valid = np.isfinite(oof)
                yy, ss = y[valid], oof[valid]
                order = np.argsort(-ss)
                top10 = max(1, int(0.10 * len(yy)))
                top20 = max(1, int(0.20 * len(yy)))
                result_rows.append({
                    "dataset": dataset, "protocol": protocol, "model": model,
                    "feature_set": feature_name, "classifier": classifier_name,
                    "n": int(len(yy)), "baseline_failure_rate": float(yy.mean()),
                    "failure_AUROC": safe_metric(roc_auc_score, yy, ss),
                    "failure_AUPRC": safe_metric(average_precision_score, yy, ss),
                    "top10_failure_rate": float(yy[order[:top10]].mean()),
                    "top20_failure_rate": float(yy[order[:top20]].mean()),
                    "top10_enrichment": float(yy[order[:top10]].mean() / max(yy.mean(), EPS)),
                })
                scored = g.loc[valid].copy()
                scored["failure_risk"] = ss
                scored["feature_set"] = feature_name
                scored["classifier"] = classifier_name
                scored_frames.append(scored)
    results = pd.DataFrame(result_rows)
    scored = pd.concat(scored_frames, ignore_index=True) if scored_frames else pd.DataFrame()
    results.to_csv(analysis_root / "failure_detector_results.csv", index=False)
    if len(scored):
        scored.to_csv(analysis_root / "failure_detector_oof_scores.csv.gz", index=False, compression="gzip")
    return results, scored


def cv_permutation_importance_failure_detector(
    predictions: pd.DataFrame,
    analysis_root: str | Path,
    features: Optional[list[str]] = None,
    n_splits: int = 5,
    n_repeats: int = 10,
) -> pd.DataFrame:
    analysis_root = ensure_dir(analysis_root)
    features = features or DEFAULT_FAILURE_FEATURES["confidence_plus_structure"]
    rows = []
    for (dataset, protocol, model), g0 in predictions.groupby(["dataset", "protocol", "model"]):
        g = g0.copy().reset_index(drop=True)
        y = g["wrong"].to_numpy(int)
        if len(np.unique(y)) < 2:
            continue
        Xdf = g[features].replace([np.inf, -np.inf], np.nan).fillna(0.0)
        X = Xdf.to_numpy(float)
        if protocol == "prot_cold":
            groups = (g["dataset"].astype(str) + "|p" + g["p_idx"].astype(str)).to_numpy()
        elif protocol == "rna_cold":
            groups = (g["dataset"].astype(str) + "|r" + g["r_idx"].astype(str)).to_numpy()
        else:
            groups = g["pair_id"].astype(str).to_numpy()
        splits = _make_meta_splits(X, y, groups, n_splits=n_splits)
        for cv_fold, (tr, te) in enumerate(splits):
            if len(np.unique(y[tr])) < 2 or len(np.unique(y[te])) < 2:
                continue
            clf = _new_failure_classifier("rf", random_state=100 + cv_fold)
            clf.fit(X[tr], y[tr])
            perm = permutation_importance(
                clf, X[te], y[te], scoring="roc_auc", n_repeats=n_repeats,
                random_state=200 + cv_fold, n_jobs=-1,
            )
            for feature, mean, std in zip(features, perm.importances_mean, perm.importances_std):
                rows.append({
                    "dataset": dataset, "protocol": protocol, "model": model,
                    "cv_fold": cv_fold, "feature": feature,
                    "importance": float(mean), "importance_std": float(std),
                })
    frame = pd.DataFrame(rows)
    frame.to_csv(analysis_root / "failure_detector_cv_permutation_importance.csv", index=False)
    return frame


def learned_risk_coverage(scored: pd.DataFrame, analysis_root: str | Path,
                          coverages=(1.0, 0.95, 0.9, 0.8, 0.7, 0.6, 0.5)) -> pd.DataFrame:
    analysis_root = ensure_dir(analysis_root)
    rows = []
    keys = ["dataset", "protocol", "model", "feature_set", "classifier"]
    for group_keys, g in scored.groupby(keys):
        ordered = g.sort_values("failure_risk", ascending=True).reset_index(drop=True)
        base = dict(zip(keys, group_keys))
        for coverage in coverages:
            k = max(1, int(math.ceil(coverage * len(ordered))))
            accepted = ordered.iloc[:k]
            metrics = classification_metrics(
                accepted["y"].to_numpy(int), accepted["raw_probability"].to_numpy(float),
                accepted["raw_prediction"].to_numpy(int),
            )
            rows.append({**base, "coverage": coverage, "accepted_n": k, **metrics})
    frame = pd.DataFrame(rows)
    frame.to_csv(analysis_root / "failure_detector_risk_coverage.csv", index=False)
    return frame


# -----------------------------------------------------------------------------
# Multi-seed / multi-version ensemble uncertainty
# -----------------------------------------------------------------------------

def build_seed_ensemble(predictions: pd.DataFrame, analysis_root: str | Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    analysis_root = ensure_dir(analysis_root)
    key_cols = ["profile", "dataset", "protocol", "model", "row_id"]
    agg = predictions.groupby(key_cols).agg(
        y=("y", "first"),
        r_idx=("r_idx", "first"),
        p_idx=("p_idx", "first"),
        ensemble_probability=("raw_probability", "mean"),
        ensemble_std=("raw_probability", "std"),
        n_versions=("raw_probability", "count"),
        mean_threshold=("threshold", "mean"),
        full_protein_degree=("full_protein_degree", "first"),
        train_rna_degree=("train_rna_degree", "mean"),
    ).reset_index()
    agg["ensemble_std"] = agg["ensemble_std"].fillna(0.0)
    agg["ensemble_prediction"] = (agg["ensemble_probability"] >= agg["mean_threshold"]).astype(int)
    agg["wrong"] = (agg["ensemble_prediction"] != agg["y"]).astype(int)
    agg["ensemble_confidence"] = confidence_from_probability(agg["ensemble_probability"])
    agg["ensemble_entropy"] = binary_entropy(agg["ensemble_probability"])
    agg["risk_disagreement"] = agg["ensemble_std"]
    agg["risk_combined"] = agg["ensemble_entropy"] + agg["ensemble_std"]
    agg.to_parquet(analysis_root / "seed_ensemble_predictions.parquet", index=False)

    rows = []
    for keys, g in agg.groupby(["profile", "dataset", "protocol", "model"]):
        y, p, pred = g["y"].to_numpy(), g["ensemble_probability"].to_numpy(), g["ensemble_prediction"].to_numpy()
        row = dict(zip(["profile", "dataset", "protocol", "model"], keys))
        row.update(classification_metrics(y, p, pred))
        row.update(reliability_metrics(y, p, pred))
        row["MEAN_STD"] = float(g["ensemble_std"].mean())
        rows.append(row)
    summary = pd.DataFrame(rows)
    summary.to_csv(analysis_root / "seed_ensemble_summary.csv", index=False)
    return agg, summary


# -----------------------------------------------------------------------------
# Distribution shift: MMD + degree/prevalence shift
# -----------------------------------------------------------------------------

def _sample_rows(X: np.ndarray, max_n: int, rng: np.random.Generator) -> np.ndarray:
    X = np.asarray(X)
    if len(X) <= max_n:
        return X
    return X[rng.choice(len(X), max_n, replace=False)]


def rbf_mmd2(X: np.ndarray, Y: np.ndarray, max_n: int = 1500, seed: int = 0) -> float:
    rng = np.random.default_rng(seed)
    X = _sample_rows(np.asarray(X, np.float32), max_n, rng)
    Y = _sample_rows(np.asarray(Y, np.float32), max_n, rng)
    # Standardize jointly so dimensions with larger raw variance do not dominate.
    Z = np.vstack([X, Y])
    mu, sd = Z.mean(0), Z.std(0) + 1e-6
    X, Y = (X - mu) / sd, (Y - mu) / sd
    Zs = _sample_rows(np.vstack([X, Y]), min(800, len(X) + len(Y)), rng)
    d = np.sum((Zs[:, None, :] - Zs[None, :, :]) ** 2, axis=-1)
    med = np.median(d[d > 0]) if np.any(d > 0) else 1.0
    gamma = 1.0 / max(med, EPS)
    def kernel(A, B):
        d2 = np.sum((A[:, None, :] - B[None, :, :]) ** 2, axis=-1)
        return np.exp(-gamma * d2)
    return float(kernel(X, X).mean() + kernel(Y, Y).mean() - 2 * kernel(X, Y).mean())


def _js_degree_shift(train_deg: np.ndarray, test_deg: np.ndarray, max_bin: int = 20) -> float:
    bins = np.arange(0, max_bin + 2)
    a = np.histogram(np.clip(train_deg, 0, max_bin + 1), bins=bins, density=False)[0].astype(float) + EPS
    b = np.histogram(np.clip(test_deg, 0, max_bin + 1), bins=bins, density=False)[0].astype(float) + EPS
    a /= a.sum(); b /= b.sum()
    return float(jensenshannon(a, b, base=2.0) ** 2)


def compute_fold_shift_table(
    data_root: str | Path,
    output_root: str | Path,
    analysis_root: str | Path,
    datasets: Iterable[str], protocols: Iterable[str], seeds: Iterable[int], n_splits: int,
    profile: str = "clean_120", rna_tag: str = "rnafm", protein_tag: str = "proteinprottrans",
) -> pd.DataFrame:
    analysis_root = ensure_dir(analysis_root)
    rows = []
    for dataset in datasets:
        bundle = ub.load_dataset(data_root, dataset, profile, output_root, rna_tag, protein_tag)
        full_r_deg = np.bincount(bundle.r_idx[bundle.y == 1], minlength=len(bundle.x_rna))
        full_p_deg = np.bincount(bundle.p_idx[bundle.y == 1], minlength=len(bundle.x_prot))
        for protocol in protocols:
            for seed in seeds:
                splits = ub.prepare_or_load_splits(bundle, output_root, protocol, seed, n_splits=n_splits)
                for fold, split in enumerate(splits):
                    tr, te = split["train"], split["test"]
                    tr_r, te_r = np.unique(bundle.r_idx[tr]), np.unique(bundle.r_idx[te])
                    tr_p, te_p = np.unique(bundle.p_idx[tr]), np.unique(bundle.p_idx[te])
                    rows.append({
                        "dataset": dataset, "protocol": protocol, "seed": seed, "fold": fold,
                        "mmd_rna": rbf_mmd2(bundle.x_rna[tr_r], bundle.x_rna[te_r], seed=seed + fold),
                        "mmd_protein": rbf_mmd2(bundle.x_prot[tr_p], bundle.x_prot[te_p], seed=seed + fold + 77),
                        "rna_degree_js": _js_degree_shift(full_r_deg[tr_r], full_r_deg[te_r]),
                        "protein_degree_js": _js_degree_shift(full_p_deg[tr_p], full_p_deg[te_p]),
                        "train_prevalence": float(bundle.y[tr].mean()),
                        "test_prevalence": float(bundle.y[te].mean()),
                        "prevalence_delta": float(abs(bundle.y[tr].mean() - bundle.y[te].mean())),
                        "n_train": len(tr), "n_test": len(te),
                    })
    frame = pd.DataFrame(rows)
    frame.to_csv(analysis_root / "fold_distribution_shift.csv", index=False)
    return frame


def correlate_shift_with_performance(shift: pd.DataFrame, fold_summary: pd.DataFrame,
                                     analysis_root: str | Path) -> pd.DataFrame:
    analysis_root = ensure_dir(analysis_root)
    merged = fold_summary.merge(shift, on=["dataset", "protocol", "seed", "fold"], how="left")
    rows = []
    shifts = ["mmd_rna", "mmd_protein", "rna_degree_js", "protein_degree_js", "prevalence_delta"]
    outcomes = ["RAW_AUROC", "RAW_MCC", "RAW_ERROR", "RAW_ECE"]
    for (dataset, protocol, model), g in merged.groupby(["dataset", "protocol", "model"]):
        for x in shifts:
            for y in outcomes:
                valid = g[[x, y]].dropna()
                rho, p = spearmanr(valid[x], valid[y]) if len(valid) >= 4 else (np.nan, np.nan)
                rows.append({
                    "dataset": dataset, "protocol": protocol, "model": model,
                    "shift_metric": x, "outcome": y, "spearman_rho": rho, "p_value": p,
                    "n": len(valid),
                })
    corr = pd.DataFrame(rows)
    merged.to_csv(analysis_root / "shift_performance_merged.csv", index=False)
    corr.to_csv(analysis_root / "shift_performance_correlations.csv", index=False)
    return corr


# -----------------------------------------------------------------------------
# Inference-time fault injection for exact pair/GNN models
# -----------------------------------------------------------------------------

def _build_sparse_adjacency_from_edges(r: np.ndarray, p: np.ndarray, n_rna: int, n_prot: int,
                                       device: torch.device) -> torch.Tensor:
    if len(r) == 0:
        idx = torch.zeros((2, 0), dtype=torch.long, device=device)
        val = torch.zeros(0, dtype=torch.float32, device=device)
    else:
        idx = torch.as_tensor(np.vstack([r, p]), dtype=torch.long, device=device)
        val = torch.ones(len(r), dtype=torch.float32, device=device)
    return torch.sparse_coo_tensor(idx, val, (n_rna, n_prot), device=device).coalesce()


def _corrupt_graph(bundle: ub.DatasetBundle, train_idx: np.ndarray, severity: float,
                   mode: str, seed: int, device: torch.device) -> torch.Tensor:
    pos = train_idx[bundle.y[train_idx] == 1]
    r, p = bundle.r_idx[pos], bundle.p_idx[pos]
    n_remove = int(round(severity * len(pos)))
    if n_remove <= 0:
        keep = np.ones(len(pos), dtype=bool)
    elif mode == "random":
        rng = np.random.default_rng(seed)
        remove = rng.choice(len(pos), n_remove, replace=False)
        keep = np.ones(len(pos), dtype=bool); keep[remove] = False
    elif mode == "hub_targeted":
        deg = np.bincount(p, minlength=len(bundle.x_prot))
        edge_score = deg[p]
        remove = np.argsort(-edge_score)[:n_remove]
        keep = np.ones(len(pos), dtype=bool); keep[remove] = False
    else:
        raise ValueError(mode)
    return _build_sparse_adjacency_from_edges(r[keep], p[keep], len(bundle.x_rna), len(bundle.x_prot), device)


def _corrupt_embeddings(x: np.ndarray, severity: float, mode: str, seed: int) -> np.ndarray:
    x = np.asarray(x, np.float32).copy()
    if severity <= 0:
        return x
    rng = np.random.default_rng(seed)
    if mode == "gaussian":
        scale = x.std(axis=0, keepdims=True) + 1e-6
        x += rng.normal(0.0, severity, size=x.shape).astype(np.float32) * scale
    elif mode == "mask":
        mask = rng.random(x.shape) < severity
        x[mask] = 0.0
    else:
        raise ValueError(mode)
    return x


def run_fault_injection(
    data_root: str | Path, output_root: str | Path, analysis_root: str | Path,
    datasets: Iterable[str], models: Iterable[str], seeds: Iterable[int], folds: Iterable[int],
    profile: str = "clean_120", protocol: str = "prot_cold",
    severities: Iterable[float] = (0.0, 0.05, 0.10, 0.20, 0.30, 0.40),
    rna_tag: str = "rnafm", protein_tag: str = "proteinprottrans",
    device: Optional[torch.device] = None,
) -> pd.DataFrame:
    analysis_root = ensure_dir(analysis_root)
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    config = ub.PairModelTrainConfig()
    rows = []
    for dataset in datasets:
        bundle = ub.load_dataset(data_root, dataset, profile, output_root, rna_tag, protein_tag)
        for seed in seeds:
            splits = ub.prepare_or_load_splits(bundle, output_root, protocol, seed, n_splits=5)
            for fold in folds:
                split = splits[fold]
                for model_name in models:
                    if ub.is_zhmolgraph_model(model_name):
                        continue
                    model_dir = ub.model_directory(Path(output_root), profile, dataset, protocol, seed, fold, model_name)
                    checkpoint = model_dir / "model.pt"
                    metrics_path = model_dir / "metrics.json"
                    if not checkpoint.exists() or not metrics_path.exists():
                        continue
                    threshold = float(json.loads(metrics_path.read_text()).get("threshold", 0.5))
                    model = ub.create_pair_model(model_name, bundle.x_rna.shape[1], bundle.x_prot.shape[1], config).to(device)
                    model.load_state_dict(torch.load(checkpoint, map_location=device))
                    model.eval()
                    tr = split["train"]; te = split["test"]
                    base_adj, r2p, p2r = pm.build_train_graph_from_split(
                        bundle.r_idx, bundle.p_idx, bundle.y, tr, len(bundle.x_rna), len(bundle.x_prot)
                    )
                    base_adj = base_adj.to(device)
                    for fault_type in ["rna_gaussian", "protein_gaussian", "both_mask", "graph_random", "graph_hub_targeted"]:
                        for severity in severities:
                            xr, xp = bundle.x_rna, bundle.x_prot
                            adj = base_adj
                            if fault_type == "rna_gaussian":
                                xr = _corrupt_embeddings(xr, severity, "gaussian", seed + fold)
                            elif fault_type == "protein_gaussian":
                                xp = _corrupt_embeddings(xp, severity, "gaussian", seed + fold + 11)
                            elif fault_type == "both_mask":
                                xr = _corrupt_embeddings(xr, severity, "mask", seed + fold)
                                xp = _corrupt_embeddings(xp, severity, "mask", seed + fold + 11)
                            elif fault_type == "graph_random":
                                adj = _corrupt_graph(bundle, tr, severity, "random", seed + fold, device)
                            elif fault_type == "graph_hub_targeted":
                                adj = _corrupt_graph(bundle, tr, severity, "hub_targeted", seed + fold, device)
                            logits = ub.score_pair_model(
                                model,
                                torch.as_tensor(xr, dtype=torch.float32, device=device),
                                torch.as_tensor(xp, dtype=torch.float32, device=device),
                                adj, r2p, p2r,
                                bundle.r_idx[te], bundle.p_idx[te], device,
                            )
                            prob = ub.sigmoid_np(logits)
                            pred = (prob >= threshold).astype(int)
                            m = classification_metrics(bundle.y[te], prob, pred)
                            rel = reliability_metrics(bundle.y[te], prob, pred)
                            rows.append({
                                "dataset": dataset, "protocol": protocol, "model": model_name,
                                "seed": seed, "fold": fold, "fault_type": fault_type,
                                "severity": severity, **m, **rel,
                            })
                    del model
                    gc.collect()
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
    frame = pd.DataFrame(rows)
    frame.to_csv(analysis_root / "fault_injection_results.csv", index=False)
    return frame


# -----------------------------------------------------------------------------
# Model-level Integrated Gradients with validation reference and confidence matching
# -----------------------------------------------------------------------------

def integrated_gradients_pairmlp_cross(
    wrapper_model: nn.Module,
    x_rna_batch: torch.Tensor,
    x_prot_batch: torch.Tensor,
    baseline_rna: torch.Tensor,
    baseline_prot: torch.Tensor,
    steps: int = 24,
    batch_size: int = 24,
) -> tuple[np.ndarray, np.ndarray]:
    wrapper_model.eval()
    inner = wrapper_model.model
    inner.eval()
    device = next(wrapper_model.parameters()).device
    xr_all, xp_all = x_rna_batch.detach().to(device), x_prot_batch.detach().to(device)
    br0, bp0 = baseline_rna.detach().to(device), baseline_prot.detach().to(device)
    alphas = torch.linspace(1.0 / steps, 1.0, steps, device=device)
    out_r, out_p = [], []
    for start in range(0, len(xr_all), batch_size):
        xr, xp = xr_all[start:start+batch_size], xp_all[start:start+batch_size]
        br = br0.unsqueeze(0).expand_as(xr); bp = bp0.unsqueeze(0).expand_as(xp)
        dr, dp = xr - br, xp - bp
        gr = torch.zeros_like(xr); gp = torch.zeros_like(xp)
        for alpha in alphas:
            xri = (br + alpha * dr).detach().clone().requires_grad_(True)
            xpi = (bp + alpha * dp).detach().clone().requires_grad_(True)
            logits = inner(xri, xpi)
            grad_r, grad_p = torch.autograd.grad(logits.sum(), [xri, xpi])
            gr += grad_r.detach(); gp += grad_p.detach()
        out_r.append((dr * (gr / steps)).detach().cpu())
        out_p.append((dp * (gp / steps)).detach().cpu())
    return torch.cat(out_r).numpy(), torch.cat(out_p).numpy()


def confidence_matched_pairs(frame: pd.DataFrame, max_pairs: int = 60,
                             min_confidence: float = 0.80) -> pd.DataFrame:
    g = frame[frame["confidence"] >= min_confidence].copy()
    wrong = g[g["wrong"] == 1].sort_values("confidence", ascending=False)
    correct = g[g["wrong"] == 0].copy()
    if len(wrong) == 0 or len(correct) == 0:
        return pd.DataFrame()
    n = min(max_pairs, len(wrong), len(correct))
    wrong = wrong.head(n)
    available = correct.index.tolist()
    chosen = []
    for _, row in wrong.iterrows():
        if not available:
            break
        vals = correct.loc[available, "confidence"].to_numpy()
        j = int(np.argmin(np.abs(vals - row["confidence"])))
        chosen.append(available.pop(j))
    correct_matched = correct.loc[chosen]
    wrong = wrong.iloc[:len(correct_matched)]
    correct_matched = correct_matched.copy(); wrong = wrong.copy()
    correct_matched["xai_group"] = "correct"
    wrong["xai_group"] = "wrong"
    return pd.concat([correct_matched, wrong], ignore_index=True)


def _ig_summary(ig_rna: np.ndarray, ig_prot: np.ndarray) -> pd.DataFrame:
    ar, ap = np.abs(ig_rna), np.abs(ig_prot)
    rs, ps = ar.sum(1), ap.sum(1)
    total = rs + ps + EPS
    concat = np.concatenate([ar, ap], axis=1)
    return pd.DataFrame({
        "ig_rna_sum": rs,
        "ig_prot_sum": ps,
        "ig_total": total,
        "ig_prot_ratio": ps / total,
        "ig_modality_imbalance": np.abs(rs - ps) / total,
        "ig_concentration": concat.max(1) / total,
    })


def run_pairmlp_cross_xai(
    data_root: str | Path, output_root: str | Path, analysis_root: str | Path,
    datasets: Iterable[str], seeds: Iterable[int], folds: Iterable[int],
    profile: str = "clean_120", protocol: str = "prot_cold",
    max_pairs: int = 60, reference_n: int = 80, min_confidence: float = 0.80,
    steps: int = 24, rna_tag: str = "rnafm", protein_tag: str = "proteinprottrans",
    device: Optional[torch.device] = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    analysis_root = ensure_dir(analysis_root)
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    config = ub.PairModelTrainConfig()
    all_rows, metric_rows = [], []
    for dataset in datasets:
        bundle = ub.load_dataset(data_root, dataset, profile, output_root, rna_tag, protein_tag)
        splits_by_seed = {
            seed: ub.prepare_or_load_splits(bundle, output_root, protocol, seed, n_splits=5)
            for seed in seeds
        }
        x_rna_t = torch.as_tensor(bundle.x_rna, dtype=torch.float32, device=device)
        x_prot_t = torch.as_tensor(bundle.x_prot, dtype=torch.float32, device=device)
        for seed in seeds:
            for fold in folds:
                split = splits_by_seed[seed][fold]
                model_dir = ub.model_directory(Path(output_root), profile, dataset, protocol, seed, fold, "PairMLP-Cross")
                if not (model_dir / "model.pt").exists():
                    continue
                metrics = json.loads((model_dir / "metrics.json").read_text())
                threshold = float(metrics.get("threshold", 0.5))
                model = ub.create_pair_model("PairMLP-Cross", bundle.x_rna.shape[1], bundle.x_prot.shape[1], config).to(device)
                model.load_state_dict(torch.load(model_dir / "model.pt", map_location=device))
                model.eval()
                tr = split["train"]
                baseline_rna = x_rna_t[np.unique(bundle.r_idx[tr])].mean(0)
                baseline_prot = x_prot_t[np.unique(bundle.p_idx[tr])].mean(0)

                # Validation reference is independent of test outcomes.
                val = np.load(model_dir / "validation_predictions.npz")
                val_idx = val["row_idx"].astype(int)
                val_prob = ub.sigmoid_np(val["logits"])
                val_pred = (val_prob >= threshold).astype(int)
                val_frame = pd.DataFrame({
                    "row_idx": val_idx, "y": val["y"].astype(int), "probability": val_prob,
                    "prediction": val_pred,
                })
                val_frame["wrong"] = (val_frame["prediction"] != val_frame["y"]).astype(int)
                val_frame["confidence"] = confidence_from_probability(val_frame["probability"])
                ref = val_frame[val_frame["wrong"] == 0].sort_values("confidence", ascending=False).head(reference_n)
                if len(ref) < 5:
                    del model
                    continue
                ref_r = bundle.r_idx[ref["row_idx"].to_numpy(int)]
                ref_p = bundle.p_idx[ref["row_idx"].to_numpy(int)]
                ref_ig_r, ref_ig_p = integrated_gradients_pairmlp_cross(
                    model, x_rna_t[ref_r], x_prot_t[ref_p], baseline_rna, baseline_prot,
                    steps=steps,
                )
                ref_pattern = np.concatenate([np.abs(ref_ig_r), np.abs(ref_ig_p)], axis=1).mean(0, keepdims=True)

                test = pd.read_csv(model_dir / "test_predictions.csv")
                test["wrong"] = (test["prediction"].astype(int) != test["y"].astype(int)).astype(int)
                test["confidence"] = confidence_from_probability(test["probability"].to_numpy())
                selected = confidence_matched_pairs(test, max_pairs=max_pairs, min_confidence=min_confidence)
                if len(selected) < 10:
                    del model
                    continue
                sr = selected["r_idx"].to_numpy(int); sp = selected["p_idx"].to_numpy(int)
                ig_r, ig_p = integrated_gradients_pairmlp_cross(
                    model, x_rna_t[sr], x_prot_t[sp], baseline_rna, baseline_prot,
                    steps=steps,
                )
                summary = _ig_summary(ig_r, ig_p)
                attr = np.concatenate([np.abs(ig_r), np.abs(ig_p)], axis=1)
                similarity = cosine_similarity(attr, ref_pattern).reshape(-1)
                out = selected.reset_index(drop=True).copy()
                out = pd.concat([out, summary], axis=1)
                out["ig_similarity_to_validation_correct"] = similarity
                out["xai_failure_risk"] = 1.0 - similarity
                out["dataset"] = dataset; out["protocol"] = protocol
                out["seed"] = seed; out["fold"] = fold
                all_rows.append(out)

                y_fail = out["wrong"].to_numpy(int)
                metric_rows.append({
                    "dataset": dataset, "protocol": protocol, "seed": seed, "fold": fold,
                    "n": len(out),
                    "mean_conf_correct": float(out.loc[out.wrong == 0, "confidence"].mean()),
                    "mean_conf_wrong": float(out.loc[out.wrong == 1, "confidence"].mean()),
                    "xai_AUROC": safe_metric(roc_auc_score, y_fail, out["xai_failure_risk"]),
                    "xai_AUPRC": safe_metric(average_precision_score, y_fail, out["xai_failure_risk"]),
                    "confidence_AUROC": safe_metric(roc_auc_score, y_fail, 1.0 - out["confidence"]),
                    "mean_similarity_correct": float(out.loc[out.wrong == 0, "ig_similarity_to_validation_correct"].mean()),
                    "mean_similarity_wrong": float(out.loc[out.wrong == 1, "ig_similarity_to_validation_correct"].mean()),
                    "mean_imbalance_correct": float(out.loc[out.wrong == 0, "ig_modality_imbalance"].mean()),
                    "mean_imbalance_wrong": float(out.loc[out.wrong == 1, "ig_modality_imbalance"].mean()),
                })
                del model
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
    details = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
    metrics = pd.DataFrame(metric_rows)
    if len(details):
        details.to_parquet(analysis_root / "xai_confidence_matched_details.parquet", index=False)
    metrics.to_csv(analysis_root / "xai_confidence_matched_metrics.csv", index=False)
    return details, metrics


# -----------------------------------------------------------------------------
# Split-conformal prediction sets
# -----------------------------------------------------------------------------

def _higher_quantile(values: np.ndarray, q: float) -> float:
    values = np.sort(np.asarray(values, dtype=float))
    if len(values) == 0:
        return np.nan
    index = int(np.ceil(q * len(values))) - 1
    return float(values[np.clip(index, 0, len(values) - 1)])


def run_split_conformal(
    predictions: pd.DataFrame,
    output_root: str | Path,
    analysis_root: str | Path,
    alphas: Iterable[float] = (0.05, 0.10, 0.20),
    use_temperature: bool = True,
) -> pd.DataFrame:
    """Protocol-matched split-conformal prediction sets.

    The validation subset is used as the conformal calibration set. Test labels
    are used only for evaluation. Each test prediction may yield a singleton
    class, both classes (uncertain), or—rarely—an empty set.
    """
    output_root = Path(output_root)
    analysis_root = ensure_dir(analysis_root)
    rows = []
    group_cols = ["profile", "dataset", "protocol", "model", "seed", "fold"]
    for keys, g in predictions.groupby(group_cols):
        meta = dict(zip(group_cols, keys))
        model_dir = ub.model_directory(
            output_root, meta["profile"], meta["dataset"], meta["protocol"],
            int(meta["seed"]), int(meta["fold"]), meta["model"],
        )
        val_path = model_dir / "validation_predictions.npz"
        if not val_path.exists():
            continue
        val = np.load(val_path)
        temperature = float(g["temperature"].iloc[0]) if use_temperature else 1.0
        val_prob = ub.sigmoid_np(val["logits"].astype(float) / temperature)
        val_y = val["y"].astype(int)
        val_true_prob = np.where(val_y == 1, val_prob, 1.0 - val_prob)
        nonconformity = 1.0 - val_true_prob

        test_prob = (
            g["temp_probability"].to_numpy(float)
            if use_temperature else g["raw_probability"].to_numpy(float)
        )
        test_y = g["y"].to_numpy(int)
        for alpha in alphas:
            n = len(nonconformity)
            finite_q = min(1.0, np.ceil((n + 1) * (1.0 - alpha)) / max(n, 1))
            qhat = _higher_quantile(nonconformity, finite_q)
            min_class_prob = 1.0 - qhat
            include_0 = (1.0 - test_prob) >= min_class_prob
            include_1 = test_prob >= min_class_prob
            set_size = include_0.astype(int) + include_1.astype(int)
            covered = np.where(test_y == 1, include_1, include_0)
            singleton = set_size == 1
            singleton_pred = include_1.astype(int)
            rows.append({
                **meta,
                "alpha": float(alpha),
                "target_coverage": float(1.0 - alpha),
                "qhat": qhat,
                "empirical_coverage": float(np.mean(covered)),
                "coverage_gap": float(np.mean(covered) - (1.0 - alpha)),
                "mean_set_size": float(np.mean(set_size)),
                "singleton_rate": float(np.mean(singleton)),
                "ambiguous_rate": float(np.mean(set_size == 2)),
                "empty_rate": float(np.mean(set_size == 0)),
                "singleton_accuracy": (
                    float(np.mean(singleton_pred[singleton] == test_y[singleton]))
                    if singleton.any() else np.nan
                ),
                "n_test": int(len(g)),
                "temperature_scaled": bool(use_temperature),
            })
    frame = pd.DataFrame(rows)
    frame.to_csv(analysis_root / "conformal_prediction_summary.csv", index=False)
    return frame


# -----------------------------------------------------------------------------
# RNA-cold candidate-retrieval case study
# -----------------------------------------------------------------------------

def _retrieval_metrics(scores: np.ndarray, true_proteins: set[int],
                       ks: Iterable[int]) -> dict[str, float]:
    order = np.argsort(-np.asarray(scores))
    rank = np.empty(len(order), dtype=int)
    rank[order] = np.arange(len(order))
    true_ranks = sorted(int(rank[p]) for p in true_proteins)
    out = {"MRR": 1.0 / (true_ranks[0] + 1), "n_true": len(true_ranks)}
    for k in ks:
        hits = sum(r < k for r in true_ranks)
        out[f"Recall@{k}"] = hits / max(len(true_ranks), 1)
        out[f"Precision@{k}"] = hits / k
    return out


def run_rna_cold_retrieval(
    data_root: str | Path,
    output_root: str | Path,
    analysis_root: str | Path,
    datasets: Iterable[str],
    models: Iterable[str],
    seeds: Iterable[int],
    folds: Iterable[int],
    profile: str = "clean_120",
    ks: Iterable[int] = (5, 10, 20, 50),
    rna_tag: str = "rnafm",
    protein_tag: str = "proteinprottrans",
    device: Optional[torch.device] = None,
    score_batch_size: int = 65536,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Rank every candidate protein for each held-out RNA.

    Requires completed RNA-cold checkpoints. ZHMolGraph is skipped because its
    separate VecNN inference path is not shared with exact pair/GNN models.
    """
    output_root = Path(output_root)
    analysis_root = ensure_dir(analysis_root)
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    config = ub.PairModelTrainConfig()
    rows = []
    for dataset in datasets:
        bundle = ub.load_dataset(data_root, dataset, profile, output_root, rna_tag, protein_tag)
        x_rna = torch.as_tensor(bundle.x_rna, dtype=torch.float32, device=device)
        x_prot = torch.as_tensor(bundle.x_prot, dtype=torch.float32, device=device)
        for seed in seeds:
            splits = ub.prepare_or_load_splits(
                bundle, output_root, "rna_cold", seed, n_splits=5
            )
            for fold in folds:
                split = splits[fold]
                tr, te = split["train"], split["test"]
                true_map: dict[int, set[int]] = {}
                for edge in te[bundle.y[te] == 1]:
                    true_map.setdefault(int(bundle.r_idx[edge]), set()).add(int(bundle.p_idx[edge]))
                if not true_map:
                    continue
                adjacency, r2p, p2r = pm.build_train_graph_from_split(
                    bundle.r_idx, bundle.p_idx, bundle.y, tr,
                    len(bundle.x_rna), len(bundle.x_prot),
                )
                adjacency = adjacency.to(device)
                query_rnas = np.array(sorted(true_map), dtype=int)
                n_prot = len(bundle.x_prot)
                all_r = np.repeat(query_rnas, n_prot)
                all_p = np.tile(np.arange(n_prot, dtype=int), len(query_rnas))
                for model_name in models:
                    if ub.is_zhmolgraph_model(model_name):
                        continue
                    model_dir = ub.model_directory(
                        output_root, profile, dataset, "rna_cold", seed, fold, model_name
                    )
                    checkpoint = model_dir / "model.pt"
                    if not checkpoint.exists():
                        continue
                    model = ub.create_pair_model(
                        model_name, bundle.x_rna.shape[1], bundle.x_prot.shape[1], config
                    ).to(device)
                    model.load_state_dict(torch.load(checkpoint, map_location=device))
                    model.eval()
                    with torch.inference_mode():
                        h_r, h_p, core = model.encode_nodes(x_rna, x_prot, adjacency, r2p, p2r)
                        score_chunks = []
                        for start in range(0, len(all_r), score_batch_size):
                            rr = torch.as_tensor(
                                all_r[start:start+score_batch_size], dtype=torch.long, device=device
                            )
                            pp = torch.as_tensor(
                                all_p[start:start+score_batch_size], dtype=torch.long, device=device
                            )
                            score_chunks.append(
                                model.score_edges(h_r, h_p, core, rr, pp).detach().cpu().numpy()
                            )
                    score_matrix = np.concatenate(score_chunks).reshape(len(query_rnas), n_prot)
                    for idx, rna_id in enumerate(query_rnas):
                        metric = _retrieval_metrics(score_matrix[idx], true_map[int(rna_id)], ks)
                        rows.append({
                            "dataset": dataset, "model": model_name, "seed": seed,
                            "fold": fold, "r_idx": int(rna_id), **metric,
                        })
                    del model
                    gc.collect()
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
    per_rna = pd.DataFrame(rows)
    if len(per_rna):
        per_rna.to_csv(analysis_root / "rna_cold_retrieval_per_rna.csv", index=False)
        metric_cols = ["MRR"] + [f"Recall@{k}" for k in ks] + [f"Precision@{k}" for k in ks]
        per_seed = (
            per_rna.groupby(["dataset","model","seed"])[metric_cols]
            .mean().reset_index()
        )
        summary = (
            per_seed.groupby(["dataset","model"])[metric_cols]
            .agg(["mean","std","count"]).reset_index()
        )
        summary.columns = [
            "_".join(str(x) for x in col if str(x)) if isinstance(col, tuple) else col
            for col in summary.columns
        ]
    else:
        summary = pd.DataFrame()
    summary.to_csv(analysis_root / "rna_cold_retrieval_summary.csv", index=False)
    return per_rna, summary



# -----------------------------------------------------------------------------
# Statistical comparisons and summary outputs
# -----------------------------------------------------------------------------

def holm_adjust(p_values: Iterable[float]) -> np.ndarray:
    p = np.asarray(list(p_values), dtype=float)
    order = np.argsort(p)
    adjusted = np.empty_like(p)
    running = 0.0
    m = len(p)
    for rank, idx in enumerate(order):
        value = (m - rank) * p[idx]
        running = max(running, value)
        adjusted[idx] = min(1.0, running)
    return adjusted


def paired_model_tests(fold_summary: pd.DataFrame, analysis_root: str | Path,
                       metrics=("RAW_AUROC", "RAW_MCC", "RAW_ECE", "RAW_BRIER")) -> pd.DataFrame:
    analysis_root = ensure_dir(analysis_root)
    rows = []
    for (dataset, protocol), g in fold_summary.groupby(["dataset", "protocol"]):
        models = sorted(g["model"].unique())
        for metric in metrics:
            for i, a in enumerate(models):
                for b in models[i+1:]:
                    aa = g[g.model == a][["seed", "fold", metric]].rename(columns={metric: "a"})
                    bb = g[g.model == b][["seed", "fold", metric]].rename(columns={metric: "b"})
                    merged = aa.merge(bb, on=["seed", "fold"]).dropna()
                    if len(merged) < 3:
                        continue
                    try:
                        stat, p = wilcoxon(merged["a"], merged["b"], zero_method="wilcox")
                    except Exception:
                        stat, p = np.nan, np.nan
                    rows.append({
                        "dataset": dataset, "protocol": protocol, "metric": metric,
                        "model_a": a, "model_b": b, "n": len(merged),
                        "mean_a": merged["a"].mean(), "mean_b": merged["b"].mean(),
                        "mean_difference_a_minus_b": (merged["a"] - merged["b"]).mean(),
                        "wilcoxon_stat": stat, "p_value": p,
                    })
    frame = pd.DataFrame(rows)
    if len(frame):
        frame["p_holm"] = np.nan
        for _, idx in frame.groupby(["dataset", "protocol", "metric"]).groups.items():
            frame.loc[idx, "p_holm"] = holm_adjust(frame.loc[idx, "p_value"].fillna(1.0))
    frame.to_csv(analysis_root / "paired_model_tests.csv", index=False)
    return frame


def _format_mean_std(mean: float, std: float, digits: int = 3) -> str:
    if pd.isna(mean):
        return "NA"
    if pd.isna(std):
        return f"{mean:.{digits}f}"
    return f"{mean:.{digits}f} ± {std:.{digits}f}"


def build_publication_dashboard(fold_summary: pd.DataFrame, selective: pd.DataFrame,
                                analysis_root: str | Path, coverage: float = 0.50) -> pd.DataFrame:
    analysis_root = ensure_dir(analysis_root)
    core_metrics = [
        "RAW_AUROC", "RAW_AUPRC", "RAW_MCC", "RAW_ERROR",
        "RAW_BRIER", "RAW_ECE", "RAW_HC90_COVERAGE", "RAW_HC90_ERROR",
        "TEMP_BRIER", "TEMP_ECE",
    ]
    agg = fold_summary.groupby(["dataset", "protocol", "model"])[core_metrics].agg(["mean", "std"])
    rows = []
    for idx, values in agg.iterrows():
        dataset, protocol, model = idx
        row = {"Dataset": dataset, "Protocol": protocol, "Model": model}
        for metric in core_metrics:
            row[metric] = _format_mean_std(values[(metric, "mean")], values[(metric, "std")])
        sub = selective[
            (selective.dataset == dataset) & (selective.protocol == protocol)
            & (selective.model == model) & (selective.risk_method == "raw_confidence")
            & np.isclose(selective.coverage, coverage)
        ]
        full = selective[
            (selective.dataset == dataset) & (selective.protocol == protocol)
            & (selective.model == model) & (selective.risk_method == "raw_confidence")
            & np.isclose(selective.coverage, 1.0)
        ]
        if len(sub) and len(full):
            row[f"MCC@{int(coverage*100)}%"] = _format_mean_std(sub.MCC.mean(), sub.MCC.std())
            row[f"Error@{int(coverage*100)}%"] = _format_mean_std(sub.ERROR.mean(), sub.ERROR.std())
            row["Abstention_MCC_gain"] = f"{sub.MCC.mean() - full.MCC.mean():+.3f}"
            row["Abstention_error_change"] = f"{sub.ERROR.mean() - full.ERROR.mean():+.3f}"
        rows.append(row)
    dashboard = pd.DataFrame(rows)
    dashboard.to_csv(analysis_root / "TABLE_overall_dependability_dashboard.csv", index=False)
    dashboard.to_excel(analysis_root / "TABLE_overall_dependability_dashboard.xlsx", index=False)
    return dashboard


def print_key_findings(fold_summary: pd.DataFrame, selective: pd.DataFrame) -> None:
    print("\nAUTOMATIC RESULT INTERPRETATION (verify before using in a paper)\n" + "=" * 72)
    for (dataset, protocol), g in fold_summary.groupby(["dataset", "protocol"]):
        mean = g.groupby("model").mean(numeric_only=True)
        best_auroc = mean["RAW_AUROC"].idxmax()
        best_mcc = mean["RAW_MCC"].idxmax()
        lowest_ece = mean["RAW_ECE"].idxmin()
        lowest_hce = mean["RAW_HC90_ERROR"].dropna().idxmin() if mean["RAW_HC90_ERROR"].notna().any() else "none"
        print(f"\n{dataset} / {protocol}")
        print(f"  • Highest AUROC: {best_auroc} ({mean.loc[best_auroc, 'RAW_AUROC']:.3f})")
        print(f"  • Highest MCC:   {best_mcc} ({mean.loc[best_mcc, 'RAW_MCC']:.3f})")
        print(f"  • Lowest ECE:    {lowest_ece} ({mean.loc[lowest_ece, 'RAW_ECE']:.3f})")
        if lowest_hce != "none":
            print(f"  • Lowest ≥0.90-confidence error: {lowest_hce} ({mean.loc[lowest_hce, 'RAW_HC90_ERROR']:.3f})")
        for model in mean.index:
            s50 = selective[
                (selective.dataset == dataset) & (selective.protocol == protocol)
                & (selective.model == model) & (selective.risk_method == "raw_confidence")
                & np.isclose(selective.coverage, 0.5)
            ]
            s100 = selective[
                (selective.dataset == dataset) & (selective.protocol == protocol)
                & (selective.model == model) & (selective.risk_method == "raw_confidence")
                & np.isclose(selective.coverage, 1.0)
            ]
            if len(s50) and len(s100):
                print(f"  • {model}: at 50% coverage, MCC change={s50.MCC.mean()-s100.MCC.mean():+.3f}, "
                      f"error change={s50.ERROR.mean()-s100.ERROR.mean():+.3f}")


# -----------------------------------------------------------------------------
# Plotting (matplotlib only; saved as ready PNG/PDF)
# -----------------------------------------------------------------------------

def _model_markers(models: Iterable[str]) -> dict[str, str]:
    markers = ["o", "s", "^", "D", "P", "X", "v", "<", ">"]
    return {m: markers[i % len(markers)] for i, m in enumerate(sorted(models))}


def plot_calibration_grid(predictions: pd.DataFrame, analysis_root: str | Path,
                          probability_col: str = "raw_probability", n_bins: int = 10) -> None:
    import matplotlib.pyplot as plt
    analysis_root = ensure_dir(analysis_root)
    for (dataset, protocol), dg in predictions.groupby(["dataset", "protocol"]):
        models = sorted(dg.model.unique())
        fig, axes = plt.subplots(1, len(models), figsize=(5 * len(models), 4), squeeze=False)
        for ax, model in zip(axes[0], models):
            g = dg[dg.model == model]
            bins = calibration_bins(g.y.to_numpy(), g[probability_col].to_numpy(), n_bins).dropna()
            ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1, color="0.4")
            ax.plot(bins.mean_probability, bins.observed_positive_rate, marker="o", linewidth=1.8)
            ece = expected_calibration_error_positive(g.y.to_numpy(), g[probability_col].to_numpy(), n_bins)
            ax.set_title(f"{model}\nECE={ece:.3f}")
            ax.set_xlabel("Mean predicted probability"); ax.set_ylabel("Observed positive rate")
            ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.grid(alpha=0.25)
        fig.suptitle(f"Reliability diagrams — {dataset} / {protocol}")
        fig.tight_layout()
        stem = analysis_root / f"FIG_reliability_{probability_col}_{dataset}_{protocol}"
        fig.savefig(stem.with_suffix(".png"), dpi=250, bbox_inches="tight")
        fig.savefig(stem.with_suffix(".pdf"), bbox_inches="tight")
        plt.show()


def plot_metric_heatmap(fold_summary: pd.DataFrame, analysis_root: str | Path,
                        metric: str = "RAW_AUROC") -> None:
    import matplotlib.pyplot as plt
    analysis_root = ensure_dir(analysis_root)
    mean = fold_summary.groupby(["dataset", "protocol", "model"])[metric].mean().reset_index()
    for protocol, g in mean.groupby("protocol"):
        table = g.pivot(index="model", columns="dataset", values=metric)
        fig, ax = plt.subplots(figsize=(1.8 * len(table.columns) + 3, 0.7 * len(table.index) + 2))
        im = ax.imshow(table.to_numpy(), aspect="auto", cmap="Greys", vmin=np.nanmin(table), vmax=np.nanmax(table))
        ax.set_xticks(np.arange(len(table.columns)), table.columns)
        ax.set_yticks(np.arange(len(table.index)), table.index)
        for i in range(len(table.index)):
            for j in range(len(table.columns)):
                value = table.iloc[i, j]
                ax.text(j, i, "NA" if pd.isna(value) else f"{value:.3f}", ha="center", va="center",
                        color="white" if np.isfinite(value) and value > np.nanmean(table.to_numpy()) else "black")
        ax.set_title(f"{metric} — {protocol}")
        fig.colorbar(im, ax=ax, fraction=0.04, pad=0.04)
        fig.tight_layout()
        fig.savefig(analysis_root / f"FIG_heatmap_{metric}_{protocol}.png", dpi=250, bbox_inches="tight")
        plt.show()


def plot_selective_curves(selective: pd.DataFrame, analysis_root: str | Path,
                          metric: str = "MCC", risk_method: str = "raw_confidence") -> None:
    import matplotlib.pyplot as plt
    analysis_root = ensure_dir(analysis_root)
    for (dataset, protocol), dg in selective[selective.risk_method == risk_method].groupby(["dataset", "protocol"]):
        fig, ax = plt.subplots(figsize=(7, 4.5))
        markers = _model_markers(dg.model.unique())
        for model, g in dg.groupby("model"):
            agg = g.groupby("coverage")[metric].agg(["mean", "std"]).reset_index().sort_values("coverage")
            ax.errorbar(agg.coverage, agg["mean"], yerr=agg["std"].fillna(0), marker=markers[model],
                        capsize=3, linewidth=1.7, label=model)
        ax.set_xlabel("Coverage (accepted fraction)"); ax.set_ylabel(metric)
        ax.set_title(f"Selective prediction — {dataset} / {protocol} / {risk_method}")
        ax.grid(alpha=0.25); ax.legend(); fig.tight_layout()
        fig.savefig(analysis_root / f"FIG_selective_{metric}_{risk_method}_{dataset}_{protocol}.png", dpi=250)
        plt.show()


def plot_failure_detector_summary(results: pd.DataFrame, analysis_root: str | Path) -> None:
    import matplotlib.pyplot as plt
    analysis_root = ensure_dir(analysis_root)
    for (dataset, protocol), g in results.groupby(["dataset", "protocol"]):
        best = g.sort_values("failure_AUROC").groupby(["model", "feature_set"], as_index=False).tail(1)
        models = sorted(best.model.unique()); feature_sets = sorted(best.feature_set.unique())
        x = np.arange(len(models)); width = 0.8 / max(len(feature_sets), 1)
        fig, ax = plt.subplots(figsize=(8, 4.5))
        for i, fs in enumerate(feature_sets):
            vals = [best[(best.model == m) & (best.feature_set == fs)].failure_AUROC.max() for m in models]
            ax.bar(x + (i - (len(feature_sets)-1)/2)*width, vals, width, label=fs, edgecolor="black", linewidth=0.5)
        ax.axhline(0.5, linestyle="--", color="0.4", linewidth=1)
        ax.set_xticks(x, models, rotation=15); ax.set_ylim(0.45, 1.0)
        ax.set_ylabel("Failure-detection AUROC")
        ax.set_title(f"Can we detect wrong predictions? — {dataset} / {protocol}")
        ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.25); fig.tight_layout()
        fig.savefig(analysis_root / f"FIG_failure_detector_{dataset}_{protocol}.png", dpi=250)
        plt.show()


def plot_permutation_importance(importance: pd.DataFrame, analysis_root: str | Path) -> None:
    import matplotlib.pyplot as plt
    analysis_root = ensure_dir(analysis_root)
    for (dataset, protocol, model), g in importance.groupby(["dataset", "protocol", "model"]):
        agg = g.groupby("feature").importance.agg(["mean", "std"]).sort_values("mean")
        fig, ax = plt.subplots(figsize=(7, 4.5))
        ax.barh(agg.index, agg["mean"], xerr=agg["std"].fillna(0), capsize=3)
        ax.set_xlabel("Held-out permutation importance (AUROC drop)")
        ax.set_title(f"Failure-risk drivers — {dataset} / {model}")
        ax.grid(axis="x", alpha=0.25); fig.tight_layout()
        fig.savefig(analysis_root / f"FIG_failure_importance_{dataset}_{protocol}_{model}.png", dpi=250)
        plt.show()


def plot_fault_curves(faults: pd.DataFrame, analysis_root: str | Path, metric="MCC") -> None:
    import matplotlib.pyplot as plt
    analysis_root = ensure_dir(analysis_root)
    for (dataset, fault_type), g in faults.groupby(["dataset", "fault_type"]):
        fig, ax = plt.subplots(figsize=(7, 4.5))
        markers = _model_markers(g.model.unique())
        for model, mg in g.groupby("model"):
            agg = mg.groupby("severity")[metric].agg(["mean", "std"]).reset_index()
            ax.errorbar(agg.severity, agg["mean"], yerr=agg["std"].fillna(0), marker=markers[model],
                        capsize=3, label=model)
        ax.set_xlabel("Fault severity"); ax.set_ylabel(metric)
        ax.set_title(f"Degradation curve — {dataset} / {fault_type}")
        ax.grid(alpha=0.25); ax.legend(); fig.tight_layout()
        fig.savefig(analysis_root / f"FIG_fault_{metric}_{dataset}_{fault_type}.png", dpi=250)
        plt.show()


def plot_xai_results(details: pd.DataFrame, analysis_root: str | Path) -> None:
    import matplotlib.pyplot as plt
    analysis_root = ensure_dir(analysis_root)
    if details.empty:
        return
    for dataset, g in details.groupby("dataset"):
        fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
        groups = [g[g.wrong == 0], g[g.wrong == 1]]
        labels = ["correct", "wrong"]
        axes[0].boxplot([x.ig_similarity_to_validation_correct for x in groups], labels=labels, showmeans=True)
        axes[0].set_title("Similarity to validation-correct attribution pattern")
        axes[0].set_ylabel("Cosine similarity"); axes[0].grid(axis="y", alpha=0.25)
        axes[1].boxplot([x.ig_modality_imbalance for x in groups], labels=labels, showmeans=True)
        axes[1].set_title("RNA/protein attribution imbalance")
        axes[1].set_ylabel("Normalized imbalance"); axes[1].grid(axis="y", alpha=0.25)
        fig.suptitle(f"Confidence-matched model-level XAI — {dataset}")
        fig.tight_layout()
        fig.savefig(analysis_root / f"FIG_xai_confidence_matched_{dataset}.png", dpi=250)
        plt.show()


In [ ]:
%%writefile journal_extensions.py
from __future__ import annotations
import json, math, shutil, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

EPS=1e-12

def _clf(name, seed=17):
    if name=="rf":
        return RandomForestClassifier(n_estimators=500, min_samples_leaf=5, class_weight="balanced", random_state=seed, n_jobs=-1)
    if name=="logreg":
        return make_pipeline(StandardScaler(), LogisticRegression(max_iter=4000, class_weight="balanced", random_state=seed))
    raise ValueError(name)

def leave_one_dataset_out_failure_detection(predictions, features, classifier="rf", output_path=None):
    """Train failure detector on two datasets and test on the third.

    No target-dataset failure labels are used for fitting. Evaluation is repeated
    separately for each base RPI model.
    """
    rows=[]; scored=[]
    for model,g_model in predictions.groupby("model"):
        datasets=sorted(g_model["dataset"].unique())
        for target in datasets:
            tr=g_model[g_model.dataset!=target].copy(); te=g_model[g_model.dataset==target].copy()
            if tr.empty or te.empty or tr.wrong.nunique()<2 or te.wrong.nunique()<2: continue
            Xtr=tr[features].replace([np.inf,-np.inf],np.nan).fillna(0).to_numpy(float)
            Xte=te[features].replace([np.inf,-np.inf],np.nan).fillna(0).to_numpy(float)
            ytr=tr.wrong.to_numpy(int); yte=te.wrong.to_numpy(int)
            clf=_clf(classifier,seed=17)
            clf.fit(Xtr,ytr); risk=clf.predict_proba(Xte)[:,1]
            order=np.argsort(-risk); k=max(1,int(.10*len(yte)))
            rows.append({
                "model":model,"target_dataset":target,"train_datasets":" + ".join([d for d in datasets if d!=target]),
                "classifier":classifier,"n_train":len(tr),"n_test":len(te),"baseline_failure_rate":float(yte.mean()),
                "failure_AUROC":float(roc_auc_score(yte,risk)),"failure_AUPRC":float(average_precision_score(yte,risk)),
                "top10_failure_rate":float(yte[order[:k]].mean()),
                "top10_enrichment":float(yte[order[:k]].mean()/max(float(yte.mean()),EPS)),
            })
            z=te[["dataset","model","seed","fold","row_id","r_idx","p_idx","y","wrong"]].copy(); z["failure_risk_lodo"]=risk
            scored.append(z)
    result=pd.DataFrame(rows); scored_df=pd.concat(scored,ignore_index=True) if scored else pd.DataFrame()
    if output_path:
        output_path=Path(output_path); output_path.mkdir(parents=True,exist_ok=True)
        result.to_csv(output_path/"TABLE_failure_detector_leave_one_dataset_out.csv",index=False)
        if len(scored_df): scored_df.to_csv(output_path/"failure_detector_lodo_scores.csv.gz",index=False,compression="gzip")
    return result, scored_df

def bootstrap_fold_ci(fold_summary, metrics=("RAW_AUROC","RAW_AUPRC","RAW_MCC","RAW_ECE"), n_boot=5000, seed=9274):
    """Bootstrap 95% CI over seed/fold experimental units, not individual edges."""
    rng=np.random.default_rng(seed); rows=[]
    for (dataset,protocol,model),g in fold_summary.groupby(["dataset","protocol","model"]):
        for metric in metrics:
            vals=g[metric].dropna().to_numpy(float)
            if len(vals)<2: continue
            boots=np.array([rng.choice(vals,size=len(vals),replace=True).mean() for _ in range(n_boot)])
            rows.append({"dataset":dataset,"protocol":protocol,"model":model,"metric":metric,"n_units":len(vals),
                         "mean":float(vals.mean()),"sd":float(vals.std(ddof=1)),"ci95_low":float(np.quantile(boots,.025)),"ci95_high":float(np.quantile(boots,.975))})
    return pd.DataFrame(rows)

def save_figure_source(df, figure_id, figure_root, caption, interpretation, pdf_path=None, png_path=None):
    figure_root=Path(figure_root); (figure_root/"source_data").mkdir(parents=True,exist_ok=True)
    src=figure_root/"source_data"/f"{figure_id}_source.csv"
    df.to_csv(src,index=False)
    manifest_path=figure_root/"figure_manifest.csv"
    row=pd.DataFrame([{"figure_id":figure_id,"source_csv":str(src.name),"pdf":str(pdf_path or ''),"png":str(png_path or ''),"caption":caption,"interpretation":interpretation}])
    if manifest_path.exists():
        old=pd.read_csv(manifest_path); old=old[old.figure_id!=figure_id]; row=pd.concat([old,row],ignore_index=True)
    row.to_csv(manifest_path,index=False)

def compact_export(output_root, analysis_root, destination_zip, include_checkpoints=False):
    """Export manuscript-relevant artifacts without zipping the whole Kaggle work tree."""
    output_root=Path(output_root); analysis_root=Path(analysis_root); destination_zip=Path(destination_zip)
    allowed={".json",".csv",".gz",".pdf",".png",".xlsx",".npy"}
    with zipfile.ZipFile(destination_zip,"w",zipfile.ZIP_DEFLATED) as z:
        # Analysis tables/figures/source data.
        for p in analysis_root.rglob("*"):
            if p.is_file() and (p.suffix in allowed or p.name.endswith('.csv.gz')):
                z.write(p,arcname=str(p.relative_to(output_root)))
        # Reproducible split manifests live inside dataset/protocol directories.
        for pattern in ["split_meta.json","split_rows.csv.gz","inner_train_indices.npy","validation_indices.npy","test_indices.npy"]:
            for p in output_root.rglob(pattern):
                z.write(p,arcname=str(p.relative_to(output_root)))
        for p in output_root.rglob("metrics.json"):
            z.write(p,arcname=str(p.relative_to(output_root)))
        for p in output_root.rglob("test_predictions.csv.gz"):
            z.write(p,arcname=str(p.relative_to(output_root)))
        for p in output_root.rglob("validation_predictions.csv.gz"):
            z.write(p,arcname=str(p.relative_to(output_root)))
        if include_checkpoints:
            for p in output_root.rglob("*.pt"):
                z.write(p,arcname=str(p.relative_to(output_root)))
    return destination_zip


In [ ]:
import exact_pair_models as pm
import exact_zhmolgraph
import journal_benchmark as ub
import journal_dependability as da
import journal_extensions as jx
for module in [pm,exact_zhmolgraph,ub,da,jx]: importlib.reload(module)
print("Modules loaded.")
print("Central models:",CENTRAL_MODELS)


## 4. Dataset audit

Check dataset composition and fail early if required inputs are missing or invalid.


In [ ]:
audit=[]
if RUN["audit"]:
    for ds in DATASETS:
        b=ub.load_dataset(DATA_ROOT,ds,PROFILE,OUTPUT_ROOT,RNA_TAG,PROTEIN_TAG)
        row=dict(b.meta)
        row["positive_rate"]=float(b.y.mean())
        row["unique_rna_in_rows"]=int(len(np.unique(b.r_idx)))
        row["unique_protein_in_rows"]=int(len(np.unique(b.p_idx)))
        audit.append(row)
        print(ds,"rows",len(b.y),"pos",int(b.y.sum()),"neg",int((1-b.y).sum()))
    audit_df=pd.DataFrame(audit)
    audit_df.to_csv(ANALYSIS_ROOT/"TABLE_dataset_audit.csv",index=False)
    display(audit_df)


## 5. Training configuration and split preflight

All dataset × seed × fold splits are generated and checked before model training.


In [ ]:
PAIR_CONFIG=ub.PairModelTrainConfig(
    max_epochs=50,early_stop_patience=5,learning_rate=1e-3,batch_size=4096,
    hidden_dim=256,pair_dropout=0.2,graph_dropout=0.1,n_heads=4,
    graphsage_layers=2,graphsage_sample_k=25,
)
ZH_CONFIG=ub.ZHMolGraphTrainConfig(
    sage_layers=2,sage_hidden=100,sage_neighbours=10,sage_batch_size=20,
    sage_lr=0.65,sage_grad_clip=5.0,sage_epochs=10,sage_num_negative=10,
    sage_loss="normal",vec_batch_size=128,vec_lr=1e-3,vec_weight_decay=0.0,
    vec_patience=15,vec_min_epochs=20,use_amp=torch.cuda.is_available(),
)

PLAN=ub.ExperimentPlan(
    name="DEPENDABLE_RPI_JOURNAL_V1",profile=PROFILE,datasets=tuple(RUN_DATASETS),
    protocols=PROTOCOLS,models=tuple(RUN_MODELS),seeds=tuple(RUN_SEEDS),
    n_splits=N_SPLITS,val_fraction=.10,folds=RUN_FOLDS,run_thenovel=False,resume=True,overwrite=False,
)
print(json.dumps(PLAN.__dict__,indent=2,default=list))

preflight_rows=[]
if RUN["preflight"]:
    for ds in DATASETS:  # Preflight the full benchmark, not only the current shard.
        b=ub.load_dataset(DATA_ROOT,ds,PROFILE,OUTPUT_ROOT,RNA_TAG,PROTEIN_TAG)
        for protocol in PROTOCOLS:
            for seed in SEEDS:
                splits=ub.prepare_or_load_splits(b,OUTPUT_ROOT,protocol,seed,N_SPLITS,.10,overwrite=False)
                for fold,s in enumerate(splits):
                    tr,va,te=s["train"],s["val"],s["test"]
                    groups=b.p_idx if protocol=="prot_cold" else b.r_idx
                    assert len(np.intersect1d(np.unique(groups[tr]),np.unique(groups[va])))==0
                    assert len(np.intersect1d(np.unique(groups[tr]),np.unique(groups[te])))==0
                    assert len(np.intersect1d(np.unique(groups[va]),np.unique(groups[te])))==0
                    assert len(np.unique(b.y[tr]))==2 and len(np.unique(b.y[va]))==2 and len(np.unique(b.y[te]))==2
                    preflight_rows.append({"dataset":ds,"protocol":protocol,"seed":seed,"fold":fold,
                                           "n_train":len(tr),"n_val":len(va),"n_test":len(te),
                                           "train_pos_rate":b.y[tr].mean(),"val_pos_rate":b.y[va].mean(),"test_pos_rate":b.y[te].mean()})
    preflight_df=pd.DataFrame(preflight_rows)
    preflight_df.to_csv(ANALYSIS_ROOT/"TABLE_split_preflight.csv",index=False)
    print("PASSED:",len(preflight_df),"outer folds ready before training")
    display(preflight_df)


## 6. Train or resume the four-model benchmark

Use `RUN_DATASETS`, `RUN_MODELS`, and `RUN_SEEDS` to split long runs across sessions.
Completed folds can be reused from the configured output directory.


In [ ]:
if RUN["training"]:
    print("Free disk before shard:",round(shutil.disk_usage(OUTPUT_ROOT).free/1024**3,2),"GB")
    run_manifest=ub.run_plan(PLAN,DATA_ROOT,OUTPUT_ROOT,pair_config=PAIR_CONFIG,zh_config=ZH_CONFIG,device=DEVICE,rna_tag=RNA_TAG,protein_tag=PROTEIN_TAG)
    display(run_manifest)


## 7. Collect predictions and reliability features


In [ ]:
if RUN["collect_predictions"]:
    predictions, temperatures = da.collect_prediction_records(
        DATA_ROOT,
        OUTPUT_ROOT,
        ANALYSIS_ROOT,
        RNA_TAG,
        PROTEIN_TAG,
        protocols=PROTOCOLS,
        datasets=DATASETS,
        models=CENTRAL_MODELS,
    )

    fold_summary, overall_summary = da.summarize_saved_predictions(
        predictions,
        ANALYSIS_ROOT,
    )

    print("Prediction rows:", len(predictions))

    display(
        fold_summary
        .groupby(["dataset", "model"])[
            [
                "RAW_AUROC",
                "RAW_AUPRC",
                "RAW_MCC",
                "RAW_ECE",
                "RAW_BRIER",
            ]
        ]
        .agg(["mean", "std"])
        .round(4)
    )

## 8. Calibration, high-confidence error, and selective prediction


In [ ]:
# Selective prediction must run before the dashboard because the dashboard
# includes MCC/error at the chosen retained coverage.
if RUN["selective_prediction"]:
    selective,selective_aurc=da.run_selective_analysis(
        predictions,ANALYSIS_ROOT,coverages=(1.0,.95,.90,.80,.70,.60,.50)
    )
    display(
        selective[
            (selective.risk_method=="raw_confidence")
            & (selective.coverage.isin([1.0,.8,.5]))
        ].head(60)
    )
    display(selective_aurc.head(30))

if RUN["calibration"]:
    # The summary builder needs the selective-curve dataframe.
    # If selective analysis was disabled in this session, reuse the saved table.
    if "selective" not in globals():
        selective_path=ANALYSIS_ROOT/"selective_risk_coverage.csv"
        if not selective_path.exists():
            raise RuntimeError(
                "Dashboard needs selective results. Enable RUN['selective_prediction'] "
                "or provide analysis/selective_risk_coverage.csv."
            )
        selective=pd.read_csv(selective_path)
    dashboard=da.build_publication_dashboard(fold_summary,selective,ANALYSIS_ROOT)
    display(dashboard)


## 9. Structural diagnostics

Full protein degree is retrospective only. RNA training support is available from
the training graph and may be used as a deployable reliability feature.


In [ ]:
if RUN["structural_diagnosis"]:
    hub_rows=[]; orphan_rows=[]
    for (ds,model),g in predictions[predictions.protocol=="prot_cold"].groupby(["dataset","model"]):
        # Retrospective held-out protein degree buckets.
        d=g.copy(); d["protein_degree_bucket"]=pd.cut(d["full_protein_degree"],[-1,1,9,99,np.inf],labels=["1","2-9","10-99","100+"])
        for bucket,h in d.groupby("protein_degree_bucket",observed=True):
            if h.y.nunique()<2: continue
            hub_rows.append({"dataset":ds,"model":model,"bucket":str(bucket),"n":len(h),"AUROC":da.safe_metric(da.roc_auc_score,h.y,h.raw_probability)})
        d["rna_support_bucket"]=pd.cut(d["train_rna_degree"],[-1,0,2,np.inf],labels=["orphan_0","low_1_2","warm_3plus"])
        for bucket,h in d.groupby("rna_support_bucket",observed=True):
            if h.y.nunique()<2: continue
            orphan_rows.append({"dataset":ds,"model":model,"bucket":str(bucket),"n":len(h),"AUROC":da.safe_metric(da.roc_auc_score,h.y,h.raw_probability),"error":float(h.wrong.mean())})
    hub_table=pd.DataFrame(hub_rows); orphan_table=pd.DataFrame(orphan_rows)
    hub_table.to_csv(ANALYSIS_ROOT/"TABLE_hub_degree_diagnostic.csv",index=False)
    orphan_table.to_csv(ANALYSIS_ROOT/"TABLE_cascade_orphan_summary.csv",index=False)
    display(hub_table.groupby(["dataset","model","bucket"])["AUROC"].mean().unstack())
    display(orphan_table.groupby(["dataset","model","bucket"])["AUROC"].mean().unstack())


## 10. Protein-grouped failure detector

The secondary target is whether the primary RPI prediction is incorrect. Secondary
cross-validation is grouped by held-out protein to match the deployment setting.


In [ ]:
LOCAL_DEPLOYABLE_FEATURES=[
    "raw_confidence","raw_margin","raw_entropy",
    "log1p_train_rna_degree","rna_orphan",
    "rna_centroid_cosine","protein_centroid_cosine",
    "rna_diagonal_ood","protein_diagonal_ood",
]
FAILURE_FEATURE_SETS={
    "confidence_only":["raw_confidence","raw_margin","raw_entropy"],
    "deployable_support_plus_ood":LOCAL_DEPLOYABLE_FEATURES,
}
if RUN["failure_detector"]:
    failure_results,failure_scored=da.evaluate_failure_detectors(
        predictions,ANALYSIS_ROOT/"failure_detector",feature_sets=FAILURE_FEATURE_SETS,classifiers=("logreg","rf"),n_splits=5)
    failure_results.to_csv(ANALYSIS_ROOT/"TABLE_failure_detector_protein_grouped.csv",index=False)
    display(failure_results.sort_values(["dataset","model","failure_AUROC"],ascending=[True,True,False]))
    importance=da.cv_permutation_importance_failure_detector(
        predictions,ANALYSIS_ROOT/"failure_detector",features=LOCAL_DEPLOYABLE_FEATURES,n_splits=5,n_repeats=10)
    importance.to_csv(ANALYSIS_ROOT/"TABLE_failure_detector_importance.csv",index=False)


## 11. Leave-one-dataset-out failure-risk transfer


In [ ]:
if RUN["leave_one_dataset_out_failure"]:
    lodo_results,lodo_scored=jx.leave_one_dataset_out_failure_detection(
        predictions,LOCAL_DEPLOYABLE_FEATURES,classifier="rf",output_path=ANALYSIS_ROOT/"lodo_failure")
    display(lodo_results)


## 12. Distribution-shift diagnostics

MMD is retained as a fold-level diagnostic and is not used as an input to the
per-prediction failure detector.


In [ ]:
if RUN["mmd_supplement"]:
    shift_table=da.compute_fold_shift_table(DATA_ROOT,OUTPUT_ROOT,ANALYSIS_ROOT,DATASETS,PROTOCOLS,SEEDS,N_SPLITS,PROFILE,RNA_TAG,PROTEIN_TAG)
    shift_table.to_csv(ANALYSIS_ROOT/"TABLE_distribution_shift.csv",index=False)
    display(shift_table.groupby(["dataset","protocol"])[["mmd_rna","mmd_protein","rna_degree_js","protein_degree_js","prevalence_delta"]].agg(["mean","std"]).round(5))


## 13. Statistical analysis


In [ ]:
if RUN["statistics"]:
    paired=da.paired_model_tests(fold_summary,ANALYSIS_ROOT,metrics=("RAW_AUROC","RAW_AUPRC","RAW_MCC","RAW_ECE","RAW_BRIER","RAW_HC90_ERROR"))
    bootstrap=jx.bootstrap_fold_ci(fold_summary,metrics=("RAW_AUROC","RAW_AUPRC","RAW_MCC","RAW_ECE","RAW_BRIER"),n_boot=5000)
    bootstrap.to_csv(ANALYSIS_ROOT/"TABLE_bootstrap_95CI.csv",index=False)
    display(paired.sort_values(["dataset","metric","p_holm"]).head(100))
    display(bootstrap)


## 14. Figures and source tables

Each generated figure is accompanied by the source table used to create it.


In [ ]:
MODEL_COLORS={"GraphSAGE-2L":"tab:blue","PairMLP-Cross":"tab:orange","ProtoContrast":"tab:green","ZHMolGraph-Best120":"tab:red"}

def save_paper_figure(fig,source_df,figure_id,caption,interpretation):
    pdf=FIGURE_ROOT/f"{figure_id}.pdf"; png=FIGURE_ROOT/f"{figure_id}.png"
    fig.savefig(pdf,bbox_inches="tight"); fig.savefig(png,dpi=400,bbox_inches="tight"); plt.close(fig)
    jx.save_figure_source(source_df,figure_id,FIGURE_ROOT,caption,interpretation,pdf.name,png.name)
    print("saved",figure_id)

if RUN["paper_figures"]:
    # 1) Central predictive comparison.
    src=fold_summary.groupby(["dataset","model"],as_index=False).agg(AUROC=("RAW_AUROC","mean"),AUROC_SD=("RAW_AUROC","std"))
    fig,ax=plt.subplots(figsize=(6.8,4.2)); x=np.arange(len(DATASETS)); w=.19
    for j,m in enumerate(CENTRAL_MODELS):
        g=src[src.model==m].set_index("dataset").reindex(DATASETS)
        ax.bar(x+(j-1.5)*w,g.AUROC,w,yerr=g.AUROC_SD,capsize=2,label=m,color=MODEL_COLORS[m])
    ax.set_xticks(x,DATASETS); ax.set_ylabel("AUROC"); ax.set_ylim(.5,1.0); ax.grid(axis="y",alpha=.25); ax.legend(fontsize=8,ncol=2); fig.tight_layout()
    save_paper_figure(fig,src,"FIG_J_AUROC",
        "Protein-cold AUROC of the four central RPI systems. Bars show mean across matched seed/fold units and error bars show one standard deviation.",
        "This is the primary predictive comparison under unseen proteins; ranking should be interpreted jointly with the reliability figures.")

    # 2) Calibration ECE.
    src=fold_summary.groupby(["dataset","model"],as_index=False).agg(ECE=("RAW_ECE","mean"),ECE_SD=("RAW_ECE","std"))
    fig,ax=plt.subplots(figsize=(6.8,4.2))
    for j,m in enumerate(CENTRAL_MODELS):
        g=src[src.model==m].set_index("dataset").reindex(DATASETS)
        ax.bar(x+(j-1.5)*w,g.ECE,w,yerr=g.ECE_SD,capsize=2,label=m,color=MODEL_COLORS[m])
    ax.set_xticks(x,DATASETS); ax.set_ylabel("Expected calibration error"); ax.grid(axis="y",alpha=.25); ax.legend(fontsize=8,ncol=2); fig.tight_layout()
    save_paper_figure(fig,src,"FIG_J_ECE",
        "Expected calibration error under protein-cold evaluation; lower values indicate closer agreement between confidence and empirical correctness.",
        "The most accurate model is not automatically the best calibrated, motivating a dependability analysis beyond AUROC.")

    # 3) Failure detector.
    bestfd=failure_results[(failure_results.classifier=="rf") & (failure_results.feature_set=="deployable_support_plus_ood")].copy()
    fig,ax=plt.subplots(figsize=(6.8,4.2))
    for j,m in enumerate(CENTRAL_MODELS):
        g=bestfd[bestfd.model==m].set_index("dataset").reindex(DATASETS)
        ax.bar(x+(j-1.5)*w,g.failure_AUROC,w,label=m,color=MODEL_COLORS[m])
    ax.axhline(.5,color="black",ls="--",lw=.8); ax.set_xticks(x,DATASETS); ax.set_ylim(.5,1.0); ax.set_ylabel("Failure-detection AUROC"); ax.grid(axis="y",alpha=.25); ax.legend(fontsize=8,ncol=2); fig.tight_layout()
    save_paper_figure(fig,bestfd,"FIG_J_FAILURE_AUROC",
        "Held-out-protein grouped failure-detection AUROC using deployable confidence, training-support, and embedding-familiarity/OOD signals.",
        "High values show that wrong primary predictions can often be identified before their labels are known; grouping by protein prevents the same held-out protein appearing in meta-training and meta-test.")

    # 4) LODO failure detector.
    if 'lodo_results' in globals() and len(lodo_results):
        src=lodo_results.copy(); fig,ax=plt.subplots(figsize=(6.8,4.2)); xx=np.arange(len(DATASETS));
        for j,m in enumerate(CENTRAL_MODELS):
            g=src[src.model==m].set_index("target_dataset").reindex(DATASETS)
            ax.bar(xx+(j-1.5)*w,g.failure_AUROC,w,label=m,color=MODEL_COLORS[m])
        ax.axhline(.5,color="black",ls="--",lw=.8); ax.set_xticks(xx,DATASETS); ax.set_ylim(.5,1.0); ax.set_ylabel("Failure AUROC on unseen dataset"); ax.grid(axis="y",alpha=.25); ax.legend(fontsize=8,ncol=2); fig.tight_layout()
        save_paper_figure(fig,src,"FIG_J_FAILURE_LODO",
            "Leave-one-dataset-out failure detection. The reliability model is fitted on two RPI datasets and evaluated on the third without using target-dataset failure labels for training.",
            "This tests whether learned warning signals represent transferable failure conditions rather than dataset-specific shortcuts.")

    display(pd.read_csv(FIGURE_ROOT/"figure_manifest.csv"))


## 15. Optional external evaluation on TheNovel

Enable only when the required NPInter2 source checkpoints are available.


In [ ]:
if RUN["thenovel_external"]:
    # TheNovel follows the original ZHMolGraph-style NPInter2 source-model
    # ensemble: five NPInter2 edge-level folds, seed 64. Train these source
    # checkpoints only if they do not already exist.
    THENOVEL_SOURCE_PLAN=ub.ExperimentPlan(
        name="JOURNAL_THENOVEL_NPINTER2_SOURCE",profile=PROFILE,
        datasets=("NPInter2",),protocols=("edge",),models=CENTRAL_MODELS,
        seeds=(64,),n_splits=5,val_fraction=.10,folds=None,
        run_thenovel=False,resume=True,overwrite=False,
    )
    source_runs=ub.run_plan(
        THENOVEL_SOURCE_PLAN,DATA_ROOT,OUTPUT_ROOT,pair_config=PAIR_CONFIG,
        zh_config=ZH_CONFIG,device=DEVICE,rna_tag=RNA_TAG,protein_tag=PROTEIN_TAG,
    )
    display(source_runs)
    thenovel=ub.run_thenovel(
        PROFILE,64,CENTRAL_MODELS,DATA_ROOT,OUTPUT_ROOT,
        pair_config=PAIR_CONFIG,zh_config=ZH_CONFIG,device=DEVICE,n_splits=5,
        rna_tag=RNA_TAG,protein_tag=PROTEIN_TAG,
    )
    thenovel.to_csv(ANALYSIS_ROOT/"TABLE_TheNovel_external.csv",index=False)
    display(thenovel)


## 16. Optional supplementary analyses

Conformal prediction, ensemble disagreement, fault injection, and Integrated
Gradients are disabled by default. Enable only the analyses required for the
intended reproduction run.


In [ ]:
if RUN["conformal_supplement"]:
    conformal=da.run_split_conformal(predictions,ANALYSIS_ROOT/"supp_conformal")
    display(conformal)

if RUN["ensemble_supplement"]:
    ensemble_predictions,ensemble_summary=da.build_seed_ensemble(predictions,ANALYSIS_ROOT/"supp_ensemble")
    display(ensemble_summary)

if RUN["fault_supplement"]:
    print("Use da.run_fault_injection(...) after all checkpoints are complete; keep as supplementary stress testing.")

if RUN["xai_supplement"]:
    print("Use da.run_pairmlp_cross_xai(...) with validation-derived references and confidence matching; keep as supplementary unless it generalizes across datasets.")


## 17. Compact artifact export

Exports tables, figure source data, figures, compact predictions, metrics, and
split manifests. Checkpoints are excluded by default to keep the archive small.


In [ ]:
if RUN["compact_export"]:
    archive=WORK_ROOT/"dependable_rpi_journal_v1_ARTIFACTS.zip"
    if archive.exists(): archive.unlink()
    jx.compact_export(OUTPUT_ROOT,ANALYSIS_ROOT,archive,include_checkpoints=False)
    print("Compact archive:",archive,"size MB=",round(archive.stat().st_size/1024**2,1))
    print("Free disk after export:",round(shutil.disk_usage(OUTPUT_ROOT).free/1024**3,2),"GB")
